# OnKith research figures

Every table and plot for the report, built from evidence already in the
repository. This notebook **trains nothing, evaluates nothing and downloads
nothing** - it reads saved metrics and draws them.

Run All. Each section either produces its figure or says exactly which artifact
is missing; a missing input never stops the notebook.

**Outputs**

```text
research_results/
├── figures/      PNG (300 dpi) and PDF for every plot
├── tables/       the numbers behind each figure, as CSV
└── summaries/    figure index and a written overview
```

## The current system

`FullPipeline_V6` is the system OnKith ships, and it has been measured directly
on **both** devices:

| Benchmark | Workstation | Raspberry Pi 5 |
|---|---|---|
| Audio pipeline, 44 clips | yes | yes |
| CPU/text privacy, 2,000 rows | yes | yes |
| Assistant round trip | yes, live LLM | yes, outbound-payload audit only |

Sections 9, 10, 12 and 13 draw the current system. Sections 2-8 are
model-only evaluation and are device-independent. Section 11 compares
recognizer generations and keeps the Canary-era deployment evidence, because a
superseded measurement is still provenance.

## Run selection

There is no "latest run" logic anywhere in this notebook. Every figure is
drawn from a run named in `CANONICAL_RUNS` in section 1, keyed by
`(pipeline version, benchmark, device class)`. A run folder that exists on
disk but is not in that table is listed as discovered-not-canonical and is
never silently substituted. `research_results/summaries/figure_index.json`
records the exact run id behind every figure this notebook writes.

**Honesty rule.** A metric is only plotted beside another metric when the two
were measured the same way. Where a historical model has no comparable number,
this notebook says "not recorded" rather than inventing one or quietly
substituting a different metric.

## 0. Setup

In [1]:
import csv
import json
import math
import statistics
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # figures are files, not windows; keeps Run All headless-safe
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

#: The pipeline the repository currently ships. Everything the notebook calls
#: "current" is measured on this one; everything else is history.
CURRENT_PIPELINE = "V6"

PIPELINE_DIRNAME = f"FullPipeline_{CURRENT_PIPELINE}"

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / PIPELINE_DIRNAME).is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
PIPELINE_ROOT = REPO_ROOT / PIPELINE_DIRNAME
if not PIPELINE_ROOT.is_dir():
    raise RuntimeError(
        f"Run this notebook from inside the OnKith repository "
        f"({PIPELINE_DIRNAME}/ was not found in any parent directory)."
    )

# The evaluation package is imported by bare name, the way the benchmarks import
# it. Sections 11 and 12 use privacy_eval.stt_comparison to score every
# recognizer with one implementation, reading saved run folders - it never loads
# a model, so nothing in this notebook can start Canary, Whisper or Parakeet.
#
# Only the current pipeline root goes on sys.path. Several archived pipelines
# ship a package called `integrated_pipeline`, and whichever reaches sys.path
# first wins for the rest of the process; naming one root is what keeps that
# unambiguous. This mirrors privacy_eval.benchmark_common.ensure_import_paths.
import sys

for _extra in (REPO_ROOT / "model" / "privacy_evaluation", PIPELINE_ROOT):
    if str(_extra) not in sys.path:
        sys.path.insert(0, str(_extra))

RESULTS = REPO_ROOT / "research_results"
FIGURES = RESULTS / "figures"
TABLES = RESULTS / "tables"
SUMMARIES = RESULTS / "summaries"
#: V6 runs land here; the frozen V3/V4/V5 folders stay in cpu_benchmarks/.
V6_BENCHMARKS = RESULTS / "v6_benchmarks"
LEGACY_BENCHMARKS = RESULTS / "cpu_benchmarks"
BENCHMARK_ROOTS = (V6_BENCHMARKS, LEGACY_BENCHMARKS)
for directory in (FIGURES, TABLES, SUMMARIES, V6_BENCHMARKS, LEGACY_BENCHMARKS):
    directory.mkdir(parents=True, exist_ok=True)

# --- publication styling -----------------------------------------------------
# Categorical hues are assigned in a fixed order and never cycled: blue is always
# the current system, orange the system it replaced, aqua a third series where
# one exists.
SERIES = {"primary": "#2a78d6", "secondary": "#eb6834", "tertiary": "#1baf7a"}
PURPLE = "#8a63d2"

# Public-facing names first, version tags second. A visitor should be able to
# read a chart without knowing the repository's version history, so the legend
# says "Current system" and the parenthesis carries the provenance.
SYSTEM_LABEL = {
    "V6": "Current system (V6, Whisper base.en)",
    "V5": "Parakeet demo (V5)",
    "V4": "Previous system (V4, Whisper base.en)",
    "V3": "Canary generation (V3)",
}
SYSTEM_SHORT = {
    "V6": "Current system",
    "V5": "Parakeet demo",
    "V4": "Previous system",
    "V3": "Canary generation",
}
# One colour per recognizer generation, fixed everywhere it appears.
SYSTEM_COLOUR = {
    "V6": SERIES["primary"],
    "V4": "#7aa8dd",
    "V3": SERIES["secondary"],
    "V5": SERIES["tertiary"],
}
ENGINE_LABEL = {
    "whisper": "faster-whisper base.en",
    "canary": "NVIDIA Canary 180M Flash",
    "parakeet": "Parakeet 110M TDT",
}
ENGINE_COLOUR = {
    "whisper": SERIES["primary"],
    "canary": SERIES["secondary"],
    "parakeet": SERIES["tertiary"],
}
DEVICE_LABEL = {"pi": "Raspberry Pi 5", "workstation": "Workstation"}
DEVICE_COLOUR = {"workstation": SERIES["primary"], "pi": SERIES["secondary"]}

INK = "#0b0b0b"
MUTED = "#52514e"
GRID = "#d9d8d4"

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.size": 10.5,
    "axes.titlesize": 12.5,
    "axes.labelsize": 10.5,
    "axes.titleweight": "bold",
    "axes.edgecolor": GRID,
    "axes.labelcolor": INK,
    "axes.titlecolor": INK,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": GRID,
    "grid.linewidth": 0.7,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "legend.frameon": False,
    "legend.fontsize": 9.5,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "lines.linewidth": 2.0,
    "lines.markersize": 6,
})

GENERATED: list[dict] = []      # every figure and table this run produced
UNAVAILABLE: list[str] = []     # every input that was missing, with the reason
FIGURE_PROVENANCE: dict[str, list[str]] = {}   # figure name -> run ids behind it


def tidy(axis, *, x_grid: bool = False) -> None:
    """Recessive frame: no top/right spines, grid on the value axis only."""

    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.grid(axis="x" if x_grid else "y")
    axis.set_axisbelow(True)


def save(figure, name: str, caption: str, *, runs: "list[str] | None" = None) -> None:
    """PNG for the report, PDF for the vector version, both at print quality.

    ``runs`` names the exact benchmark run ids the figure was drawn from. It is
    written into ``summaries/figure_index.json`` so a reader can trace any
    figure back to raw CSV/JSON without guessing which run was "latest".
    """

    png, pdf = FIGURES / f"{name}.png", FIGURES / f"{name}.pdf"
    figure.savefig(png)
    figure.savefig(pdf)
    plt.close(figure)
    provenance = sorted(set(runs or []))
    FIGURE_PROVENANCE[name] = provenance
    GENERATED.append({"kind": "figure", "name": name,
                      "files": [png.name, f"{pdf.name}"], "caption": caption,
                      "runs": provenance})
    suffix = f"  <- {len(provenance)} run(s)" if provenance else ""
    print(f"  figure  {png.relative_to(REPO_ROOT).as_posix()}  (+ .pdf){suffix}")


def save_table(rows, fieldnames, name: str, caption: str,
               *, runs: "list[str] | None" = None) -> None:
    if not rows:
        return
    path = TABLES / f"{name}.csv"
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fieldnames, lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)
    GENERATED.append({"kind": "table", "name": name, "files": [path.name],
                      "caption": caption, "rows": len(rows),
                      "runs": sorted(set(runs or []))})
    print(f"  table   {path.relative_to(REPO_ROOT).as_posix()}  ({len(rows)} rows)")


def load_json(path: Path, label: str):
    if not Path(path).is_file():
        UNAVAILABLE.append(f"{label}: {Path(path).relative_to(REPO_ROOT).as_posix()} not found")
        return None
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        UNAVAILABLE.append(f"{label}: unreadable ({exc})")
        return None


# Only the repository's own name is printed. An absolute path here would be
# stored in the notebook's committed output and would publish somebody's home
# directory with it.
print("repository :", REPO_ROOT.name, "(resolved upwards from the working directory)")
print("pipeline   :", PIPELINE_ROOT.name, "(current)")
print("figures    :", FIGURES.relative_to(REPO_ROOT).as_posix())
print("tables     :", TABLES.relative_to(REPO_ROOT).as_posix())

repository : OnKith (resolved upwards from the working directory)
pipeline   : FullPipeline_V6 (current)
figures    : research_results/figures
tables     : research_results/tables


## 1. Discover the evidence

Two things happen here. First the model-only evaluation artifacts are loaded -
those are device-independent and back sections 2 to 8. Then every benchmark run
folder on disk is catalogued and matched against `CANONICAL_RUNS`, the single
table that decides which run backs which figure.

`CANONICAL_RUNS` is the only place a run id is written down. Nothing downstream
picks a run by directory order, by modification time or by "latest".

In [2]:
MODEL_V2 = REPO_ROOT / "model" / "DeBERTa-v3-xsmall"
TINYBERT_V1 = REPO_ROOT / "model" / "TinyBERT-4" / "artifacts" / "tinybert_pii"
METRICS = MODEL_V2 / "artifacts" / "model_v2" / "metrics"
EVALUATION = REPO_ROOT / "model" / "privacy_evaluation"

DEBERTA_FP32 = load_json(METRICS / "evaluation_production_path_fp32_onnx.json", "DeBERTa FP32 ONNX")
DEBERTA_INT8 = load_json(METRICS / "evaluation_production_path_int8_onnx.json", "DeBERTa INT8 ONNX")
ACCEPTED = load_json(METRICS / "accepted_int8_candidate.json", "accepted INT8 candidate")
OOD_FINAL_DEBERTA = load_json(METRICS / "evaluation_ood_final_int8.json", "DeBERTa frozen OOD final")
DEPLOYED_INT8 = PIPELINE_ROOT / "models" / "deberta_pii" / "model_int8.onnx"
DEPLOY_MANIFEST = load_json(
    PIPELINE_ROOT / "models" / "deberta_pii" / "model_manifest.json",
    "deployment manifest",
)

TINYBERT_IID = load_json(EVALUATION / "baselines" / "tinybert_iid_baseline.json", "TinyBERT IID baseline")
TINYBERT_OOD_DEV = load_json(EVALUATION / "results" / "tinybert_dev_summary.json", "TinyBERT OOD dev")
TINYBERT_OOD_FINAL = load_json(EVALUATION / "results" / "tinybert_final_summary.json", "TinyBERT OOD final")

BILSTM = load_json(REPO_ROOT / "model" / "artifacts" / "final_eval_metrics.json", "BiLSTM baseline")
PREFLIGHT = load_json(
    MODEL_V2 / "artifacts" / "preflight" / "metrics" / "final_benchmark.json",
    "workstation preflight benchmark",
)

VIEW_ORDER = ["test", "val_original", "val_supplementary", "combined_validation", "ood_dev"]
VIEW_LABELS = {
    "test": "Test",
    "val_original": "Validation\n(original)",
    "val_supplementary": "Validation\n(supplementary)",
    "combined_validation": "Combined\nvalidation",
    "ood_dev": "OOD dev",
    "ood_final": "OOD final\n(frozen)",
}
# Short forms for the axes that carry six groups; the descriptive names above
# stay available for captions and prose.
VIEW_TICKS = {
    "test": "Test",
    "val_original": "Val\n(orig)",
    "val_supplementary": "Val\n(supp)",
    "combined_validation": "Combined\nval",
    "ood_dev": "OOD\ndev",
    "ood_final": "OOD final\n(frozen)",
}

sources = [
    ("DeBERTa INT8, five evaluation views", DEBERTA_INT8),
    ("DeBERTa FP32, five evaluation views", DEBERTA_FP32),
    ("DeBERTa frozen OOD final (one-shot)", OOD_FINAL_DEBERTA),
    ("Accepted INT8 candidate / quality gate", ACCEPTED),
    ("TinyBERT IID baseline (historical)", TINYBERT_IID),
    ("TinyBERT OOD dev", TINYBERT_OOD_DEV),
    ("TinyBERT OOD final", TINYBERT_OOD_FINAL),
    ("BiLSTM baseline", BILSTM),
    ("Workstation latency preflight", PREFLIGHT),
]
print(f"{'source':44} available")
print("-" * 56)
for label, payload in sources:
    print(f"{label:44} {'yes' if payload else 'NO'}")


# ==========================================================================
# Canonical run selection
# ==========================================================================
# One run per (pipeline version, benchmark, device class). Written out in full
# so that "which committed run backs this number?" has an answer that does not
# depend on directory order, on file timestamps or on the word "latest". A
# folder on disk that is not named here is reported below and never drawn.
CANONICAL_RUNS: dict = {
    # --- the current system ------------------------------------------------
    ("V6", "audio_pipeline", "workstation"):
        "v6_benchmarks/20260823T142306Z_windows-amd64_audio-v6-whisper_full_workstation",
    ("V6", "audio_pipeline", "pi"):
        "v6_benchmarks/20260823T204339Z_linux-aarch64_audio-v6-whisper_full",
    ("V6", "cpu_text", "workstation"):
        "v6_benchmarks/20260823T142451Z_windows-amd64_v6-whisper_full_workstation",
    ("V6", "cpu_text", "pi"):
        "v6_benchmarks/20260823T203840Z_linux-aarch64_v6-whisper_full",
    ("V6", "end_to_end", "workstation"):
        "v6_benchmarks/20260823T142736Z_windows-amd64_e2e-v6-whisper_full_workstation",
    # The Pi round trip ran with no GROQ_API_KEY: the local pipeline and the
    # outbound payload are real measurements, the network leg was not
    # attempted. Section 13 draws only what was measured and says so.
    ("V6", "end_to_end", "pi"):
        "v6_benchmarks/20260823T204627Z_linux-aarch64_e2e-v6-whisper_full",

    # --- history, kept as provenance ---------------------------------------
    # V5, the Parakeet demonstration. Workstation only; never run on the Pi.
    ("V5", "audio_pipeline", "workstation"):
        "cpu_benchmarks/20260823T095553Z_windows-amd64_audio-parakeet_full",
    ("V5", "cpu_text", "workstation"):
        "cpu_benchmarks/20260823T095708Z_windows-amd64_parakeet_full",
    ("V5", "end_to_end", "workstation"):
        "cpu_benchmarks/20260823T095954Z_windows-amd64_e2e-parakeet_full",
    # V4, the system V6 inherits its local core from. Its text runs are what
    # section 12's "integration changed nothing local" check compares against.
    ("V4", "cpu_text", "workstation"):
        "cpu_benchmarks/20260820T145843Z_windows-amd64_whisper_full_workstation",
    ("V4", "cpu_text", "pi"):
        "cpu_benchmarks/20260822T122249Z_linux-aarch64_whisper_full",
    ("V4", "audio_pipeline", "workstation"):
        "cpu_benchmarks/20260820T145659Z_windows-amd64_audio-whisper_full_workstation",
    # Chosen over the earlier 20260820T071755Z Pi folder because that one ran
    # before the spoken-address rewrite reached the privacy path, so its
    # end-to-end privacy numbers are not comparable with V6's. This folder ran
    # the same privacy path V6 runs. It records no peak RSS, so its memory
    # column stays blank rather than being guessed.
    ("V4", "audio_pipeline", "pi"):
        "cpu_benchmarks/20260821T201653Z_linux-aarch64_audio-whisper_full",
    # V3, the Canary generation. A frozen historical baseline; never rerun.
    ("V3", "cpu_text", "workstation"):
        "cpu_benchmarks/20260817T190227Z_windows-amd64_full_workstation",
    ("V3", "cpu_text", "pi"):
        "cpu_benchmarks/20260818T082425Z_linux-aarch64_full",
    ("V3", "audio_pipeline", "workstation"):
        "cpu_benchmarks/20260818T062503Z_windows-amd64_audio_full_workstation",
    ("V3", "audio_pipeline", "pi"):
        "cpu_benchmarks/20260818T081310Z_linux-aarch64_audio_full",
}


def benchmark_kind(directory: Path) -> str:
    """Text, audio and round-trip runs share two results roots; config says which.

    Runs recorded before the audio benchmark existed carry no marker, and they
    are all text runs. The two round-trip benchmarks tag themselves with the
    pipeline that wrote them, so both fold to one kind here.
    """

    try:
        config = json.loads((directory / "benchmark_config.json").read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return "unreadable"
    kind = str(config.get("benchmark", "cpu_text"))
    return "end_to_end" if kind.endswith("_end_to_end") else kind


def pipeline_version_of(directory: Path) -> str:
    """Which pipeline wrote a run folder.

    ``pipeline_version`` was added when V4 replaced V3. Folders without it are
    all V3 runs - the key's absence is the identification, not a guess.
    """

    try:
        config = json.loads((directory / "benchmark_config.json").read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return "unknown"
    return str(config.get("pipeline_version") or "V3")


def device_class_of(directory: Path) -> str:
    try:
        system = json.loads((directory / "system_info.json").read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return "unknown"
    if system.get("is_raspberry_pi") or "aarch64" in str(system.get("architecture", "")).lower():
        return "pi"
    return "workstation"


DISCOVERED = sorted(
    (directory
     for root in BENCHMARK_ROOTS
     for directory in root.glob("*")
     if directory.is_dir() and (directory / "benchmark_config.json").is_file()),
    key=lambda path: path.name,
)

# Resolve the registry against what is actually on disk, and refuse to guess.
CANONICAL_PATHS: dict = {}
for key, relative in CANONICAL_RUNS.items():
    candidate = RESULTS / relative
    if not (candidate / "benchmark_config.json").is_file():
        UNAVAILABLE.append(f"canonical run missing on disk: {'/'.join(key)} -> {relative}")
        continue
    observed = (pipeline_version_of(candidate), benchmark_kind(candidate),
                device_class_of(candidate))
    if observed != key:
        UNAVAILABLE.append(
            f"canonical run mislabelled: {relative} is registered as "
            f"{'/'.join(key)} but its own metadata says {'/'.join(observed)}; not used"
        )
        continue
    CANONICAL_PATHS[key] = candidate

CANONICAL_IDS = {path.name for path in CANONICAL_PATHS.values()}
NON_CANONICAL = [d for d in DISCOVERED if d.name not in CANONICAL_IDS]
for directory in NON_CANONICAL:
    UNAVAILABLE.append(
        f"{directory.name}: discovered but not canonical for "
        f"{pipeline_version_of(directory)}/{benchmark_kind(directory)}/"
        f"{device_class_of(directory)}; kept on disk as evidence, out of the figures"
    )


def canonical(version: str, kind: str, device: str):
    """The one registered run folder, or None if it is not on disk."""

    return CANONICAL_PATHS.get((version, kind, device))


print(f"\nbenchmark run folders on disk : {len(DISCOVERED)}")
print(f"canonical (drawn in figures)  : {len(CANONICAL_PATHS)}")
print(f"superseded / not canonical    : {len(NON_CANONICAL)}")
print()
print(f"{'pipeline':9} {'benchmark':15} {'device':15} run")
print("-" * 112)
for key in sorted(CANONICAL_PATHS, reverse=True):
    version, kind, device = key
    print(f"{version:9} {kind:15} {DEVICE_LABEL[device]:15} {CANONICAL_PATHS[key].name}")
if NON_CANONICAL:
    print("\nsuperseded / not canonical (kept on disk as evidence):")
    for directory in NON_CANONICAL:
        print(f"  {directory.name}")

# The run sets the later sections consume, split by benchmark kind.
CANONICAL_TEXT = {k: v for k, v in CANONICAL_PATHS.items() if k[1] == "cpu_text"}
CANONICAL_AUDIO = {k: v for k, v in CANONICAL_PATHS.items() if k[1] == "audio_pipeline"}
CANONICAL_E2E = {k: v for k, v in CANONICAL_PATHS.items() if k[1] == "end_to_end"}

source                                       available
--------------------------------------------------------
DeBERTa INT8, five evaluation views          yes
DeBERTa FP32, five evaluation views          yes
DeBERTa frozen OOD final (one-shot)          yes
Accepted INT8 candidate / quality gate       yes
TinyBERT IID baseline (historical)           yes
TinyBERT OOD dev                             yes
TinyBERT OOD final                           yes
BiLSTM baseline                              yes
Workstation latency preflight                yes

benchmark run folders on disk : 24
canonical (drawn in figures)  : 17
superseded / not canonical    : 7

pipeline  benchmark       device          run
----------------------------------------------------------------------------------------------------------------
V6        end_to_end      Workstation     20260823T142736Z_windows-amd64_e2e-v6-whisper_full_workstation
V6        end_to_end      Raspberry Pi 5  20260823T204627Z_linux-aarch64_e2e-

## 2. Model progression

Three generations of privacy model. Only numbers measured the same way are put
on the same axis:

* **BiLSTM** was scored with token-level *binary* accuracy on its own validation
  split. There is no span-level evaluation for it, so it has no bar here. Its
  recorded number is in the table, flagged as not comparable.
* **TinyBERT-4 INT8** and **DeBERTa-v3-xsmall INT8** were scored on the *same*
  40,909-row held-out AI4Privacy English validation split, through the same
  metric definitions, in `deberta_vs_tinybert_int8_validation.ipynb`. Those two
  are directly comparable and are what the figure shows.

In [3]:
progression_rows = []
head_to_head = (DEPLOY_MANIFEST or {}).get("held_out_validation")

if BILSTM:
    progression_rows.append({
        "model": "BiLSTM baseline",
        "generation": "v0",
        "precision": "fp32",
        "metric": "token-level binary F1",
        "value": BILSTM["val_f1"],
        "precision_value": BILSTM["val_precision"],
        "recall_value": BILSTM["val_recall"],
        "evaluation": "own validation split",
        "comparable_to_span_metrics": False,
    })

if head_to_head:
    for key, name, generation in (
        ("tinybert_int8_model_v1", "TinyBERT-4 INT8", "v1"),
        ("deberta_int8", "DeBERTa-v3-xsmall INT8", "v2"),
    ):
        entry = head_to_head[key]
        progression_rows.append({
            "model": name,
            "generation": generation,
            "precision": "int8",
            "metric": "typed entity F1",
            "value": entry["typed_f1"],
            "precision_value": None,
            "recall_value": None,
            "evaluation": f"shared held-out validation, {head_to_head['rows']:,} rows",
            "comparable_to_span_metrics": True,
            "private_character_leakage": entry["private_character_leakage"],
            "non_private_overmasking_rate": entry["non_private_overmasking_rate"],
        })

if head_to_head:
    comparable = [row for row in progression_rows if row["comparable_to_span_metrics"]]
    names = [row["model"] for row in comparable]
    f1 = [row["value"] for row in comparable]
    leakage = [row["private_character_leakage"] for row in comparable]
    overmask = [row["non_private_overmasking_rate"] for row in comparable]

    figure, (left, right) = plt.subplots(1, 2, figsize=(10.5, 4.2))
    colours = [SERIES["secondary"], SERIES["primary"]]

    # Bars start at zero. The two models are genuinely close on this split, and a
    # cropped axis that turned 0.0086 F1 into a towering difference would be a lie;
    # the gap that matters is in the error rates on the right.
    bars = left.bar(names, f1, color=colours, width=0.55)
    for bar, value in zip(bars, f1):
        left.annotate(f"{value:.4f}", (bar.get_x() + bar.get_width() / 2, value),
                      ha="center", va="bottom", xytext=(0, 4),
                      textcoords="offset points", fontsize=10, color=INK)
    left.set_ylim(0, 1.08)
    left.set_ylabel("Typed entity F1")
    left.set_title(f"Exact-span, correct-type F1  (+{f1[1] - f1[0]:.4f})")
    tidy(left)

    width = 0.35
    positions = range(len(names))
    error_bars_leak = right.bar([p - width / 2 for p in positions], leakage,
                                width=width, color=colours[0], label="Private character leakage")
    error_bars_over = right.bar([p + width / 2 for p in positions], overmask,
                                width=width, color=colours[1], label="Non-private overmasking")
    for group, values in ((error_bars_leak, leakage), (error_bars_over, overmask)):
        for bar, value in zip(group, values):
            right.annotate(f"{value:.4f}", (bar.get_x() + bar.get_width() / 2, value),
                           ha="center", va="bottom", xytext=(0, 3),
                           textcoords="offset points", fontsize=8.5, color=MUTED)
    right.set_xticks(list(positions), names)
    right.set_ylabel("Rate (lower is better)")
    right.set_title("Character-level error rates")
    right.set_ylim(0, max(leakage + overmask) * 1.6)
    right.legend(loc="upper center", ncol=2)
    tidy(right)

    figure.suptitle(
        "Privacy model generations on the shared held-out validation split "
        f"({head_to_head['rows']:,} rows)",
        fontsize=13, y=1.02,
    )
    figure.text(
        0.5, -0.06,
        "The earlier BiLSTM baseline has no span-level evaluation on record "
        f"(token-level binary F1 {BILSTM['val_f1']:.4f} on its own split), so it is "
        "listed in the table rather than plotted here."
        if BILSTM else "",
        ha="center", fontsize=9, color=MUTED,
    )
    save(figure, "model_progression",
         "Typed entity F1 and character-level error rates for TinyBERT INT8 and "
         "DeBERTa INT8 on the same held-out validation split.")
else:
    print("skipped: no shared head-to-head validation record")

save_table(
    progression_rows,
    ["model", "generation", "precision", "metric", "value", "precision_value",
     "recall_value", "evaluation", "comparable_to_span_metrics",
     "private_character_leakage", "non_private_overmasking_rate"],
    "model_progression",
    "Model generations with the metric each one actually has on record.",
)

  figure  research_results/figures/model_progression.png  (+ .pdf)
  table   research_results/tables/model_progression.csv  (3 rows)


## 3. Quantization: FP32 against INT8

Both runtimes were decoded by the same production decoder, so the difference is
quantization and nothing else. The dashed line is the release gate: a view fails
if it loses more than 0.02 typed entity F1.

In [4]:
GATE_LIMIT = 0.02
quantization_rows = []

if DEBERTA_FP32 and DEBERTA_INT8:
    fp32 = DEBERTA_FP32["summary"]
    int8 = DEBERTA_INT8["summary"]
    views = [view for view in VIEW_ORDER if view in fp32 and view in int8]

    for view in views:
        quantization_rows.append({
            "view": view,
            "rows": int8[view]["rows"],
            "fp32_typed_f1": fp32[view]["typed_entity_f1"],
            "int8_typed_f1": int8[view]["typed_entity_f1"],
            "typed_f1_drop": fp32[view]["typed_entity_f1"] - int8[view]["typed_entity_f1"],
            "fp32_leakage": fp32[view]["private_character_leakage"],
            "int8_leakage": int8[view]["private_character_leakage"],
            "leakage_rise": int8[view]["private_character_leakage"]
            - fp32[view]["private_character_leakage"],
            "fp32_character_recall": fp32[view]["private_character_recall"],
            "int8_character_recall": int8[view]["private_character_recall"],
        })

    figure, (top, bottom) = plt.subplots(
        2, 1, figsize=(9.5, 7.0), gridspec_kw={"height_ratios": [2.1, 1]}
    )
    positions = range(len(views))
    width = 0.38
    fp32_values = [fp32[view]["typed_entity_f1"] for view in views]
    int8_values = [int8[view]["typed_entity_f1"] for view in views]

    fp32_bars = top.bar([p - width / 2 for p in positions], fp32_values, width=width,
                        color=SERIES["tertiary"], label="FP32 ONNX")
    int8_bars = top.bar([p + width / 2 for p in positions], int8_values, width=width,
                        color=SERIES["primary"], label="INT8 ONNX (deployed)")
    for group, values in ((fp32_bars, fp32_values), (int8_bars, int8_values)):
        for bar, value in zip(group, values):
            top.annotate(f"{value:.3f}", (bar.get_x() + bar.get_width() / 2, value),
                         ha="center", va="bottom", xytext=(0, 3),
                         textcoords="offset points", fontsize=8.5, color=INK)
    top.set_xticks(list(positions), [VIEW_TICKS[view] for view in views])
    top.set_ylabel("Typed entity F1")
    top.set_ylim(0, 1.30)
    top.set_title("Typed entity F1 by evaluation view")
    top.legend(loc="upper left", ncol=2)
    tidy(top)

    drops = [row["typed_f1_drop"] for row in quantization_rows]
    drop_bars = bottom.bar(list(positions), drops, width=0.5, color=SERIES["secondary"])
    for bar, value in zip(drop_bars, drops):
        bottom.annotate(f"{value:+.4f}", (bar.get_x() + bar.get_width() / 2, value),
                        ha="center", va="bottom", xytext=(0, 3),
                        textcoords="offset points", fontsize=9, color=INK)
    bottom.axhline(GATE_LIMIT, linestyle="--", linewidth=1.4, color=MUTED)
    bottom.annotate(f"release gate {GATE_LIMIT}", (-0.45, GATE_LIMIT),
                    ha="left", va="bottom", xytext=(0, 4), textcoords="offset points",
                    fontsize=9, color=MUTED)
    bottom.set_xticks(list(positions), [VIEW_TICKS[view] for view in views])
    bottom.set_ylabel("F1 lost to INT8")
    bottom.set_ylim(0, max(GATE_LIMIT, max(drops)) * 1.75)
    bottom.set_title("Cost of quantization, against the acceptance limit")
    tidy(bottom)

    figure.suptitle("DeBERTa-v3-xsmall: FP32 against INT8 through one decoder",
                    fontsize=13, y=0.98)
    figure.tight_layout(rect=(0, 0, 1, 0.95))
    save(figure, "deberta_fp32_vs_int8",
         "FP32 and INT8 typed entity F1 on every evaluation view, with the F1 lost "
         "to quantization against the 0.02 release gate.")

    print(f"{'view':22} {'FP32':>8} {'INT8':>8} {'drop':>9} {'gate':>6}")
    for row in quantization_rows:
        print(f"{row['view']:22} {row['fp32_typed_f1']:>8.4f} {row['int8_typed_f1']:>8.4f} "
              f"{row['typed_f1_drop']:>+9.4f} {'PASS' if row['typed_f1_drop'] < GATE_LIMIT else 'FAIL':>6}")
else:
    print("skipped: the FP32 or INT8 production-path evaluation is missing")

save_table(
    quantization_rows,
    ["view", "rows", "fp32_typed_f1", "int8_typed_f1", "typed_f1_drop",
     "fp32_leakage", "int8_leakage", "leakage_rise",
     "fp32_character_recall", "int8_character_recall"],
    "deberta_quantization",
    "FP32 against INT8 on every view, with the deltas the release gate reads.",
)

  figure  research_results/figures/deberta_fp32_vs_int8.png  (+ .pdf)
view                       FP32     INT8      drop   gate
test                     0.9549   0.9527   +0.0022   PASS
val_original             0.9610   0.9588   +0.0023   PASS
val_supplementary        0.8878   0.8849   +0.0029   PASS
combined_validation      0.9558   0.9534   +0.0023   PASS
ood_dev                  0.7064   0.6882   +0.0182   PASS
  table   research_results/tables/deberta_quantization.csv  (5 rows)


## 4. Where the deployed model is strong and where it is not

The same INT8 artifact across every view it was measured on, including the
one-shot frozen OOD final benchmark. The in-distribution views and the
out-of-distribution views are deliberately drawn together: the gap between them
is the honest statement of what this model does on speech it has never seen.

In [5]:
split_rows = []

if DEBERTA_INT8:
    summary = dict(DEBERTA_INT8["summary"])
    if OOD_FINAL_DEBERTA:
        headline = OOD_FINAL_DEBERTA["headline"]
        metrics = OOD_FINAL_DEBERTA["metrics"]
        summary["ood_final"] = {
            "typed_entity_f1": headline["typed_entity_f1"],
            "typed_entity_precision": metrics["typed_entity"]["precision"],
            "typed_entity_recall": metrics["typed_entity"]["recall"],
            "binary_entity_f1": headline["binary_entity_f1"],
            "private_character_recall": headline["private_character_recall"],
            "private_character_leakage": headline["private_character_leakage"],
            "non_private_overmasking_rate": headline["non_private_overmasking_rate"],
            "hard_negative_false_positive_rate": headline["hard_negative_false_positive_rate"],
            "rows": OOD_FINAL_DEBERTA["fixture"]["case_count"],
        }

    views = [view for view in VIEW_ORDER + ["ood_final"] if view in summary]
    for view in views:
        entry = summary[view]
        split_rows.append({
            "view": view,
            "rows": entry["rows"],
            "typed_precision": entry["typed_entity_precision"],
            "typed_recall": entry["typed_entity_recall"],
            "typed_f1": entry["typed_entity_f1"],
            "binary_f1": entry["binary_entity_f1"],
            "private_character_recall": entry["private_character_recall"],
            "private_character_leakage": entry["private_character_leakage"],
            "non_private_overmasking_rate": entry["non_private_overmasking_rate"],
            "hard_negative_false_positive_rate": entry["hard_negative_false_positive_rate"],
        })

    figure, (left, right) = plt.subplots(1, 2, figsize=(13.5, 4.8))
    positions = range(len(views))
    width = 0.27
    triples = (
        ("Precision", [row["typed_precision"] for row in split_rows], SERIES["tertiary"]),
        ("Recall", [row["typed_recall"] for row in split_rows], SERIES["secondary"]),
        ("F1", [row["typed_f1"] for row in split_rows], SERIES["primary"]),
    )
    for index, (name, values, colour) in enumerate(triples):
        offset = (index - 1) * width
        bars = left.bar([p + offset for p in positions], values, width=width,
                        color=colour, label=name)
        if name == "F1":
            for bar, value in zip(bars, values):
                left.annotate(f"{value:.3f}", (bar.get_x() + bar.get_width() / 2, value),
                              ha="center", va="bottom", xytext=(0, 3),
                              textcoords="offset points", fontsize=8.5, color=INK)
    left.set_xticks(list(positions), [VIEW_TICKS[view] for view in views])
    left.set_ylim(0, 1.32)
    left.set_ylabel("Typed entity score")
    left.set_title("Exact-span, correct-type performance")
    left.legend(loc="upper left", ncol=3)
    tidy(left)

    recall = [row["private_character_recall"] for row in split_rows]
    bars = right.bar(list(positions), recall, width=0.55, color=SERIES["primary"])
    for bar, value in zip(bars, recall):
        right.annotate(f"{value:.3f}", (bar.get_x() + bar.get_width() / 2, value),
                       ha="center", va="bottom", xytext=(0, 3),
                       textcoords="offset points", fontsize=9, color=INK)
    right.set_xticks(list(positions), [VIEW_TICKS[view] for view in views])
    right.set_ylim(0, 1.15)
    right.set_ylabel("Private characters covered")
    right.set_title("Character recall - the privacy-relevant measure")
    right.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    tidy(right)

    figure.suptitle("Deployed DeBERTa-v3-xsmall INT8 across every evaluation view",
                    fontsize=13, y=1.01)
    save(figure, "deberta_split_performance",
         "Typed entity precision/recall/F1 and private-character recall for the "
         "deployed INT8 model on each evaluation view.")

    for row in split_rows:
        print(f"{row['view']:22} rows {row['rows']:>7,}  typed F1 {row['typed_f1']:.4f}  "
              f"char recall {row['private_character_recall']:.4f}  "
              f"leakage {row['private_character_leakage']:.4f}")
else:
    print("skipped: the INT8 production-path evaluation is missing")

save_table(
    split_rows,
    ["view", "rows", "typed_precision", "typed_recall", "typed_f1", "binary_f1",
     "private_character_recall", "private_character_leakage",
     "non_private_overmasking_rate", "hard_negative_false_positive_rate"],
    "deberta_split_performance",
    "Every metric of the deployed INT8 model on every evaluation view.",
)

  figure  research_results/figures/deberta_split_performance.png  (+ .pdf)
test                   rows  19,136  typed F1 0.9527  char recall 0.9973  leakage 0.0027
val_original           rows  40,909  typed F1 0.9588  char recall 0.9983  leakage 0.0017
val_supplementary      rows   6,907  typed F1 0.8849  char recall 0.9843  leakage 0.0157
combined_validation    rows  47,816  typed F1 0.9534  char recall 0.9969  leakage 0.0031
ood_dev                rows     360  typed F1 0.6882  char recall 0.8725  leakage 0.1275
ood_final              rows     360  typed F1 0.6244  char recall 0.8390  leakage 0.1610
  table   research_results/tables/deberta_split_performance.csv  (6 rows)


## 5. Per-entity performance

Entity types, not BIO labels: 31 entity rows are readable, 63 tag rows are not,
and the entity is the unit the mask policy is written in. Measured on the
combined validation view, the same one checkpoint selection used.

Bars are ordered worst to best, and each is annotated with its gold support, so
a low F1 on three spans is not mistaken for a low F1 on three thousand.

In [6]:
per_entity_rows = []

if DEBERTA_INT8 and "combined_validation" in DEBERTA_INT8["views"]:
    per_label = DEBERTA_INT8["views"]["combined_validation"]["per_label"]
    per_entity_rows = sorted(
        (
            {
                "entity": entity,
                "precision": entry["precision"],
                "recall": entry["recall"],
                "f1": entry["f1"],
                "support": entry["support"],
                "predicted": entry["predicted"],
                "true_positives": entry["true_positives"],
            }
            for entity, entry in per_label.items()
        ),
        key=lambda row: row["f1"],
    )

    height = max(5.0, 0.30 * len(per_entity_rows) + 1.4)
    figure, axis = plt.subplots(figsize=(9.0, height))
    labels = [row["entity"] for row in per_entity_rows]
    values = [row["f1"] for row in per_entity_rows]
    positions = range(len(labels))
    bars = axis.barh(list(positions), values, height=0.62, color=SERIES["primary"])
    for bar, row in zip(bars, per_entity_rows):
        axis.annotate(
            f"{row['f1']:.3f}   n={row['support']:,}",
            (row["f1"], bar.get_y() + bar.get_height() / 2),
            va="center", ha="left", xytext=(5, 0), textcoords="offset points",
            fontsize=8.5, color=MUTED,
        )
    axis.set_yticks(list(positions), labels)
    axis.set_xlim(0, 1.18)
    axis.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    axis.set_xlabel("Typed entity F1 (exact span, correct type)")
    axis.set_title(
        "Per-entity performance, deployed INT8 model\n"
        f"combined validation, {DEBERTA_INT8['summary']['combined_validation']['rows']:,} rows",
        loc="left",
    )
    tidy(axis, x_grid=True)
    save(figure, "per_entity_f1",
         "Typed entity F1 per entity type on the combined validation view, "
         "annotated with gold support.")

    weakest = per_entity_rows[:5]
    print("weakest entity types:")
    for row in weakest:
        print(f"  {row['entity']:20} F1 {row['f1']:.4f}  support {row['support']:,}")
else:
    print("skipped: no per-label breakdown available")

save_table(
    per_entity_rows,
    ["entity", "precision", "recall", "f1", "support", "predicted", "true_positives"],
    "per_entity_f1",
    "Per-entity precision, recall, F1 and support on the combined validation view.",
)

  figure  research_results/figures/per_entity_f1.png  (+ .pdf)
weakest entity types:
  AMOUNT               F1 0.0000  support 2
  BANKNAME             F1 0.0000  support 34
  CURRENCY             F1 0.0000  support 3
  TIMEZONE             F1 0.0000  support 17
  PASSWORD             F1 0.8049  support 175
  table   research_results/tables/per_entity_f1.csv  (30 rows)


## 6. Out-of-distribution: the two deployment models on the same fixtures

The strongest comparison in the project. Both models were scored on the two
purpose-built OOD fixtures - 360 development cases and 360 frozen final cases,
byte-identical files, the same metric code, both in their model-only form. The
frozen set was spent exactly once per model.

This is where a model trained on generated corpora meets speech-shaped input it
has never seen, and it separates the two generations far more sharply than the
in-distribution split does.

In [7]:
def tinybert_ood(summary, stage: str = "int8_model"):
    if not summary or stage not in summary.get("stages", {}):
        return None
    overall = summary["stages"][stage]["overall"]
    return {
        "typed_f1": overall["typed_entity"]["f1"],
        "typed_precision": overall["typed_entity"]["precision"],
        "typed_recall": overall["typed_entity"]["recall"],
        "character_recall": overall["privacy_character"]["private_character_recall"],
        "character_leakage": overall["privacy_character"]["private_character_leakage_rate"],
        "overmasking": overall["privacy_character"]["non_private_character_overmasking_rate"],
        "hard_negative_fp": overall["case_level"]["hard_negative_false_positive_case_rate"],
        "rows": overall["case_count"],
    }


def deberta_ood(view: str):
    if view == "ood_final":
        if not OOD_FINAL_DEBERTA:
            return None
        metrics, headline = OOD_FINAL_DEBERTA["metrics"], OOD_FINAL_DEBERTA["headline"]
        return {
            "typed_f1": headline["typed_entity_f1"],
            "typed_precision": metrics["typed_entity"]["precision"],
            "typed_recall": metrics["typed_entity"]["recall"],
            "character_recall": headline["private_character_recall"],
            "character_leakage": headline["private_character_leakage"],
            "overmasking": headline["non_private_overmasking_rate"],
            "hard_negative_fp": headline["hard_negative_false_positive_rate"],
            "rows": OOD_FINAL_DEBERTA["fixture"]["case_count"],
        }
    if not DEBERTA_INT8 or view not in DEBERTA_INT8["summary"]:
        return None
    entry = DEBERTA_INT8["summary"][view]
    return {
        "typed_f1": entry["typed_entity_f1"],
        "typed_precision": entry["typed_entity_precision"],
        "typed_recall": entry["typed_entity_recall"],
        "character_recall": entry["private_character_recall"],
        "character_leakage": entry["private_character_leakage"],
        "overmasking": entry["non_private_overmasking_rate"],
        "hard_negative_fp": entry["hard_negative_false_positive_rate"],
        "rows": entry["rows"],
    }


ood_rows = []
pairs = [
    ("OOD dev", tinybert_ood(TINYBERT_OOD_DEV), deberta_ood("ood_dev")),
    ("OOD final (frozen)", tinybert_ood(TINYBERT_OOD_FINAL), deberta_ood("ood_final")),
]
usable = [(name, old, new) for name, old, new in pairs if old and new]

for name, old, new in usable:
    for model, entry in (("TinyBERT-4 INT8", old), ("DeBERTa-v3-xsmall INT8", new)):
        ood_rows.append({"fixture": name, "model": model, **entry})

if usable:
    figure, axes = plt.subplots(1, 3, figsize=(13.0, 4.4))
    panels = (
        ("typed_f1", "Typed entity F1", "higher is better"),
        ("character_recall", "Private character recall", "higher is better"),
        ("hard_negative_fp", "Hard-negative false-positive rate", "lower is better"),
    )
    names = [name for name, _, _ in usable]
    positions = range(len(names))
    width = 0.36

    for axis, (key, title, direction) in zip(axes, panels):
        old_values = [old[key] for _, old, _ in usable]
        new_values = [new[key] for _, _, new in usable]
        old_bars = axis.bar([p - width / 2 for p in positions], old_values, width=width,
                            color=SERIES["secondary"], label="TinyBERT-4 INT8 (v1)")
        new_bars = axis.bar([p + width / 2 for p in positions], new_values, width=width,
                            color=SERIES["primary"], label="DeBERTa-v3-xsmall INT8 (v2)")
        for group, values in ((old_bars, old_values), (new_bars, new_values)):
            for bar, value in zip(group, values):
                axis.annotate(f"{value:.3f}", (bar.get_x() + bar.get_width() / 2, value),
                              ha="center", va="bottom", xytext=(0, 3),
                              textcoords="offset points", fontsize=8.5, color=INK)
        axis.set_xticks(list(positions), names)
        axis.set_ylim(0, 1.05)
        axis.set_title(f"{title}\n({direction})", fontsize=11)
        tidy(axis)
    axes[0].set_ylabel("Score")
    axes[0].legend(loc="upper left")

    figure.suptitle(
        "Out-of-distribution behaviour on identical 360-case fixtures, model-only paths",
        fontsize=13, y=1.03,
    )
    save(figure, "ood_model_comparison",
         "TinyBERT INT8 against DeBERTa INT8 on the OOD development and frozen "
         "final fixtures, model-only in both cases.")

    for row in ood_rows:
        print(f"{row['fixture']:20} {row['model']:24} typed F1 {row['typed_f1']:.4f}  "
              f"char recall {row['character_recall']:.4f}  hard-neg FP {row['hard_negative_fp']:.4f}")
else:
    print("skipped: OOD results are not available for both models")

save_table(
    ood_rows,
    ["fixture", "model", "rows", "typed_f1", "typed_precision", "typed_recall",
     "character_recall", "character_leakage", "overmasking", "hard_negative_fp"],
    "ood_model_comparison",
    "Both deployment models on the OOD development and frozen final fixtures.",
)

  figure  research_results/figures/ood_model_comparison.png  (+ .pdf)
OOD dev              TinyBERT-4 INT8          typed F1 0.4406  char recall 0.3500  hard-neg FP 0.6389
OOD dev              DeBERTa-v3-xsmall INT8   typed F1 0.6882  char recall 0.8725  hard-neg FP 0.2222
OOD final (frozen)   TinyBERT-4 INT8          typed F1 0.4649  char recall 0.3209  hard-neg FP 0.6528
OOD final (frozen)   DeBERTa-v3-xsmall INT8   typed F1 0.6244  char recall 0.8390  hard-neg FP 0.2361
  table   research_results/tables/ood_model_comparison.csv  (4 rows)


## 7. Deployment size

What has to fit on the device. Sizes are read from the artifacts themselves
where they exist, and from the recorded metrics where the file is a local build
artifact rather than a committed one.

In [8]:
size_rows = []


def record_size(model: str, precision: str, path: Path | None, fallback_bytes=None, note: str = ""):
    if path is not None and Path(path).is_file():
        size = Path(path).stat().st_size
        source = Path(path).relative_to(REPO_ROOT).as_posix()
    elif fallback_bytes:
        size, source = int(fallback_bytes), "recorded in saved metrics"
    else:
        return
    size_rows.append({
        "model": model, "precision": precision, "bytes": size,
        "mib": size / 1024 ** 2, "source": source, "note": note,
    })


record_size("DeBERTa-v3-xsmall", "FP32 ONNX",
            MODEL_V2 / "artifacts" / "model_v2" / "models" / "fp32" / "model_fp32.onnx",
            (ACCEPTED or {}).get("model", {}).get("fp32_bytes"),
            "reference graph, not deployed")
record_size("DeBERTa-v3-xsmall", "INT8 ONNX",
            DEPLOYED_INT8,
            (ACCEPTED or {}).get("model", {}).get("int8_bytes"),
            "deployed")
if TINYBERT_IID:
    record_size("TinyBERT-4", "FP32 ONNX", None,
                TINYBERT_IID["onnx_fp32"]["model_size_mib"] * 1024 ** 2, "previous generation")
    record_size("TinyBERT-4", "INT8 ONNX",
                TINYBERT_V1 / "onnx_model" / "model_int8.onnx",
                TINYBERT_IID["onnx_int8"]["model_size_mib"] * 1024 ** 2,
                "previous deployment")
record_size("BiLSTM baseline", "FP32 PyTorch",
            REPO_ROOT / "model" / "artifacts" / "pii_masking_bilstm.pt",
            None, "first baseline")

if size_rows:
    ordered = sorted(size_rows, key=lambda row: row["mib"])
    labels = [f"{row['model']}\n{row['precision']}" for row in ordered]
    values = [row["mib"] for row in ordered]
    # One highlight, not a palette: the row labels already carry identity, so
    # colour is spent on the single thing the reader is looking for.
    colours = [
        SERIES["primary"] if row["note"] == "deployed" else "#b8b7b2" for row in ordered
    ]
    figure, axis = plt.subplots(figsize=(9.0, 4.4))
    bars = axis.barh(range(len(labels)), values, height=0.6, color=colours)
    for bar, row in zip(bars, ordered):
        axis.annotate(f"{row['mib']:.1f} MiB" + ("   deployed" if row["note"] == "deployed" else ""),
                      (row["mib"], bar.get_y() + bar.get_height() / 2),
                      va="center", ha="left", xytext=(6, 0), textcoords="offset points",
                      fontsize=9.5, color=INK if row["note"] == "deployed" else MUTED)
    axis.set_yticks(range(len(labels)), labels)
    axis.set_xlim(0, max(values) * 1.32)
    axis.set_xlabel("Artifact size on disk (MiB)")
    axis.set_title("What has to ship to the device", loc="left")
    tidy(axis, x_grid=True)
    save(figure, "model_size_comparison",
         "On-disk size of every privacy-model artifact, deployed one highlighted.")

    for row in ordered:
        print(f"{row['model']:20} {row['precision']:14} {row['mib']:>8.2f} MiB   {row['note']}")
else:
    print("skipped: no artifact sizes available")

save_table(size_rows, ["model", "precision", "bytes", "mib", "source", "note"],
           "model_size_comparison", "On-disk size of each privacy-model artifact.")

  figure  research_results/figures/model_size_comparison.png  (+ .pdf)
TinyBERT-4           INT8 ONNX         13.72 MiB   previous deployment
BiLSTM baseline      FP32 PyTorch      14.57 MiB   first baseline
TinyBERT-4           FP32 ONNX         54.49 MiB   previous generation
DeBERTa-v3-xsmall    INT8 ONNX         78.47 MiB   deployed
DeBERTa-v3-xsmall    FP32 ONNX        269.87 MiB   reference graph, not deployed
  table   research_results/tables/model_size_comparison.csv  (5 rows)


## 8. Reference latency by input length

From the workstation preflight benchmark, single-threaded and controlled. It
measures the *architecture* at four input lengths; the head was untrained, so
these numbers say nothing about quality and everything about cost.

This is a development-machine measurement of the architecture, not of the
deployment. Raspberry Pi latency comes from sections 9, 10 and 12.

In [9]:
latency_rows = []

if PREFLIGHT and "one_thread_controlled" in PREFLIGHT:
    controlled = PREFLIGHT["one_thread_controlled"]
    models = controlled["models"]
    display = {
        "deberta_fp32": ("DeBERTa-v3-xsmall FP32", SERIES["tertiary"], "--"),
        "deberta_int8": ("DeBERTa-v3-xsmall INT8", SERIES["primary"], "-"),
        "tinybert_int8": ("TinyBERT-4 INT8", SERIES["secondary"], "-"),
    }
    lengths = sorted({int(key) for entry in models.values() for key in entry})

    figure, axis = plt.subplots(figsize=(8.6, 4.8))
    for key, (name, colour, style) in display.items():
        if key not in models:
            continue
        values = [models[key][str(length)]["median_ms"] for length in lengths
                  if str(length) in models[key]]
        present = [length for length in lengths if str(length) in models[key]]
        axis.plot(present, values, marker="o", color=colour, linestyle=style, label=name)
        axis.annotate(f"{values[-1]:.0f} ms", (present[-1], values[-1]),
                      xytext=(6, 0), textcoords="offset points",
                      va="center", fontsize=9.5, color=colour)
        for length in present:
            entry = models[key][str(length)]
            latency_rows.append({
                "model": name, "tokens": length, "repetitions": entry["repetitions"],
                "mean_ms": entry["mean_ms"], "median_ms": entry["median_ms"],
                "p95_ms": entry["p95_ms"], "min_ms": entry["min_ms"], "max_ms": entry["max_ms"],
                "threads": controlled["intra_op_num_threads"],
                "host": PREFLIGHT["host"]["processor"],
            })

    axis.set_xticks(lengths)
    axis.set_xlabel("Input length (tokens)")
    axis.set_ylabel("Median single-inference latency (ms)")
    axis.set_xlim(min(lengths) - 20, max(lengths) + 90)
    axis.set_ylim(0, None)
    axis.set_title(
        "Single-threaded inference cost against input length\n"
        f"{PREFLIGHT['host']['processor']}, {controlled['repetitions']} repetitions "
        f"after {controlled['warmups']} warm-ups",
        loc="left",
    )
    axis.legend(loc="upper left")
    tidy(axis)
    save(figure, "latency_by_input_length",
         "Median single-threaded inference latency against input length for all "
         "three deployment candidates, on the development workstation.")
else:
    print("skipped: no preflight latency benchmark available")

save_table(
    latency_rows,
    ["model", "tokens", "repetitions", "mean_ms", "median_ms", "p95_ms",
     "min_ms", "max_ms", "threads", "host"],
    "latency_by_input_length",
    "Single-threaded latency at four input lengths on the development workstation.",
)

  figure  research_results/figures/latency_by_input_length.png  (+ .pdf)
  table   research_results/tables/latency_by_input_length.csv  (12 rows)


## 9. CPU deployment benchmark - the current system on both devices

What `Full_Pipeline_CPU_Benchmark.py` measured: the same 2,000 committed rows,
through the same hash-verified INT8 artifact, on the workstation and on a
Raspberry Pi 5. The figures in this section draw the **current system (V6)**
on both devices; the run table below lists every text benchmark folder in the
repository, current and historical, with the canonical ones marked.

The end-to-end audio benchmark is section 10, the device comparison is
section 12, and the assistant round trip is section 13.

In [10]:
def load_text_run(directory: Path) -> dict:
    return {
        "directory": directory,
        "run_id": directory.name,
        "pipeline_version": pipeline_version_of(directory),
        "device_class": device_class_of(directory),
        "config": load_json(directory / "benchmark_config.json", f"{directory.name} config"),
        "system": load_json(directory / "system_info.json", f"{directory.name} system"),
        "latency": load_json(directory / "deberta_latency_summary.json", f"{directory.name} latency"),
        "pipeline": load_json(directory / "pipeline_latency_summary.json", f"{directory.name} pipeline"),
        "evaluation": load_json(directory / "evaluation_metrics.json", f"{directory.name} evaluation"),
        "memory": load_json(directory / "memory_metrics.json", f"{directory.name} memory"),
        "artifacts": load_json(directory / "artifact_manifest.json", f"{directory.name} artifacts"),
    }


def engine_of(run) -> str:
    """Which recognizer a run measured.

    Runs written before V4 carry no ``stt_engine`` key. Every one of them is a
    Canary run - the key was added in the change that replaced Canary - so its
    absence is the identification rather than a guess.
    """

    return str((run["config"] or {}).get("stt_engine") or "canary").lower()


def device_name(run) -> str:
    """The public-facing device name: 'Raspberry Pi 5' or 'Workstation'."""

    return DEVICE_LABEL.get(run["device_class"], run["device_class"])


def system_name(run, *, short: bool = False) -> str:
    table = SYSTEM_SHORT if short else SYSTEM_LABEL
    return table.get(run["pipeline_version"], run["pipeline_version"])


def arm_name(run) -> str:
    """Two-line axis label: what the system is, then which device ran it."""

    return f"{SYSTEM_SHORT.get(run['pipeline_version'], run['pipeline_version'])}\n{device_name(run)}"


# Every text run on disk, for the listing; the canonical subset, for the figures.
ALL_TEXT_RUNS = [load_text_run(d) for d in DISCOVERED if benchmark_kind(d) == "cpu_text"]
ALL_TEXT_RUNS = [run for run in ALL_TEXT_RUNS if run["config"] and run["system"]]
TEXT_BY_KEY = {
    (version, device): load_text_run(path)
    for (version, _, device), path in CANONICAL_TEXT.items()
}
# The current system, workstation first so the Pi reads as the comparison.
CURRENT_TEXT = [
    TEXT_BY_KEY[key] for key in ((CURRENT_PIPELINE, "workstation"), (CURRENT_PIPELINE, "pi"))
    if key in TEXT_BY_KEY
]
CURRENT_TEXT_WS = TEXT_BY_KEY.get((CURRENT_PIPELINE, "workstation"))
CURRENT_TEXT_PI = TEXT_BY_KEY.get((CURRENT_PIPELINE, "pi"))

# A smoke run scores ~50 rows to prove the machine works, not to measure it. It
# must never share an axis with a 2,000-row run.
for run in list(CURRENT_TEXT):
    if run["config"]["mode"] == "smoke":
        CURRENT_TEXT.remove(run)
        UNAVAILABLE.append(
            f"{run['run_id']}: smoke mode ({run['config']['rows_scored']} rows), "
            "excluded from the comparison figures"
        )

if not CURRENT_TEXT:
    print(f"No canonical {CURRENT_PIPELINE} text benchmark run is available.")
else:
    print(f"{'run':62} {'system':18} {'device':16} {'mode':6} {'rows':>6} canonical")
    print("-" * 128)
    for run in ALL_TEXT_RUNS:
        print(f"{run['run_id']:62} {system_name(run, short=True):18} "
              f"{device_name(run):16} {run['config']['mode']:6} "
              f"{run['config']['rows_scored']:>6,} "
              f"{'yes' if run['run_id'] in CANONICAL_IDS else '-'}")

run_rows = [
    {
        "run_id": run["run_id"],
        "pipeline_version": run["pipeline_version"],
        "system": system_name(run, short=True),
        "device": device_name(run),
        "canonical": run["run_id"] in CANONICAL_IDS,
        "is_current_system": run["pipeline_version"] == CURRENT_PIPELINE,
        "os": run["system"].get("os"),
        "architecture": run["system"].get("architecture"),
        "board": run["system"].get("board_model"),
        "cpu": run["system"].get("cpu_model"),
        "logical_cpus": run["system"].get("logical_cpus"),
        "total_ram_gib": (run["system"].get("total_ram_bytes") or 0) / 1024 ** 3,
        "python": run["system"].get("python"),
        "onnxruntime": run["system"].get("onnxruntime"),
        "mode": run["config"]["mode"],
        "rows_scored": run["config"]["rows_scored"],
        "int8_sha256": (run["artifacts"] or {}).get("files", {})
        .get("model_int8.onnx", {}).get("sha256"),
        "typed_f1": (run["evaluation"] or {}).get("production_path", {})
        .get("typed_entity", {}).get("f1"),
        "private_character_recall": (run["evaluation"] or {}).get("production_path", {})
        .get("private_character_recall"),
        "mean_latency_ms": (run["latency"] or {}).get("end_to_end_masking_call", {}).get("mean_ms"),
        "p95_latency_ms": (run["latency"] or {}).get("end_to_end_masking_call", {}).get("p95_ms"),
        "throughput_rows_per_second": (run["latency"] or {}).get("throughput_rows_per_second"),
        "peak_rss_mib": ((run["memory"] or {}).get("process_peak_rss_bytes") or 0) / 1024 ** 2,
    }
    for run in ALL_TEXT_RUNS
]
save_table(
    run_rows,
    ["run_id", "pipeline_version", "system", "device", "canonical", "is_current_system",
     "os", "architecture", "board", "cpu", "logical_cpus", "total_ram_gib", "python",
     "onnxruntime", "mode", "rows_scored", "int8_sha256", "typed_f1",
     "private_character_recall", "mean_latency_ms", "p95_latency_ms",
     "throughput_rows_per_second", "peak_rss_mib"],
    "cpu_benchmark_runs",
    "Every CPU/text benchmark run in the repository, current and historical, with "
    "its device, environment and headline numbers. `canonical` marks the run the "
    "figures are drawn from.",
    runs=[run["run_id"] for run in ALL_TEXT_RUNS],
)

run                                                            system             device           mode     rows canonical
--------------------------------------------------------------------------------------------------------------------------------
20260817T190227Z_windows-amd64_full_workstation                Canary generation  Workstation      full    2,000 yes
20260818T082425Z_linux-aarch64_full                            Canary generation  Raspberry Pi 5   full    2,000 yes
20260819T080341Z_windows-amd64_whisper_full_workstation        Previous system    Workstation      full    2,000 -
20260820T072331Z_linux-aarch64_whisper_full                    Previous system    Raspberry Pi 5   full    2,000 -
20260820T145843Z_windows-amd64_whisper_full_workstation        Previous system    Workstation      full    2,000 yes
20260821T201941Z_linux-aarch64_whisper_full                    Previous system    Raspberry Pi 5   full    2,000 -
20260821T205958Z_linux-aarch64_whisper_full         

In [11]:
# --- latency distribution -----------------------------------------------------
def read_latency_csv(directory: Path):
    path = directory / "deberta_latency.csv"
    if not path.is_file():
        return []
    with path.open(encoding="utf-8", newline="") as stream:
        return [
            {"latency_ms": float(row["latency_ms"]), "characters": int(row["characters"])}
            for row in csv.DictReader(stream)
        ]


plotted = [(run, read_latency_csv(run["directory"])) for run in CURRENT_TEXT]
plotted = [(run, samples) for run, samples in plotted if samples]

if plotted:
    figure, (left, right) = plt.subplots(1, 2, figsize=(12.5, 4.6))
    percentile_notes = []

    for run, samples in plotted:
        values = sorted(sample["latency_ms"] for sample in samples)
        colour = DEVICE_COLOUR[run["device_class"]]
        left.hist(values, bins=40, color=colour, alpha=0.6,
                  label=f"{device_name(run)} (n={len(values):,})")
        # Empirical CDF: percentiles are read off it directly, no binning artefact.
        right.plot(values, [rank / len(values) for rank in range(1, len(values) + 1)],
                   color=colour, label=device_name(run))
        summary = (run["latency"] or {}).get("end_to_end_masking_call", {})
        marked = [(fraction, key) for fraction, key in
                  ((0.50, "p50_ms"), (0.95, "p95_ms"), (0.99, "p99_ms")) if key in summary]
        for fraction, key in marked:
            right.plot([summary[key]], [fraction], marker="o", color=colour)
        # P95 and P99 sit in the flat top of the CDF, where two devices' labels
        # land on top of each other. One readout per device instead.
        if marked:
            percentile_notes.append((
                device_name(run),
                "   ".join(f"P{int(fraction * 100)} {summary[key]:,.0f} ms"
                           for fraction, key in marked),
                colour,
            ))

    for order, (name, note, colour) in enumerate(percentile_notes):
        right.text(0.98, 0.42 - order * 0.085, f"{name}    {note}",
                   transform=right.transAxes, ha="right", va="center",
                   fontsize=8.5, color=colour)

    left.set_xlabel("Masking latency per row (ms) - lower is better")
    left.set_ylabel("Rows")
    left.set_title("Latency distribution")
    left.legend(loc="upper right")
    tidy(left)

    right.set_xlabel("Masking latency per row (ms) - lower is better")
    right.set_ylabel("Share of rows at or below")
    right.set_ylim(0, 1.02)
    right.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    right.set_title("Cumulative distribution, with P50 / P95 / P99")
    if len(plotted) > 1:
        right.legend(loc="lower right")
    tidy(right)

    figure.suptitle(
        f"Privacy masking latency on CPU - {SYSTEM_LABEL[CURRENT_PIPELINE]}\n"
        f"{plotted[0][0]['config']['rows_scored']:,} committed rows, identical INT8 artifact on both devices",
        fontsize=12.5, y=1.05)
    save(figure, "cpu_latency_distribution",
         "Per-row masking latency distribution and cumulative distribution for the "
         "current system on the workstation and on a Raspberry Pi 5.",
         runs=[run["run_id"] for run, _ in plotted])

    percentile_rows = [
        {"run_id": run["run_id"], "system": system_name(run, short=True),
         "device": device_name(run),
         **{key: value for key, value in
            (run["latency"] or {}).get("end_to_end_masking_call", {}).items()}}
        for run, _ in plotted
    ]
    save_table(
        percentile_rows,
        ["run_id", "system", "device", "count", "mean_ms", "std_ms", "min_ms", "p50_ms",
         "p90_ms", "p95_ms", "p99_ms", "max_ms", "total_ms"],
        "cpu_latency_percentiles",
        "Latency percentiles of the full masking call, current system, per device.",
        runs=[run["run_id"] for run, _ in plotted],
    )
else:
    print("no per-row latency samples in the canonical current-system runs")

  figure  research_results/figures/cpu_latency_distribution.png  (+ .pdf)  <- 2 run(s)
  table   research_results/tables/cpu_latency_percentiles.csv  (2 rows)


In [12]:
# --- stage breakdown ----------------------------------------------------------
# Drawn on the Raspberry Pi 5 because that is the deployment target; the
# workstation numbers are in the table beside it. Neither is chosen by recency.
STAGE_RUN = CURRENT_TEXT_PI or CURRENT_TEXT_WS
PIPELINE_STAGE_LABELS = {
    "vad_silero": "Silero VAD (3 s synthetic probe)",
    "stt_canary": "Canary STT (3 s synthetic probe)",
    "stt_whisper": "Whisper STT (3 s synthetic probe)",
    "stt_parakeet": "Parakeet STT (3 s synthetic probe)",
    "transcript_merge": "Transcript merge",
    "wake_word_normalisation": "Wake-word normalisation",
    "inactivity_buffer": "Inactivity buffer",
    "privacy_masking": "Privacy masking",
    "digit_normalisation": "Spoken-digit rendering",
}


def stage_values_of(run):
    """(inside one masking call, across the assembled pipeline, unmeasurable)."""

    inside, latency = {}, (run["latency"] or {})
    for name, key in (("Neural decode (ONNX + span consolidation)", "neural_decode"),
                      ("Structured regex layer", "structured"),
                      ("Span merge and mask policy", "span_merge")):
        entry = latency.get("stages", {}).get(key)
        if entry and entry.get("count"):
            inside[name] = entry["mean_ms"]

    pipeline = (run["pipeline"] or {}).get("stages", {})
    across, missing = {}, []
    for key, label in PIPELINE_STAGE_LABELS.items():
        entry = pipeline.get(key)
        if not entry:
            continue
        if entry.get("available") and entry.get("count"):
            across[label] = entry["mean_ms"]
        else:
            missing.append(label)
    return inside, across, missing


if STAGE_RUN:
    stage_values, pipeline_values, pipeline_missing = stage_values_of(STAGE_RUN)

    if stage_values or pipeline_values:
        panels = int(bool(stage_values)) + int(bool(pipeline_values))
        figure, axes = plt.subplots(panels, 1, figsize=(9.6, 3.0 * panels + 1.0))
        axes = [axes] if panels == 1 else list(axes)

        for axis, (title, values) in zip(
            axes,
            [item for item in (("Inside one masking call", stage_values),
                               ("Across the assembled pipeline", pipeline_values)) if item[1]],
        ):
            names = list(values)
            magnitudes = [values[name] for name in names]
            bars = axis.barh(range(len(names)), magnitudes, height=0.6,
                             color=SERIES["primary"])
            for bar, value in zip(bars, magnitudes):
                axis.annotate(f"{value:.2f} ms", (value, bar.get_y() + bar.get_height() / 2),
                              va="center", ha="left", xytext=(6, 0),
                              textcoords="offset points", fontsize=9.5, color=INK)
            axis.set_yticks(range(len(names)), names)
            # Linear only. A log axis would make the sub-millisecond text stages
            # look comparable to the neural decode, which is the opposite of the
            # point; the annotations carry the small values instead.
            axis.set_xlim(0, max(magnitudes) * 1.30)
            axis.set_xlabel("Mean latency (ms) - lower is better")
            axis.set_title(title, loc="left")
            axis.invert_yaxis()
            tidy(axis, x_grid=True)

        subtitle = (f"{SYSTEM_LABEL[STAGE_RUN['pipeline_version']]} on "
                    f"{device_name(STAGE_RUN)} - {STAGE_RUN['config']['mode']} run")
        if pipeline_missing:
            subtitle += f"\nnot measurable on this device: {', '.join(pipeline_missing)}"
        figure.suptitle(f"Where the time goes\n{subtitle}", fontsize=12.5, y=1.0)
        figure.tight_layout()
        save(figure, "pipeline_latency_breakdown",
             "Mean latency of each stage inside a masking call and across the "
             "assembled pipeline, for the current system on the Raspberry Pi 5. The "
             "two audio stages are driven by a synthesised probe tone rather than "
             "speech; the real-speech cost is in the audio benchmark section.",
             runs=[STAGE_RUN["run_id"]])

        # The table carries both devices, so the workstation numbers are not lost.
        breakdown_rows = []
        for run in CURRENT_TEXT:
            inside, across, _ = stage_values_of(run)
            breakdown_rows += [
                {"run_id": run["run_id"], "system": system_name(run, short=True),
                 "device": device_name(run), "scope": scope, "stage": name, "mean_ms": value}
                for scope, values in (("masking_call", inside), ("pipeline", across))
                for name, value in values.items()
            ]
        save_table(
            breakdown_rows,
            ["run_id", "system", "device", "scope", "stage", "mean_ms"],
            "pipeline_latency_breakdown",
            "Mean latency per stage for the current system on both devices. The "
            "figure draws the Raspberry Pi rows.",
            runs=[run["run_id"] for run in CURRENT_TEXT],
        )
    else:
        print("no stage latency recorded in the canonical current-system run")
else:
    print("no canonical current-system text run to break down")

  figure  research_results/figures/pipeline_latency_breakdown.png  (+ .pdf)  <- 1 run(s)
  table   research_results/tables/pipeline_latency_breakdown.csv  (20 rows)


In [13]:
# --- memory and throughput ----------------------------------------------------
memory_runs = [run for run in CURRENT_TEXT
               if (run["memory"] or {}).get("process_peak_rss_bytes")]

if memory_runs:
    figure, (left, right) = plt.subplots(1, 2, figsize=(11.5, 4.3))
    names = [device_name(run) for run in memory_runs]
    positions = range(len(names))

    peak = [(run["memory"]["process_peak_rss_bytes"]) / 1024 ** 2 for run in memory_runs]
    stack = [
        (run["memory"].get("privacy_stack_resident_bytes") or 0) / 1024 ** 2
        for run in memory_runs
    ]
    width = 0.36
    peak_bars = left.bar([p - width / 2 for p in positions], peak, width=width,
                         color=SERIES["primary"], label="Process peak RSS")
    stack_bars = left.bar([p + width / 2 for p in positions], stack, width=width,
                          color=SERIES["tertiary"], label="Privacy stack resident")
    for group, values in ((peak_bars, peak), (stack_bars, stack)):
        for bar, value in zip(group, values):
            left.annotate(f"{value:,.0f}", (bar.get_x() + bar.get_width() / 2, value),
                          ha="center", va="bottom", xytext=(0, 3),
                          textcoords="offset points", fontsize=9, color=INK)
    for index, run in enumerate(memory_runs):
        total = run["system"].get("total_ram_bytes")
        if total:
            left.annotate(f"{total / 1024 ** 3:.1f} GiB RAM", (index, 0),
                          ha="center", va="top", xytext=(0, -18),
                          textcoords="offset points", fontsize=8.5, color=MUTED)
    left.set_xticks(list(positions), names)
    left.set_ylabel("MiB (lower is better)")
    left.set_ylim(0, max(peak) * 1.25)
    left.set_title("Memory footprint")
    left.legend(loc="upper left")
    tidy(left)

    throughput = [
        (run["latency"] or {}).get("throughput_rows_per_second") or 0 for run in memory_runs
    ]
    bars = right.bar(list(positions), throughput, width=0.5,
                     color=[DEVICE_COLOUR[run["device_class"]] for run in memory_runs])
    for bar, value in zip(bars, throughput):
        right.annotate(f"{value:.2f}", (bar.get_x() + bar.get_width() / 2, value),
                       ha="center", va="bottom", xytext=(0, 3),
                       textcoords="offset points", fontsize=9.5, color=INK)
    right.set_xticks(list(positions), names)
    right.set_ylabel("Rows masked per second (higher is better)")
    right.set_ylim(0, max(throughput) * 1.25 if throughput else 1)
    right.set_title("Throughput")
    tidy(right)

    figure.suptitle(
        f"Deployment resource use - {SYSTEM_LABEL[CURRENT_PIPELINE]}",
        fontsize=13, y=1.02)
    save(figure, "cpu_memory_and_throughput",
         "Peak process RSS, resident privacy-stack cost and masking throughput for "
         "the current system on the workstation and on a Raspberry Pi 5.",
         runs=[run["run_id"] for run in memory_runs])

    save_table(
        [{"run_id": run["run_id"], "system": system_name(run, short=True),
          "device": device_name(run),
          "process_rss_mib": (run["memory"].get("process_rss_bytes") or 0) / 1024 ** 2,
          "process_peak_rss_mib": run["memory"]["process_peak_rss_bytes"] / 1024 ** 2,
          "privacy_stack_resident_mib":
              (run["memory"].get("privacy_stack_resident_bytes") or 0) / 1024 ** 2,
          "system_ram_gib": (run["system"].get("total_ram_bytes") or 0) / 1024 ** 3,
          "throughput_rows_per_second":
              (run["latency"] or {}).get("throughput_rows_per_second")}
         for run in memory_runs],
        ["run_id", "system", "device", "process_rss_mib", "process_peak_rss_mib",
         "privacy_stack_resident_mib", "system_ram_gib", "throughput_rows_per_second"],
        "cpu_memory_and_throughput",
        "Memory footprint and masking throughput of the current system, per device.",
        runs=[run["run_id"] for run in memory_runs],
    )
else:
    print("no memory metrics in the canonical current-system runs")

  figure  research_results/figures/cpu_memory_and_throughput.png  (+ .pdf)  <- 2 run(s)
  table   research_results/tables/cpu_memory_and_throughput.csv  (2 rows)


In [14]:
# --- cross-device parity ------------------------------------------------------
# The workstation is always the reference and the Pi the comparison, so the sign
# of every delta means the same thing in every run of this notebook.
def read_parity(directory: Path, column: str = "production_digest") -> dict:
    path = directory / "prediction_parity_data.jsonl"
    if not path.is_file():
        return {}
    records = {}
    with path.open(encoding="utf-8") as stream:
        for line in stream:
            if line.strip():
                record = json.loads(line)
                records[record["row_id"]] = record[column]
    return records


def read_ood_parity(directory: Path) -> dict:
    path = directory / "ood_dev_parity_data.jsonl"
    if not path.is_file():
        return {}
    records = {}
    with path.open(encoding="utf-8") as stream:
        for line in stream:
            if line.strip():
                record = json.loads(line)
                records[record["case_id"]] = record["digest"]
    return records


parity_rows = []
if CURRENT_TEXT_WS and CURRENT_TEXT_PI:
    reference, other = CURRENT_TEXT_WS, CURRENT_TEXT_PI
    left_metrics = (reference["evaluation"] or {}).get("production_path") or {}
    right_metrics = (other["evaluation"] or {}).get("production_path") or {}

    def agreement(loader, label):
        a, b = loader(reference["directory"]), loader(other["directory"])
        shared = sorted(set(a) & set(b))
        agree = sum(1 for key in shared if a[key] == b[key])
        return {"scope": label, "shared": len(shared), "identical": agree,
                "rate": agree / len(shared) if shared else None}

    scopes = [
        agreement(read_parity, "Predicted spans\n(2,000 benchmark rows)"),
        agreement(lambda d: read_parity(d, "masked_text_sha256"),
                  "Released masked text\n(2,000 benchmark rows)"),
        agreement(read_ood_parity, "Predicted spans\n(360 OOD dev cases)"),
    ]
    scopes = [scope for scope in scopes if scope["shared"]]

    parity_rows = [
        {"reference_run": reference["run_id"], "reference_device": device_name(reference),
         "comparison_run": other["run_id"], "comparison_device": device_name(other),
         "pipeline_version": other["pipeline_version"],
         "scope": scope["scope"].replace("\n", " "),
         "shared_items": scope["shared"], "identical": scope["identical"],
         "agreement": scope["rate"],
         "reference_typed_f1": left_metrics.get("typed_entity", {}).get("f1"),
         "comparison_typed_f1": right_metrics.get("typed_entity", {}).get("f1"),
         "typed_f1_delta": (right_metrics.get("typed_entity", {}).get("f1") or 0)
         - (left_metrics.get("typed_entity", {}).get("f1") or 0),
         "reference_character_recall": left_metrics.get("private_character_recall"),
         "comparison_character_recall": right_metrics.get("private_character_recall"),
         "character_recall_delta": (right_metrics.get("private_character_recall") or 0)
         - (left_metrics.get("private_character_recall") or 0),
         "same_model_hash": (reference["artifacts"] or {}).get("files", {})
         .get("model_int8.onnx", {}).get("sha256")
         == (other["artifacts"] or {}).get("files", {})
         .get("model_int8.onnx", {}).get("sha256")}
        for scope in scopes
    ]

    figure, (left, right) = plt.subplots(1, 2, figsize=(12.4, 4.6))
    labels = [scope["scope"] for scope in scopes]
    rates = [scope["rate"] for scope in scopes]
    positions = range(len(labels))
    bars = left.bar(list(positions), rates, width=0.5, color=SERIES["primary"])
    for bar, scope in zip(bars, scopes):
        left.annotate(f"{scope['rate']:.2%}\n{scope['identical']:,}/{scope['shared']:,}",
                      (bar.get_x() + bar.get_width() / 2, scope["rate"]),
                      ha="center", va="bottom", xytext=(0, 4),
                      textcoords="offset points", fontsize=9, color=INK)
    left.set_xticks(list(positions), labels, fontsize=9)
    left.set_ylim(0, 1.16)
    left.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    left.set_ylabel("Identical on both devices (higher is better)")
    left.set_title("Bit-level agreement across devices", loc="left")
    tidy(left)

    metric_names = ["Typed entity F1", "Private character recall"]
    deltas = [parity_rows[0]["typed_f1_delta"], parity_rows[0]["character_recall_delta"]]
    positions = range(len(metric_names))
    bars = right.bar(list(positions), deltas, width=0.4,
                     color=[SERIES["primary"] if abs(value) < 1e-9 else SERIES["secondary"]
                            for value in deltas])
    for bar, value in zip(bars, deltas):
        right.annotate(f"{value:+.6f}", (bar.get_x() + bar.get_width() / 2, value),
                       ha="center", va="bottom" if value >= 0 else "top",
                       xytext=(0, 4 if value >= 0 else -4),
                       textcoords="offset points", fontsize=9.5, color=INK)
    right.axhline(0, color=MUTED, linewidth=1.0)
    right.set_xticks(list(positions), metric_names)
    right.set_ylabel("Raspberry Pi 5 minus Workstation")
    right.set_title("Quality difference across devices", loc="left")
    limit = max(0.001, max(abs(value) for value in deltas) * 2.2)
    right.set_ylim(-limit, limit)
    tidy(right)

    same_hash = parity_rows[0]["same_model_hash"]
    figure.suptitle(
        f"Same INT8 artifact, same rows, different CPU - "
        f"{SYSTEM_LABEL[other['pipeline_version']]}\n"
        f"identical model SHA-256 on both devices: {'yes' if same_hash else 'NO'}",
        fontsize=12.5, y=1.05)
    save(figure, "deployment_parity",
         "Cross-device agreement and quality difference for the current system "
         "running the identical INT8 artifact on the workstation and a Raspberry "
         "Pi 5. Disagreement is INT8 kernel arithmetic differing between x86-64 "
         "and aarch64, not a different model.",
         runs=[reference["run_id"], other["run_id"]])

    print(f"{device_name(reference)} vs {device_name(other)}  "
          f"({SYSTEM_LABEL[other['pipeline_version']]})")
    for scope in scopes:
        print(f"  {scope['scope'].replace(chr(10), ' '):46} "
              f"{scope['identical']:>5,}/{scope['shared']:<5,} {scope['rate']:.4%}")
    print(f"  typed F1 delta          {parity_rows[0]['typed_f1_delta']:+.6f}")
    print(f"  character recall delta  {parity_rows[0]['character_recall_delta']:+.6f}")
    print(f"  same model hash         {same_hash}")
else:
    print("cross-device parity needs the canonical current-system run on both devices")

save_table(
    parity_rows,
    ["reference_run", "reference_device", "comparison_run", "comparison_device",
     "pipeline_version", "scope", "shared_items", "identical", "agreement",
     "reference_typed_f1", "comparison_typed_f1", "typed_f1_delta",
     "reference_character_recall", "comparison_character_recall",
     "character_recall_delta", "same_model_hash"],
    "deployment_parity",
    "Cross-device agreement and metric deltas for the current system, workstation "
    "as reference.",
    runs=[run["run_id"] for run in CURRENT_TEXT],
)

  figure  research_results/figures/deployment_parity.png  (+ .pdf)  <- 2 run(s)
Workstation vs Raspberry Pi 5  (Current system (V6, Whisper base.en))
  Predicted spans (2,000 benchmark rows)         1,967/2,000 98.3500%
  Released masked text (2,000 benchmark rows)    1,971/2,000 98.5500%
  Predicted spans (360 OOD dev cases)              348/360   96.6667%
  typed F1 delta          -0.000389
  character recall delta  -0.000054
  same model hash         True
  table   research_results/tables/deployment_parity.csv  (3 rows)


## 10. Audio pipeline deployment benchmark - the current system

What `Full_Pipeline_Audio_Benchmark.py` measured: 44 committed WAV clips driven
through the real deployment, from Silero VAD and the speech recognizer to the
DeBERTa INT8 masker and the post-mask text.

The figures in this section draw the **current system (V6) on the Raspberry
Pi 5**, because that is the deployment target and the harder of the two
devices. The workstation numbers are in every table beside it, and section 12
puts the two side by side. Section 11 compares recognizer generations and keeps
the Canary-era deployment evidence.

This is a **controlled deployment benchmark on a few dozen clips**, not a
speech-recognition benchmark, and it does not replace the held-out text
evaluation in the sections above. The privacy clips are a **synthetic privacy
speech benchmark** - text-to-speech reading sentences written for this project,
with entirely fictional personal information - and are reported separately from
the LibriSpeech dev-clean clips wherever the distinction matters.

In [15]:
# --- load the canonical audio runs ---------------------------------------------
# One implementation of the entity-level privacy metrics, applied to every saved
# folder regardless of when it was written. Loads no model and no recognizer.
from privacy_eval import rescore as _rescoring


def read_audio_clips(directory: Path) -> list:
    """Per-clip rows, with the numeric columns parsed and blanks left as None."""

    path = directory / "audio_clip_results.csv"
    if not path.is_file():
        return []
    numeric = {
        "audio_duration_s", "expected_entities", "predicted_entities", "wer",
        "audio_load_latency_ms", "vad_latency_ms", "stt_latency_ms",
        "normalization_latency_ms", "privacy_latency_ms", "postprocessing_latency_ms",
        "end_to_end_latency_ms", "stt_real_time_factor", "pipeline_real_time_factor",
        "vad_speech_frame_ratio", "peak_rss_mb",
    }
    rows = []
    with path.open(encoding="utf-8", newline="") as stream:
        for row in csv.DictReader(stream):
            parsed = dict(row)
            for column in numeric:
                value = row.get(column, "")
                parsed[column] = float(value) if value not in ("", None) else None
            rows.append(parsed)
    return rows


def load_audio_run(directory: Path) -> dict:
    run = {
        "directory": directory,
        "run_id": directory.name,
        "pipeline_version": pipeline_version_of(directory),
        "device_class": device_class_of(directory),
        "config": load_json(directory / "benchmark_config.json", f"{directory.name} config"),
        "system": load_json(directory / "system_info.json", f"{directory.name} system"),
        "latency": load_json(directory / "audio_latency_summary.json", f"{directory.name} audio latency"),
        "stt": load_json(directory / "audio_stt_metrics.json", f"{directory.name} audio STT"),
        "privacy": load_json(directory / "audio_privacy_metrics.json", f"{directory.name} audio privacy"),
        "memory": load_json(directory / "audio_memory_metrics.json", f"{directory.name} audio memory"),
    }
    run["clips"] = read_audio_clips(directory)
    return run


AUDIO_BY_KEY = {
    (version, device): load_audio_run(path)
    for (version, _, device), path in CANONICAL_AUDIO.items()
}
AUDIO_BY_KEY = {k: v for k, v in AUDIO_BY_KEY.items() if v["config"] and v["system"]}
AUDIO_RUNS = [AUDIO_BY_KEY[key] for key in sorted(AUDIO_BY_KEY, reverse=True)]
CURRENT_AUDIO_WS = AUDIO_BY_KEY.get((CURRENT_PIPELINE, "workstation"))
CURRENT_AUDIO_PI = AUDIO_BY_KEY.get((CURRENT_PIPELINE, "pi"))
CURRENT_AUDIO = [run for run in (CURRENT_AUDIO_WS, CURRENT_AUDIO_PI) if run]
# Every single-device figure in this section is drawn on the deployment target.
AUDIO_FOCUS = CURRENT_AUDIO_PI or CURRENT_AUDIO_WS

if not AUDIO_RUNS:
    print("No canonical audio benchmark run is available.")
else:
    print(f"{'run':62} {'system':18} {'device':16} {'clips':>5} {'STT':>4} "
          f"{'WER':>7} {'RTF':>6}")
    print("-" * 126)
    for run in AUDIO_RUNS:
        stt = run["stt"] or {}
        latency = run["latency"] or {}
        wer = (stt.get("overall") or {}).get("aggregate_wer")
        # The real-time factor block reuses a statistics helper whose field
        # names all end in _ms. These values are ratios, not milliseconds.
        rtf = ((latency.get("real_time_factor") or {}).get("stt") or {}).get("mean_ms")
        print(f"{run['run_id']:62} {system_name(run, short=True):18} "
              f"{device_name(run):16} {run['config'].get('clips_scored', 0):>5} "
              f"{'yes' if run['config'].get('stt_available') else 'no':>4} "
              f"{'-' if wer is None else format(wer, '.4f'):>7} "
              f"{'-' if rtf is None else format(rtf, '.3f'):>6}")
    if AUDIO_FOCUS:
        print()
        print(f"section 10 figures are drawn from : {AUDIO_FOCUS['run_id']}")

# --- the deployment table -----------------------------------------------------
# A run that never reached the recognizer measured no word error rate, no
# real-time factor and no end-to-end privacy. The zeros it records are the
# absence of a measurement, not a result, so it stays in the listing above and
# out of the deployment table.
deployment_rows = []
for run in AUDIO_RUNS:
    if not run["config"].get("stt_available"):
        reason = (run["stt"] or {}).get("reason") or "reason not recorded"
        UNAVAILABLE.append(
            f"{run['run_id']}: speech recognizer unavailable ({reason[:120]}), "
            "listed in the run table but excluded from the deployment summary"
        )
        continue
    latency = run["latency"] or {}
    stages = latency.get("stages") or {}
    rtf = latency.get("real_time_factor") or {}
    stt = ((run["stt"] or {}).get("overall") or {})
    normalised = ((run["stt"] or {}).get("number_normalised") or {})
    rescored = _rescoring.rescore_run(run["directory"]) or {}
    durations = [clip["audio_duration_s"] for clip in run["clips"]
                 if clip["audio_duration_s"] is not None]
    deployment_rows.append({
        "run_id": run["run_id"],
        "pipeline_version": run["pipeline_version"],
        "system": system_name(run, short=True),
        "is_current_system": run["pipeline_version"] == CURRENT_PIPELINE,
        "device": device_name(run),
        "stt_engine": engine_of(run),
        "stt_model": run["config"].get("stt_model") or (run["stt"] or {}).get("model"),
        "architecture": run["system"].get("architecture"),
        "audio_clips": run["config"].get("clips_scored"),
        "mean_audio_duration_s": statistics.fmean(durations) if durations else None,
        "total_audio_seconds": sum(durations) if durations else None,
        "aggregate_wer": stt.get("aggregate_wer"),
        "aggregate_wer_number_normalised": normalised.get("aggregate_wer"),
        "mean_stt_rtf": (rtf.get("stt") or {}).get("mean_ms"),
        "p90_stt_rtf": (rtf.get("stt") or {}).get("p90_ms"),
        "mean_pipeline_rtf": (rtf.get("pipeline") or {}).get("mean_ms"),
        "p90_pipeline_rtf": (rtf.get("pipeline") or {}).get("p90_ms"),
        "p50_end_to_end_ms": (stages.get("end_to_end") or {}).get("p50_ms"),
        "p95_end_to_end_ms": (stages.get("end_to_end") or {}).get("p95_ms"),
        # None, not zero: two historical folders never recorded a peak.
        "peak_rss_mib": (
            (run["memory"] or {}).get("process_peak_rss_bytes") / 1024 ** 2
            if (run["memory"] or {}).get("process_peak_rss_bytes") else None
        ),
        # Recomputed on the current scoring definition for every run, so a
        # folder written before the alignment fix is not compared against one
        # written after it. See privacy_eval/rescore.py.
        "entity_recall_after_stt": rescored.get("entity_recall_after_stt"),
        "entities_protected": rescored.get("entities_protected"),
        "entity_protection_rate": rescored.get("entity_protection_rate"),
        "character_recall_after_stt": rescored.get("character_recall_after_stt"),
        "character_leakage_after_stt": rescored.get("character_leakage_after_stt"),
        "fully_sanitized_clip_rate": rescored.get("fully_sanitized_clip_rate"),
        "pipeline_has_spoken_form_stage": bool(run["config"].get("spoken_form_normalisation")),
        # Spoken numbers the display renderer rebuilt into numerals after the
        # masker had already released the text. None means the run predates the
        # measurement, which is not the same as zero.
        "identifiers_rebuilt_after_masking": (
            (run["privacy"] or {}).get("postprocessing_amplification") or {}
        ).get("identifiers_reconstructed"),
    })

save_table(
    deployment_rows,
    ["run_id", "pipeline_version", "system", "is_current_system", "device",
     "stt_engine", "stt_model", "architecture", "audio_clips",
     "mean_audio_duration_s", "total_audio_seconds", "aggregate_wer",
     "aggregate_wer_number_normalised", "mean_stt_rtf", "p90_stt_rtf",
     "mean_pipeline_rtf", "p90_pipeline_rtf", "p50_end_to_end_ms",
     "p95_end_to_end_ms", "peak_rss_mib", "entity_recall_after_stt",
     "entities_protected", "entity_protection_rate", "character_recall_after_stt",
     "character_leakage_after_stt", "fully_sanitized_clip_rate",
     "pipeline_has_spoken_form_stage", "identifiers_rebuilt_after_masking"],
    "audio_deployment_summary",
    "One row per canonical audio run: 44 clips end to end, with word error rate, "
    "real-time factor, latency, memory and end-to-end privacy. The privacy "
    "columns are recomputed for every run on the current scoring definition by "
    "privacy_eval.rescore, so a folder written before the alignment fix is not "
    "compared against one written after it. `aggregate_wer` is each run's own raw "
    "figure and is not comparable between recognizers; "
    "`aggregate_wer_number_normalised` is.",
    runs=[run["run_id"] for run in AUDIO_RUNS],
)

run                                                            system             device           clips  STT     WER    RTF
------------------------------------------------------------------------------------------------------------------------------
20260823T142306Z_windows-amd64_audio-v6-whisper_full_workstation Current system     Workstation         44  yes  0.1795  0.371
20260823T204339Z_linux-aarch64_audio-v6-whisper_full           Current system     Raspberry Pi 5      44  yes  0.1841  0.617
20260823T095553Z_windows-amd64_audio-parakeet_full             Parakeet demo      Workstation         44  yes  0.0644  0.114
20260820T145659Z_windows-amd64_audio-whisper_full_workstation  Previous system    Workstation         44  yes  0.1795  0.360
20260821T201653Z_linux-aarch64_audio-whisper_full              Previous system    Raspberry Pi 5      44  yes  0.1841  0.581
20260818T062503Z_windows-amd64_audio_full_workstation          Canary generation  Workstation         44  yes  0.0403  0.

  table   research_results/tables/audio_deployment_summary.csv  (7 rows)


In [16]:
# --- which stage dominates the runtime ----------------------------------------
STAGE_LABELS = {
    "audio_load": "Audio load",
    "vad": "Silero VAD",
    "stt": "Speech recognition",
    "normalization": "Transcript merge / normalisation",
    "privacy": "DeBERTa INT8 masking",
    "postprocessing": "Spoken-digit rendering",
}

if AUDIO_FOCUS and (AUDIO_FOCUS["latency"] or {}).get("stages"):
    stages = AUDIO_FOCUS["latency"]["stages"]
    shares = AUDIO_FOCUS["latency"].get("stage_share_of_end_to_end") or {}
    measured = [(label, stages[key]["mean_ms"], shares.get(key, 0.0))
                for key, label in STAGE_LABELS.items()
                if stages.get(key, {}).get("count")]

    if measured:
        figure, (left, right) = plt.subplots(1, 2, figsize=(12.8, 4.4))
        names = [item[0] for item in measured]
        means = [item[1] for item in measured]
        fractions = [item[2] for item in measured]

        bars = left.barh(range(len(names)), means, height=0.6, color=SERIES["primary"])
        for bar, value in zip(bars, means):
            left.annotate(f"{value:,.1f} ms", (value, bar.get_y() + bar.get_height() / 2),
                          va="center", ha="left", xytext=(6, 0),
                          textcoords="offset points", fontsize=9.5, color=INK)
        left.set_yticks(range(len(names)), names)
        left.set_xlim(0, max(means) * 1.32)
        left.set_xlabel("Mean latency per clip (ms) - lower is better")
        left.set_title("Mean cost per stage", loc="left")
        left.invert_yaxis()
        tidy(left, x_grid=True)

        bottom = 0.0
        palette = [SERIES["primary"], SERIES["secondary"], SERIES["tertiary"],
                   PURPLE, "#c9a227", MUTED]
        for index, (name, fraction) in enumerate(zip(names, fractions)):
            right.bar(0, fraction, bottom=bottom, width=0.5,
                      color=palette[index % len(palette)], label=name)
            if fraction > 0.03:
                right.annotate(f"{fraction:.1%}", (0, bottom + fraction / 2),
                               ha="center", va="center", fontsize=9.5, color="white")
            bottom += fraction
        right.set_xlim(-0.6, 1.9)
        right.set_xticks([])
        right.set_ylim(0, 1.0)
        right.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
        right.set_ylabel("Share of measured processing time")
        right.set_title("Where the time goes", loc="left")
        right.legend(loc="center right", fontsize=9)
        tidy(right)

        dominant = max(measured, key=lambda item: item[2])
        figure.suptitle(
            f"Full audio pipeline, stage by stage\n"
            f"{SYSTEM_LABEL[AUDIO_FOCUS['pipeline_version']]} on {device_name(AUDIO_FOCUS)} - "
            f"{dominant[0].lower()} dominates at {dominant[2]:.0%} of processing time",
            fontsize=12.5, y=1.06)
        save(figure, "audio_pipeline_stage_latency",
             "Mean per-clip latency of every pipeline stage from audio to post-mask "
             "text, and each stage's share of total processing time, for the current "
             "system on the Raspberry Pi 5.",
             runs=[AUDIO_FOCUS["run_id"]])

        # Both devices in the table, so the workstation numbers are not lost.
        stage_rows = []
        for run in CURRENT_AUDIO:
            run_stages = (run["latency"] or {}).get("stages") or {}
            run_shares = (run["latency"] or {}).get("stage_share_of_end_to_end") or {}
            stage_rows += [
                {"run_id": run["run_id"], "system": system_name(run, short=True),
                 "device": device_name(run), "stage": label,
                 "mean_ms": run_stages[key]["mean_ms"],
                 "share_of_processing": run_shares.get(key, 0.0)}
                for key, label in STAGE_LABELS.items()
                if run_stages.get(key, {}).get("count")
            ]
        save_table(
            stage_rows,
            ["run_id", "system", "device", "stage", "mean_ms", "share_of_processing"],
            "audio_pipeline_stage_latency",
            "Per-stage mean latency and share of processing time for the current "
            "system on both devices. The figure draws the Raspberry Pi rows.",
            runs=[run["run_id"] for run in CURRENT_AUDIO],
        )
else:
    print("no audio stage latency in the canonical current-system run")

  figure  research_results/figures/audio_pipeline_stage_latency.png  (+ .pdf)  <- 1 run(s)
  table   research_results/tables/audio_pipeline_stage_latency.csv  (12 rows)


In [17]:
# --- how cost scales with clip length -----------------------------------------
if AUDIO_FOCUS and AUDIO_FOCUS["clips"]:
    run = AUDIO_FOCUS
    clips = run["clips"]
    subsets = sorted({clip["subset"] for clip in clips})
    colours = {name: colour for name, colour in
               zip(subsets, [SERIES["primary"], SERIES["secondary"], SERIES["tertiary"]])}
    has_stt = any(clip["stt_latency_ms"] is not None for clip in clips)

    panels = 3 if has_stt else 1
    figure, axes = plt.subplots(1, panels, figsize=(4.6 * panels + 0.6, 4.3))
    axes = [axes] if panels == 1 else list(axes)

    def scatter(axis, y_key, y_label, title, scale=1.0):
        for subset in subsets:
            points = [
                (clip["audio_duration_s"], clip[y_key] * scale)
                for clip in clips
                if clip["subset"] == subset
                and clip[y_key] is not None and clip["audio_duration_s"] is not None
            ]
            if points:
                axis.scatter([p[0] for p in points], [p[1] for p in points],
                             color=colours[subset], alpha=0.85, s=42,
                             label=subset.replace("_", " "))
        axis.set_xlabel("Audio duration (s)")
        axis.set_ylabel(y_label)
        axis.set_title(title, loc="left")
        axis.set_ylim(bottom=0)
        tidy(axis)

    if has_stt:
        scatter(axes[0], "end_to_end_latency_ms", "End-to-end latency (s)",
                "Duration against end-to-end latency", scale=1 / 1000.0)
        # A pipeline that keeps up with speech sits below this line. Drawn only
        # as far as the y axis reaches: a y = x line across a 30 s x axis and a
        # 7 s y axis renders as a near-vertical stripe and reads as an artefact.
        top = axes[0].get_ylim()[1]
        limit = min(top, max(clip["audio_duration_s"] for clip in clips))
        axes[0].plot([0, limit], [0, limit], color=MUTED, linestyle="--",
                     linewidth=1.2, label="real time (y = x)")
        axes[0].legend(loc="upper left", fontsize=9)

        scatter(axes[1], "stt_latency_ms", "STT latency (s)",
                "Duration against STT latency", scale=1 / 1000.0)

        scatter(axes[2], "stt_real_time_factor", "STT real-time factor",
                "Duration against STT RTF (lower is faster)")
        axes[2].axhline(1.0, color=SERIES["secondary"], linestyle="--", linewidth=1.3)
        axes[2].annotate("RTF = 1 (real time)", (0.02, 1.0), xycoords=("axes fraction", "data"),
                         xytext=(0, 5), textcoords="offset points",
                         fontsize=9, color=SERIES["secondary"])
    else:
        scatter(axes[0], "end_to_end_latency_ms", "End-to-end latency (s)",
                "Duration against end-to-end latency (no STT)", scale=1 / 1000.0)

    figure.suptitle(
        f"Deployment scaling with clip length - "
        f"{SYSTEM_LABEL[run['pipeline_version']]} on {device_name(run)}",
        fontsize=13, y=1.04)
    figure.tight_layout()
    save(figure, "audio_duration_scaling",
         "Audio duration against end-to-end latency, STT latency and STT real-time "
         "factor for the current system on the Raspberry Pi 5. RTF below 1 means "
         "the device processes speech faster than it arrives; points above the "
         "dashed line are clips it did not.",
         runs=[run["run_id"]])

    save_table(
        [{"run_id": one["run_id"], "system": system_name(one, short=True),
          "device": device_name(one), "clip_id": clip["clip_id"],
          "subset": clip["subset"], "duration_bucket": clip["duration_bucket"],
          "audio_duration_s": clip["audio_duration_s"],
          "stt_latency_ms": clip["stt_latency_ms"],
          "end_to_end_latency_ms": clip["end_to_end_latency_ms"],
          "stt_real_time_factor": clip["stt_real_time_factor"],
          "pipeline_real_time_factor": clip["pipeline_real_time_factor"],
          "wer": clip["wer"]}
         for one in CURRENT_AUDIO for clip in one["clips"]],
        ["run_id", "system", "device", "clip_id", "subset", "duration_bucket",
         "audio_duration_s", "stt_latency_ms", "end_to_end_latency_ms",
         "stt_real_time_factor", "pipeline_real_time_factor", "wer"],
        "audio_clip_scaling",
        "Per-clip duration, latency and real-time factor for the current system on "
        "both devices. The figure draws the Raspberry Pi rows.",
        runs=[one["run_id"] for one in CURRENT_AUDIO],
    )

    # --- duration bucket against latency --------------------------------------
    buckets = (run["latency"] or {}).get("end_to_end_by_duration_bucket") or {}
    populated = {name: entry for name, entry in buckets.items() if entry.get("count")}
    if len(populated) > 1:
        figure, axis = plt.subplots(figsize=(8.2, 4.2))

        def bucket_start(name: str) -> float:
            """Bucket keys are strings; the axis is ordinal, so sort numerically."""

            try:
                return float(name.split("-", 1)[0].rstrip("s"))
            except ValueError:
                return math.inf

        names = sorted(populated, key=bucket_start)
        positions = range(len(names))
        means = [populated[name]["mean_ms"] / 1000.0 for name in names]
        errors = [
            [(populated[name]["mean_ms"] - populated[name]["min_ms"]) / 1000.0 for name in names],
            [(populated[name]["max_ms"] - populated[name]["mean_ms"]) / 1000.0 for name in names],
        ]
        bars = axis.bar(list(positions), means, width=0.5, color=SERIES["primary"],
                        yerr=errors, capsize=5, ecolor=MUTED)
        for bar, name, value in zip(bars, names, means):
            axis.annotate(f"{value:.2f} s\nn={populated[name]['count']}",
                          (bar.get_x() + bar.get_width() / 2, value),
                          ha="center", va="bottom", xytext=(0, 16),
                          textcoords="offset points", fontsize=9, color=INK)
        axis.set_xticks(list(positions), names)
        axis.set_ylabel("End-to-end latency (s) - lower is better")
        axis.set_xlabel("Clip duration bucket")
        axis.set_title(
            f"End-to-end latency by clip length\n"
            f"{SYSTEM_LABEL[run['pipeline_version']]} on {device_name(run)}")
        # Headroom for the two-line annotation that sits above the whisker.
        axis.set_ylim(0, max(populated[name]["max_ms"] for name in names) / 1000.0 * 1.22)
        tidy(axis)
        save(figure, "audio_latency_by_duration_bucket",
             "Mean end-to-end latency per clip-duration bucket for the current "
             "system on the Raspberry Pi 5, with the observed minimum and maximum "
             "as whiskers.",
             runs=[run["run_id"]])

        save_table(
            [{"run_id": one["run_id"], "system": system_name(one, short=True),
              "device": device_name(one), "bucket": name, **entry}
             for one in CURRENT_AUDIO
             for name, entry in sorted(
                 ((n, e) for n, e in
                  ((one["latency"] or {}).get("end_to_end_by_duration_bucket") or {}).items()
                  if e.get("count")),
                 key=lambda item: bucket_start(item[0]))],
            ["run_id", "system", "device", "bucket", "count", "mean_ms", "std_ms",
             "min_ms", "p50_ms", "p90_ms", "p95_ms", "p99_ms", "max_ms", "total_ms"],
            "audio_latency_by_duration_bucket",
            "End-to-end latency statistics per clip-duration bucket, current system, "
            "both devices.",
            runs=[one["run_id"] for one in CURRENT_AUDIO],
        )
else:
    print("no per-clip audio results in the canonical current-system run")

  figure  research_results/figures/audio_duration_scaling.png  (+ .pdf)  <- 1 run(s)
  table   research_results/tables/audio_clip_scaling.csv  (88 rows)


  figure  research_results/figures/audio_latency_by_duration_bucket.png  (+ .pdf)  <- 1 run(s)
  table   research_results/tables/audio_latency_by_duration_bucket.csv  (8 rows)


In [18]:
# --- latency distribution and the memory the device has to find ---------------
if AUDIO_FOCUS and AUDIO_FOCUS["clips"]:
    run = AUDIO_FOCUS
    values = sorted(clip["end_to_end_latency_ms"] / 1000.0 for clip in run["clips"]
                    if clip["end_to_end_latency_ms"] is not None)
    summary = ((run["latency"] or {}).get("stages") or {}).get("end_to_end") or {}
    memory = run["memory"] or {}
    stage_rss = memory.get("stage_rss_bytes") or {}
    stage_order = [("baseline", "Baseline process"), ("vad_loaded", "VAD loaded"),
                   ("stt_loaded", "STT loaded"), ("privacy_loaded", "DeBERTa loaded"),
                   ("all_models_loaded", "All models loaded")]
    stage_points = [(label, stage_rss[key] / 1024 ** 2)
                    for key, label in stage_order if stage_rss.get(key)]
    if memory.get("process_peak_rss_bytes"):
        stage_points.append(("Peak during run",
                             memory["process_peak_rss_bytes"] / 1024 ** 2))

    panels = 1 + int(bool(stage_points))
    figure, axes = plt.subplots(1, panels, figsize=(6.2 * panels, 4.4))
    axes = [axes] if panels == 1 else list(axes)

    left = axes[0]
    if values:
        left.plot(values, [rank / len(values) for rank in range(1, len(values) + 1)],
                  color=SERIES["primary"], marker="o", markersize=4)
        for fraction, key in ((0.50, "p50_ms"), (0.90, "p90_ms"),
                              (0.95, "p95_ms"), (0.99, "p99_ms")):
            if key in summary:
                seconds = summary[key] / 1000.0
                left.plot([seconds], [fraction], marker="o", color=SERIES["secondary"])
                left.annotate(f"P{int(fraction * 100)} {seconds:.2f} s",
                              (seconds, fraction), xytext=(6, -3),
                              textcoords="offset points", fontsize=9,
                              color=SERIES["secondary"])
    left.set_xlabel("End-to-end latency per clip (s) - lower is better")
    left.set_ylabel("Share of clips at or below")
    left.set_ylim(0, 1.03)
    left.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    left.set_title(f"End-to-end latency distribution (n={len(values)})", loc="left")
    tidy(left)

    if stage_points:
        right = axes[1]
        names = [item[0] for item in stage_points]
        magnitudes = [item[1] for item in stage_points]
        colours = [SERIES["primary"]] * (len(names) - 1) + [SERIES["secondary"]]
        bars = right.bar(range(len(names)), magnitudes, width=0.55, color=colours)
        for bar, value in zip(bars, magnitudes):
            right.annotate(f"{value:,.0f}", (bar.get_x() + bar.get_width() / 2, value),
                           ha="center", va="bottom", xytext=(0, 3),
                           textcoords="offset points", fontsize=9, color=INK)
        total = run["system"].get("total_ram_bytes")
        # The deployment target is a 4 GB Pi; that budget is drawn, not assumed.
        right.axhline(4096, color=MUTED, linestyle="--", linewidth=1.2)
        right.annotate("4 GB Raspberry Pi budget", (0.02, 4096),
                       xycoords=("axes fraction", "data"), xytext=(0, 4),
                       textcoords="offset points", fontsize=9, color=MUTED)
        right.set_xticks(range(len(names)), names, rotation=20, ha="right")
        right.set_ylabel("Resident set size (MiB) - lower is better")
        right.set_ylim(0, max(4096 * 1.15, max(magnitudes) * 1.25))
        right.set_title("Memory as the stack loads"
                        + (f" - host has {total / 1024 ** 3:.1f} GiB" if total else ""),
                        loc="left")
        tidy(right)

        save_table(
            [{"run_id": one["run_id"], "system": system_name(one, short=True),
              "device": device_name(one), "stage": label,
              "rss_mib": ((one["memory"] or {}).get("stage_rss_bytes") or {})[key] / 1024 ** 2}
             for one in CURRENT_AUDIO
             for key, label in stage_order
             if ((one["memory"] or {}).get("stage_rss_bytes") or {}).get(key)]
            + [{"run_id": one["run_id"], "system": system_name(one, short=True),
                "device": device_name(one), "stage": "Peak during run",
                "rss_mib": (one["memory"] or {})["process_peak_rss_bytes"] / 1024 ** 2}
               for one in CURRENT_AUDIO
               if (one["memory"] or {}).get("process_peak_rss_bytes")],
            ["run_id", "system", "device", "stage", "rss_mib"],
            "audio_memory_breakdown",
            "Resident memory after each model loads, and the peak during the run, "
            "for the current system on both devices.",
            runs=[one["run_id"] for one in CURRENT_AUDIO],
        )

    figure.suptitle(
        f"Audio pipeline cost - {SYSTEM_LABEL[run['pipeline_version']]} on {device_name(run)}",
        fontsize=13, y=1.03)
    figure.tight_layout()
    save(figure, "audio_latency_and_memory",
         "Cumulative distribution of end-to-end latency with P50/P90/P95/P99, and "
         "resident memory as each model loads, for the current system on the "
         "Raspberry Pi 5, against the 4 GB Raspberry Pi budget.",
         runs=[run["run_id"]])
else:
    print("no audio latency or memory metrics in the canonical current-system run")

  table   research_results/tables/audio_memory_breakdown.csv  (12 rows)


  figure  research_results/figures/audio_latency_and_memory.png  (+ .pdf)  <- 1 run(s)


In [19]:
# --- privacy: reference text against the real audio pipeline -------------------
privacy_rows = []
if AUDIO_FOCUS and (AUDIO_FOCUS["privacy"] or {}).get("control_arms"):
    run = AUDIO_FOCUS
    privacy = run["privacy"]
    arms = privacy["control_arms"]
    end = privacy.get("end_to_end") or {}
    stt_ran = bool(run["config"].get("stt_available")) and end.get("expected_entities")

    ARM_LABELS = {
        "reference_text": "Reference text\n(as written)",
        "reference_uncased": "Reference text\n(uncased, unpunctuated)",
    }
    # The "comparable" block scores each gold entity over its own characters with
    # whitespace excluded - the same definition the end-to-end arm has to use,
    # because recognized text carries no gold offsets. Plotting evaluate_spans'
    # character recall beside it would put two denominators on one axis.
    labels, recall, leakage = [], [], []
    for key, label in ARM_LABELS.items():
        if key not in arms:
            continue
        comparable = arms[key].get("comparable")
        if not comparable:
            UNAVAILABLE.append(
                f"{run['run_id']}: control arm {key} predates the comparable-metric "
                "fix; rerun Full_Pipeline_Audio_Benchmark.py"
            )
            continue
        labels.append(label)
        recall.append(comparable["character_recall"])
        leakage.append(comparable["character_leakage"])
        privacy_rows.append({
            "run_id": run["run_id"], "system": system_name(run, short=True),
            "device": device_name(run), "arm": key,
            "typed_entity_f1": arms[key]["typed_entity"]["f1"],
            "character_recall": comparable["character_recall"],
            "character_leakage": comparable["character_leakage"],
            "entity_recall": comparable["entity_recall"],
        })
    if stt_ran:
        labels.append("Audio -> STT -> privacy\n(end to end)")
        recall.append(end["character_recall_after_stt"])
        leakage.append(end["character_leakage_after_stt"])
        privacy_rows.append({
            "run_id": run["run_id"], "system": system_name(run, short=True),
            "device": device_name(run), "arm": "audio_stt_pipeline",
            "typed_entity_f1": None,
            "character_recall": end["character_recall_after_stt"],
            "character_leakage": end["character_leakage_after_stt"],
            "entity_recall": end["entity_recall_after_stt"],
        })

    attribution = privacy.get("failure_attribution") or {}
    panels = 1 + int(bool(stt_ran and sum(
        attribution.get(key, 0) for key in
        ("protected", "privacy_model_caused", "stt_caused", "stt_caused_but_still_masked")
    )))
    figure, axes = plt.subplots(1, panels, figsize=(6.4 * panels, 4.5))
    axes = [axes] if panels == 1 else list(axes)

    left = axes[0]
    positions = range(len(labels))
    width = 0.36
    recall_bars = left.bar([p - width / 2 for p in positions], recall, width=width,
                           color=SERIES["primary"], label="Private character recall (higher is better)")
    leak_bars = left.bar([p + width / 2 for p in positions], leakage, width=width,
                         color=SERIES["secondary"], label="Leakage (lower is better)")
    # Percent labels on a percent axis: decimals here would read as a second,
    # different number beside the tick labels.
    for group, values in ((recall_bars, recall), (leak_bars, leakage)):
        for bar, value in zip(group, values):
            left.annotate(f"{value:.2%}", (bar.get_x() + bar.get_width() / 2, value),
                          ha="center", va="bottom", xytext=(0, 3),
                          textcoords="offset points", fontsize=9, color=INK)
    left.set_xticks(list(positions), labels, fontsize=9)
    left.set_ylim(0, 1.18)
    left.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    left.set_ylabel("Private characters")
    left.set_title("Same clips, three routes to the privacy model", loc="left")
    left.legend(loc="upper right", fontsize=8.5)
    tidy(left)

    if panels == 2:
        right = axes[1]
        buckets = [
            ("Protected", attribution.get("protected", 0), SERIES["primary"]),
            ("STT altered,\nmasked anyway",
             attribution.get("stt_caused_but_still_masked", 0), SERIES["tertiary"]),
            ("Privacy-model\nfailure", attribution.get("privacy_model_caused", 0),
             SERIES["secondary"]),
            ("STT-caused,\nnothing masked", attribution.get("stt_caused", 0), PURPLE),
        ]
        names = [item[0] for item in buckets]
        counts = [item[1] for item in buckets]
        bars = right.bar(range(len(names)), counts, width=0.55,
                         color=[item[2] for item in buckets])
        total = sum(counts) or 1
        for bar, value in zip(bars, counts):
            right.annotate(f"{value}\n{value / total:.0%}",
                           (bar.get_x() + bar.get_width() / 2, value),
                           ha="center", va="bottom", xytext=(0, 3),
                           textcoords="offset points", fontsize=9, color=INK)
        right.set_xticks(range(len(names)), names, fontsize=9)
        right.set_ylabel("Expected private entities")
        right.set_ylim(0, max(counts) * 1.3 if counts else 1)
        right.set_title("Where end-to-end protection succeeds and fails", loc="left")
        tidy(right)

        save_table(
            [{"run_id": one["run_id"], "system": system_name(one, short=True),
              "device": device_name(one), "outcome": name.replace("\n", " "),
              "entities": ((one["privacy"] or {}).get("failure_attribution") or {}).get(key, 0),
              "share": (((one["privacy"] or {}).get("failure_attribution") or {}).get(key, 0)
                        / (sum(((one["privacy"] or {}).get("failure_attribution") or {}).get(k, 0)
                               for k in ("protected", "stt_caused_but_still_masked",
                                         "privacy_model_caused", "stt_caused")) or 1))}
             for one in CURRENT_AUDIO
             for name, key in (("Protected", "protected"),
                               ("STT altered, masked anyway", "stt_caused_but_still_masked"),
                               ("Privacy-model failure", "privacy_model_caused"),
                               ("STT-caused, nothing masked", "stt_caused"))],
            ["run_id", "system", "device", "outcome", "entities", "share"],
            "audio_failure_attribution",
            "Every expected private entity, attributed to protection, a "
            "privacy-model failure, or the speech recognizer - current system, both "
            "devices.",
            runs=[one["run_id"] for one in CURRENT_AUDIO],
        )

    # The privacy-annotated clips, not every clip in the run: the LibriSpeech
    # subset is scored for recognition only, so naming its clips here would
    # claim a privacy denominator that does not exist.
    scored_clips = len(privacy.get("privacy_scored_clips") or []) or privacy.get(
        "clips_scored", 0)
    figure.suptitle(
        "Privacy after speech recognition - synthetic privacy speech benchmark\n"
        f"{SYSTEM_LABEL[run['pipeline_version']]} on {device_name(run)}, "
        f"{scored_clips} privacy-annotated clips: a controlled deployment check, "
        "not the model's validation set",
        fontsize=12, y=1.06)
    figure.tight_layout()
    save(figure, "audio_privacy_reference_vs_pipeline",
         "Private-character recall and leakage for the same clips scored from "
         "written reference text, from uncased unpunctuated reference text, and end "
         "to end from audio; plus the attribution of every end-to-end failure. "
         "Current system on the Raspberry Pi 5.",
         runs=[run["run_id"]])

    save_table(
        privacy_rows,
        ["run_id", "system", "device", "arm", "typed_entity_f1", "entity_recall",
         "character_recall", "character_leakage"],
        "audio_privacy_arms",
        "Privacy metrics for each route to the model: written reference text, "
        "uncased unpunctuated reference text, and the real audio pipeline.",
        runs=[run["run_id"]],
    )

    # --- word error rate by subset --------------------------------------------
    by_subset = ((run["stt"] or {}).get("by_subset") or {})
    populated = {name: entry for name, entry in by_subset.items() if entry.get("clips")}
    if len(populated) > 1:
        figure, axis = plt.subplots(figsize=(7.6, 4.2))
        names = list(populated)
        positions = range(len(names))
        aggregate = [populated[name]["aggregate_wer"] for name in names]
        median = [populated[name]["median_wer"] for name in names]
        width = 0.36
        left_bars = axis.bar([p - width / 2 for p in positions], aggregate, width=width,
                             color=SERIES["primary"], label="Aggregate WER")
        right_bars = axis.bar([p + width / 2 for p in positions], median, width=width,
                              color=SERIES["tertiary"], label="Median clip WER")
        for group, values in ((left_bars, aggregate), (right_bars, median)):
            for bar, value in zip(group, values):
                axis.annotate(f"{value:.2%}", (bar.get_x() + bar.get_width() / 2, value),
                              ha="center", va="bottom", xytext=(0, 3),
                              textcoords="offset points", fontsize=9, color=INK)
        axis.set_xticks(list(positions),
                        [f"{name.replace('_', ' ')}\n({populated[name]['clips']} clips)"
                         for name in names])
        axis.set_ylim(0, max(aggregate + median) * 1.35)
        axis.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
        axis.set_ylabel("Word error rate (lower is better)")
        axis.set_title(
            f"Speech recognition on each subset\n"
            f"{SYSTEM_LABEL[run['pipeline_version']]} on {device_name(run)}")
        axis.legend(loc="upper right", fontsize=9)
        tidy(axis)
        save(figure, "audio_word_error_rate",
             "Word error rate on the synthetic privacy clips and on the LibriSpeech "
             "dev-clean clips, reported separately because they are different "
             "distributions. Current system on the Raspberry Pi 5.",
             runs=[run["run_id"]])

        save_table(
            [{"run_id": one["run_id"], "system": system_name(one, short=True),
              "device": device_name(one), "subset": name, **entry}
             for one in CURRENT_AUDIO
             for name, entry in (((one["stt"] or {}).get("by_subset") or {}).items())
             if entry.get("clips")],
            ["run_id", "system", "device", "subset", "clips", "reference_words",
             "aggregate_wer", "mean_wer", "median_wer", "min_wer", "max_wer",
             "substitutions", "deletions", "insertions"],
            "audio_word_error_rate",
            "Word error rate per fixture subset, current system, both devices.",
            runs=[one["run_id"] for one in CURRENT_AUDIO],
        )
else:
    print("no audio privacy metrics in the canonical current-system run")

  table   research_results/tables/audio_failure_attribution.csv  (8 rows)


  figure  research_results/figures/audio_privacy_reference_vs_pipeline.png  (+ .pdf)  <- 1 run(s)
  table   research_results/tables/audio_privacy_arms.csv  (3 rows)


  figure  research_results/figures/audio_word_error_rate.png  (+ .pdf)  <- 1 run(s)
  table   research_results/tables/audio_word_error_rate.csv  (4 rows)


In [20]:
# --- what the render step rebuilt after masking --------------------------------
# Canary ran with pnc=no, so a spoken number reached the model as digit words.
# Where the model missed one, the display renderer - which runs after masking by
# design - rewrote the surviving words into a numeral. That leak is already
# counted as a privacy-model failure above, so it is reported beside the
# attribution rather than added to it: one entity, counted once.
#
# Whisper writes numerals itself, so from V4 onwards there is nothing left for
# the renderer to assemble and the count is zero. That is a change of
# recognizer, not a fix: the renderer is unchanged and would do the same to any
# digit-word run reaching it. Every generation is therefore shown, and a zero is
# drawn as a zero.
amplification_rows = []
amplification_by_run = {}
for run in AUDIO_RUNS:
    block = (run["privacy"] or {}).get("postprocessing_amplification")
    if block is None:
        if (run["privacy"] or {}).get("end_to_end"):
            UNAVAILABLE.append(
                f"{run['run_id']}: predates the postprocessing-amplification "
                "measurement; rerun Full_Pipeline_Audio_Benchmark.py"
            )
        continue
    amplification_by_run[run["run_id"]] = (run, block)
    for case in block.get("cases", []):
        amplification_rows.append({
            "run_id": run["run_id"],
            "system": system_name(run, short=True),
            "pipeline_version": run["pipeline_version"],
            "device": device_name(run),
            "engine": engine_of(run),
            "clip_id": case["clip_id"],
            "subset": case["subset"],
            "unmasked_entity_labels": ", ".join(case["unmasked_entity_labels"]),
            "digits": case["digits"],
            "rendered_value": case["rendered_value"],
            "privacy_input_text": case["privacy_input_text"],
            "final_masked_output": case["final_masked_output"],
        })

if not amplification_by_run:
    print("no postprocessing-amplification measurement in the canonical runs")
else:
    for run_id, (run, block) in amplification_by_run.items():
        print(f"  {system_name(run, short=True):18} {device_name(run):16} "
              f"{engine_of(run):9} {block['identifiers_reconstructed']} rebuilt "
              f"identifier(s) on {block['clips_affected']} clip(s)")

    if amplification_rows:
        save_table(
            amplification_rows,
            ["run_id", "system", "pipeline_version", "device", "engine", "clip_id",
             "subset", "unmasked_entity_labels", "digits", "rendered_value",
             "privacy_input_text", "final_masked_output"],
            "audio_postprocessing_amplification",
            "Spoken numbers the privacy model left unmasked and the display "
            "renderer then rewrote into numerals of identifier length, per "
            "recognizer generation. Every value is fictional: telephone numbers "
            "come from the NANP 555-01xx range reserved for fiction.",
            runs=list(amplification_by_run),
        )

    # One panel: a count per generation, then the worked examples underneath.
    arms_ = sorted(amplification_by_run.values(), key=lambda item: item[0]["run_id"])
    lines = []
    for run, block in arms_:
        lines.append(
            f"{system_name(run, short=True):<18} {device_name(run):<16} "
            f"{ENGINE_LABEL[engine_of(run)]:<26} "
            f"{block['identifiers_reconstructed']} rebuilt on "
            f"{block['clips_affected']} clip(s)"
        )
    lines.append("")
    shown = set()
    for row in amplification_rows:
        key = (row["clip_id"], row["rendered_value"])
        if key in shown:
            continue
        shown.add(key)
        labels = row["unmasked_entity_labels"] or "no gold entity"
        lines.append(f"{row['clip_id']}  ({labels}, {row['engine']})")
        lines.append(f"  heard     {row['privacy_input_text']}")
        lines.append(f"  released  {row['final_masked_output']}")
        lines.append("")
    if lines and not lines[-1]:
        lines = lines[:-1]

    digits = next(iter(amplification_by_run.values()))[1]["identifier_digits"]
    LINE_INCHES = 9 * 1.5 / 72
    figure, axis = plt.subplots(figsize=(12.6, 0.55 + LINE_INCHES * len(lines) + 0.78))
    axis.axis("off")
    axis.set_title(
        "Spoken numbers rebuilt after masking, per recognizer generation",
        loc="left", fontsize=12, pad=8,
    )
    axis.text(0.0, 1.0, "\n".join(lines), va="top", ha="left", fontsize=9,
              family="monospace", linespacing=1.5, color=INK,
              transform=axis.transAxes)
    figure.text(
        0.012, 0.02,
        f"Identifier scale is {digits}+ digits, taken from the deployment's own "
        "structured recognizer. The rendering step is display-only - storage keeps "
        "the words - so this is the form the leak\ntakes on screen, not additional "
        "leaked information. Whisper writes numerals itself, so it leaves the "
        "renderer nothing to assemble; the renderer is unchanged.\nEvery value is "
        "fictional.",
        va="bottom", ha="left", fontsize=8.5, color=MUTED,
    )
    figure.tight_layout(rect=(0, 0.15, 1, 1))
    save(figure, "audio_postprocessing_amplification",
         "The spoken-digit finding: utterances where the privacy model did not mask "
         "a spoken number and the display renderer then rendered it as a numeral. "
         "Shown for every recognizer generation, because the Whisper-era count is "
         "zero by change of orthography rather than by a change to the renderer.",
         runs=list(amplification_by_run))

  Current system     Workstation      whisper   0 rebuilt identifier(s) on 0 clip(s)
  Current system     Raspberry Pi 5   whisper   0 rebuilt identifier(s) on 0 clip(s)
  Parakeet demo      Workstation      parakeet  1 rebuilt identifier(s) on 1 clip(s)
  Previous system    Workstation      whisper   0 rebuilt identifier(s) on 0 clip(s)
  Previous system    Raspberry Pi 5   whisper   0 rebuilt identifier(s) on 0 clip(s)
  Canary generation  Workstation      canary    2 rebuilt identifier(s) on 2 clip(s)
  Canary generation  Raspberry Pi 5   canary    2 rebuilt identifier(s) on 2 clip(s)
  table   research_results/tables/audio_postprocessing_amplification.csv  (5 rows)


  figure  research_results/figures/audio_postprocessing_amplification.png  (+ .pdf)  <- 7 run(s)


## 11. Recognizer generations on the same clips

Three recognizers have been measured on the same 44 committed clips, through
the same privacy pipeline, by the same harness:

* **Canary 180M Flash (V3)** - workstation and Raspberry Pi 5. A **frozen
  historical baseline**. Canary is never reloaded and never rerun; every number
  below is derived from the run folders those benchmarks wrote.
* **faster-whisper `base.en`** - workstation and Raspberry Pi 5. The Whisper
  arms below are drawn from the **current system (V6)**, because that is what
  the repository ships. V4 used the same recognizer and the same privacy core;
  its folders are in the run table as provenance and section 12 checks the two
  against each other.
* **Parakeet 110M TDT (V5)** - workstation only. Never run on the Pi, and drawn
  as pending rather than as a zero.

### Reading the word error rates

The audio benchmark's `aggregate_wer` is the standard definition: Levenshtein
over lowercased, punctuation-stripped words. Canary ran with `pnc: no` and wrote
spoken numbers as words, which is the form the reference transcripts are written
in. Whisper writes numerals - "63" where the reference says "sixty three",
"55550188" where it says "five five five, zero one eight eight". Those are
correct transcriptions and the raw metric charges eight substitutions for the
second one.

So the raw figure is **comparable across devices for one engine and not across
engines**, and every engine-to-engine chart below uses the number-normalised
figure instead: the same Levenshtein, after spoken numbers have been folded to
numerals on *both* sides, by one implementation
(`model/privacy_evaluation/privacy_eval/stt_comparison.py`) applied to every
engine's saved transcripts. It does not launder a wrong number into a match -
Whisper's inserted digit in `privacy_024` still scores as an error.

### The entity-level metrics, and why they are comparable

`entity_recall_after_stt`, `stt_preservation_rate` and the character
recall/leakage family follow each gold entity from the reference transcript into
the recognized one by aligning words. That alignment used to break on the same
spelling change, and on punctuation: Whisper's `555-0147.` matched none of the
seven reference tokens, and `Carter.` was scored at six masked characters out of
seven because the sentence full stop was counted as part of the name. Both
defects **overstated Whisper's privacy failures** and neither could occur under
Canary, which emitted no numerals and no punctuation.

`locate_entity` now folds spoken numbers on both sides and trims a located
region to its alphanumeric core. Run folders written under the new definition
record it as `entity_scoring` in `benchmark_config.json`; folders written before
it are re-scored here by `privacy_eval.rescore`, which reconstructs the
predicted spans exactly from the `privacy_input_text` / `masked_text` pair every
run saved - no recognizer, no model, and a self-check against each folder's own
recorded coverage. **Every column below is therefore one definition applied to
every arm**, and the tables say which figure is rescored.

The alignment-free measure is kept alongside it, because it depends on none of
this: after masking and after the spoken-digit renderer, is an identifier-scale
number still legible in the text the device released?

In [21]:
# --- score every canonical audio run with one implementation -------------------
# Reads audio_predictions.jsonl out of each run folder. No model is loaded, so
# pointing this at the frozen Canary folders cannot start Canary.
from privacy_eval import rescore as rescoring
from privacy_eval import stt_comparison as sttcmp

ENGINE_RUNS = {}
for run in AUDIO_RUNS:
    scored = sttcmp.score_run(run["directory"])
    if not scored:
        UNAVAILABLE.append(
            f"{run['run_id']}: no stored transcripts, excluded from the engine "
            "comparison"
        )
        continue
    scored["pipeline_version"] = run["pipeline_version"]
    scored["device_class"] = run["device_class"]
    scored["device"] = device_name(run)
    scored["system"] = system_name(run, short=True)
    scored["architecture"] = run["system"].get("architecture")
    scored["released"] = sttcmp.released_identifier_summary(
        sttcmp.read_predictions(run["directory"])
    )
    scored["memory"] = run["memory"] or {}
    scored["latency"] = run["latency"] or {}
    scored["stored_privacy"] = run["privacy"] or {}
    # One definition of the entity-level outcomes for every arm, recomputed from
    # what each folder saved. `recovery.exact` is the guard: it replays each
    # run's own recorded mask coverage from the reconstructed spans.
    scored["rescored"] = rescoring.rescore_run(run["directory"])
    # Did this run include the spoken-address rewrite in front of the detectors?
    scored["has_spoken_form_stage"] = bool(run["config"].get("spoken_form_normalisation"))
    ENGINE_RUNS[(run["pipeline_version"], run["device_class"])] = scored

# Full runs only. A smoke run is four clips and must not share an axis with 44.
ENGINE_RUNS = {key: item for key, item in ENGINE_RUNS.items() if item["mode"] != "smoke"}

for item in ENGINE_RUNS.values():
    recovery = (item.get("rescored") or {}).get("recovery") or {}
    if item.get("rescored") and not recovery.get("exact"):
        UNAVAILABLE.append(
            f"{item['run_id']}: span reconstruction did not reproduce the run's "
            f"own mask coverage ({recovery.get('mismatched')} of "
            f"{recovery.get('entities_checked')} entities); its rescored "
            "privacy figures are not trustworthy"
        )

# The arms the charts draw, in a fixed order. Whisper is represented by the
# current system; V4 used the same recognizer and appears in the table below as
# provenance rather than as a second Whisper bar.
ARM_SPEC = [
    ("V3", "workstation"),
    ("V3", "pi"),
    (CURRENT_PIPELINE, "workstation"),
    (CURRENT_PIPELINE, "pi"),
    ("V5", "workstation"),
    ("V5", "pi"),
]
ARMS = [
    (f"{SYSTEM_SHORT[version]}\n{DEVICE_LABEL[device]}", ENGINE_RUNS.get((version, device)),
     version, device)
    for version, device in ARM_SPEC
]
PENDING = [name.replace("\n", " ") for name, run, _, _ in ARMS if run is None]
MEASURED_ARMS = [(name, run, version, device)
                 for name, run, version, device in ARMS if run is not None]

print(f"{'arm':36} {'run':62} {'clips':>5} {'rewrite':>8} {'recovery':>9}")
print("-" * 130)
for name, run, version, device in ARMS:
    label = name.replace("\n", " - ")
    if run is None:
        print(f"{label:36} {'NOT MEASURED ON THIS DEVICE':62} {'-':>5} {'-':>8} {'-':>9}")
    else:
        recovery = (run.get("rescored") or {}).get("recovery") or {}
        print(f"{label:36} {run['run_id']:62} {run['clips']:>5} "
              f"{'yes' if run['has_spoken_form_stage'] else 'no':>8} "
              f"{'exact' if recovery.get('exact') else 'CHECK':>9}")
if PENDING:
    print()
    print("Unmeasured arms are drawn as pending. No zero, no estimate, no bar.")
    for label in PENDING:
        UNAVAILABLE.append(f"{label}: never measured on that device, held back from the figures")

# The historical Whisper (V4) folders, listed so the provenance is visible even
# though the charts draw the current system instead.
V4_AUDIO = {key: item for key, item in ENGINE_RUNS.items() if key[0] == "V4"}
if V4_AUDIO:
    print()
    print("Whisper was also measured under the previous system (V4); the charts "
          "below draw the current system instead:")
    for (version, device), item in sorted(V4_AUDIO.items()):
        print(f"  {SYSTEM_SHORT[version]:18} {DEVICE_LABEL[device]:16} {item['run_id']}")

arm                                  run                                                            clips  rewrite  recovery
----------------------------------------------------------------------------------------------------------------------------------
Canary generation - Workstation      20260818T062503Z_windows-amd64_audio_full_workstation             44       no     exact
Canary generation - Raspberry Pi 5   20260818T081310Z_linux-aarch64_audio_full                         44       no     exact
Current system - Workstation         20260823T142306Z_windows-amd64_audio-v6-whisper_full_workstation    44      yes     exact
Current system - Raspberry Pi 5      20260823T204339Z_linux-aarch64_audio-v6-whisper_full              44      yes     exact
Parakeet demo - Workstation          20260823T095553Z_windows-amd64_audio-parakeet_full                44      yes     exact
Parakeet demo - Raspberry Pi 5       NOT MEASURED ON THIS DEVICE                                        -        -   

In [22]:
# --- recognition accuracy, engine against engine -------------------------------
if MEASURED_ARMS:
    figure, (left, right) = plt.subplots(1, 2, figsize=(14.4, 4.8))

    # Left: the two word error rates side by side, so the size of the
    # orthography effect is visible rather than asserted.
    names = [name for name, _, _, _ in MEASURED_ARMS]
    raw = [run["overall"]["mean_wer_raw"] or 0.0 for _, run, _, _ in MEASURED_ARMS]
    normalised = [run["overall"]["aggregate_wer_number_normalised"]
                  for _, run, _, _ in MEASURED_ARMS]
    positions = range(len(names))
    width = 0.38
    left.bar([p - width / 2 for p in positions], raw, width,
             color=GRID, edgecolor=MUTED, label="raw (not engine-comparable)")
    bars = left.bar([p + width / 2 for p in positions], normalised, width,
                    color=[ENGINE_COLOUR[run["engine"]] for _, run, _, _ in MEASURED_ARMS],
                    label="number-normalised (engine-comparable)")
    for bar, value in zip(bars, normalised):
        left.annotate(f"{value:.1%}", (bar.get_x() + bar.get_width() / 2, value),
                      ha="center", va="bottom", xytext=(0, 3),
                      textcoords="offset points", fontsize=9.5, color=INK)
    left.set_xticks(list(positions), names, fontsize=8.5)
    left.set_ylim(0, max(raw + normalised) * 1.55)
    left.set_ylabel("Word error rate (lower is better)")
    left.yaxis.set_major_formatter(PercentFormatter(1.0))
    left.set_title("Word error rate on the same 44 clips", loc="left")
    left.legend(loc="upper left")
    tidy(left)

    # Right: the two things a privacy device cannot get wrong about a number.
    digit_names, digit_values, released_values, colours = [], [], [], []
    for name, run, _, _ in MEASURED_ARMS:
        digits = run["digit_preservation"]
        released = run["released"]
        if digits["accuracy"] is None or released["clean_rate"] is None:
            continue
        digit_names.append(name)
        digit_values.append(digits["accuracy"])
        released_values.append(released["clean_rate"])
        colours.append(ENGINE_COLOUR[run["engine"]])

    if digit_names:
        positions = range(len(digit_names))
        exact_bars = right.bar([p - width / 2 for p in positions], digit_values, width,
                               color=colours, alpha=0.55,
                               label="long numbers transcribed exactly")
        released_bars = right.bar([p + width / 2 for p in positions], released_values, width,
                                  color=colours,
                                  label="no identifier left in the released text")
        # Both series are annotated, because one of them is genuinely zero and a
        # zero-height bar with no label reads as a missing measurement.
        for group, values in ((exact_bars, digit_values), (released_bars, released_values)):
            for bar, value in zip(group, values):
                right.annotate(f"{value:.0%}", (bar.get_x() + bar.get_width() / 2, value),
                               ha="center", va="bottom", xytext=(0, 3),
                               textcoords="offset points", fontsize=9, color=INK)
        right.set_xticks(list(positions), digit_names, fontsize=8.5)
        right.set_ylim(0, 1.34)
        right.yaxis.set_major_formatter(PercentFormatter(1.0))
        right.set_ylabel("Share of the clips carrying a long number")
        right.set_title("What happens to a number read aloud", loc="left")
        right.legend(loc="upper left", fontsize=8.5)
        tidy(right)

    subtitle = "same clips, same privacy pipeline, one scoring implementation"
    if PENDING:
        subtitle += f"\nnot measured: {', '.join(PENDING)}"
    figure.suptitle(f"Recognizer generations: recognition\n{subtitle}",
                    fontsize=12.5, y=1.04)
    figure.tight_layout()
    save(figure, "stt_engine_accuracy",
         "Word error rate and number handling for every measured recognizer on the "
         "committed audio benchmark. The raw word error rate is shown in grey to "
         "expose how much of it is spelling: Canary wrote numbers as words, Whisper "
         "writes numerals, and only the number-normalised bars compare the engines. "
         "The right panel separates transcribing a long number correctly from "
         "keeping it out of the released text - the two are not the same thing, and "
         "the engines differ in opposite directions.",
         runs=[run["run_id"] for _, run, _, _ in MEASURED_ARMS])

    # The table carries every arm, including the historical V4 Whisper folders.
    def engine_row(label, run, version, device):
        return {
            "arm": label.replace("\n", " - "),
            "run_id": run["run_id"],
            "engine": run["engine"],
            "pipeline_version": version,
            "system": SYSTEM_SHORT.get(version, version),
            "is_current_system": version == CURRENT_PIPELINE,
            "drawn_in_figures": (version, device) in
                                {(v, d) for _, _, v, d in MEASURED_ARMS},
            "stt_model": run["stt_model"],
            "device": DEVICE_LABEL[device],
            "clips": run["clips"],
            "aggregate_wer_raw": run["overall"]["aggregate_wer_raw"],
            "mean_wer_raw": run["overall"]["mean_wer_raw"],
            "aggregate_wer_number_normalised":
                run["overall"]["aggregate_wer_number_normalised"],
            "median_wer_number_normalised":
                run["overall"]["median_wer_number_normalised"],
            "identifiers_scored": run["digit_preservation"]["identifiers"],
            "identifiers_transcribed_exactly": run["digit_preservation"]["exact"],
            "clips_with_identifiers": run["released"]["clips_with_identifiers"],
            "clips_released_without_an_identifier": run["released"]["clips_released_clean"],
        }

    engine_rows = [engine_row(name, run, version, device)
                   for name, run, version, device in MEASURED_ARMS]
    engine_rows += [
        engine_row(f"{SYSTEM_SHORT[version]}\n{DEVICE_LABEL[device]}", run, version, device)
        for (version, device), run in sorted(V4_AUDIO.items())
    ]
    save_table(
        engine_rows,
        ["arm", "run_id", "engine", "pipeline_version", "system", "is_current_system",
         "drawn_in_figures", "stt_model", "device", "clips", "aggregate_wer_raw",
         "mean_wer_raw", "aggregate_wer_number_normalised",
         "median_wer_number_normalised", "identifiers_scored",
         "identifiers_transcribed_exactly", "clips_with_identifiers",
         "clips_released_without_an_identifier"],
        "stt_engine_comparison",
        "Every measured recognizer arm, scored from its saved transcripts by one "
        "implementation. Arms that were never measured on a device are absent "
        "rather than zero. `drawn_in_figures` marks the arms the charts show; the "
        "previous system's Whisper rows are provenance for the current ones.",
        runs=[row["run_id"] for row in engine_rows],
    )
else:
    print("no canonical audio runs with stored transcripts")

  figure  research_results/figures/stt_engine_accuracy.png  (+ .pdf)  <- 5 run(s)
  table   research_results/tables/stt_engine_comparison.csv  (7 rows)


In [23]:
# --- cost: is either recognizer fast enough on the device that matters? --------
if MEASURED_ARMS:
    figure, (left, right) = plt.subplots(1, 2, figsize=(14.4, 4.8))
    names = [name for name, _, _, _ in MEASURED_ARMS]
    colours = [ENGINE_COLOUR[run["engine"]] for _, run, _, _ in MEASURED_ARMS]
    positions = list(range(len(names)))

    stt_rtf = [run["stt_real_time_factor"]["mean"] for _, run, _, _ in MEASURED_ARMS]
    pipeline_rtf = [
        (run["pipeline_real_time_factor"] or {}).get("mean") for _, run, _, _ in MEASURED_ARMS
    ]

    width = 0.38
    left.bar([p - width / 2 for p in positions], stt_rtf, width, color=colours,
             label="speech recognition")
    left.bar([p + width / 2 for p in positions],
             [value or 0.0 for value in pipeline_rtf], width, color=colours,
             alpha=0.5, label="whole pipeline")
    for position, value in zip(positions, stt_rtf):
        left.annotate(f"{value:.2f}x", (position - width / 2, value), ha="center",
                      va="bottom", xytext=(0, 3), textcoords="offset points",
                      fontsize=9.5, color=INK)
    # Real time is the line that decides whether a device keeps up with speech.
    left.axhline(1.0, color=INK, linewidth=1.1, linestyle="--")
    # Anchored left, clear of the value labels sitting on top of the bars.
    left.annotate("real time", (-0.45, 1.0), ha="left", va="bottom",
                  xytext=(0, 4), textcoords="offset points", fontsize=9,
                  color=MUTED)
    left.set_xticks(positions, names, fontsize=8.5)
    left.set_ylim(0, max([1.0] + stt_rtf + [v or 0.0 for v in pipeline_rtf]) * 1.3)
    left.set_ylabel("Real-time factor (lower is faster)")
    left.set_title("Processing cost per second of audio", loc="left")
    left.legend(loc="upper left")
    tidy(left)

    p50 = [run["end_to_end_latency_ms"]["median"] / 1000.0 for _, run, _, _ in MEASURED_ARMS]
    peak = [
        (run["memory"].get("process_peak_rss_bytes") or 0) / 1024 ** 2
        for _, run, _, _ in MEASURED_ARMS
    ]
    bars = right.bar(positions, p50, 0.55, color=colours)
    for bar, seconds, rss in zip(bars, p50, peak):
        label = f"{seconds:.1f} s"
        if rss:
            label += f"\n{rss:,.0f} MiB peak"
        right.annotate(label, (bar.get_x() + bar.get_width() / 2, seconds),
                       ha="center", va="bottom", xytext=(0, 3),
                       textcoords="offset points", fontsize=9.5, color=INK)
    right.set_xticks(positions, names, fontsize=8.5)
    right.set_ylim(0, max(p50) * 1.4)
    right.set_ylabel("Median end-to-end latency (s) - lower is better")
    right.set_title("Audio in to post-mask text out", loc="left")
    tidy(right)

    subtitle = "median over the 44 committed clips"
    if PENDING:
        subtitle += f"\nnot measured: {', '.join(PENDING)}"
    figure.suptitle(f"Recognizer generations: cost\n{subtitle}",
                    fontsize=12.5, y=1.04)
    figure.tight_layout()
    save(figure, "stt_engine_cost",
         "Real-time factor and end-to-end latency for every measured recognizer. "
         "The dashed line is real time: a bar below it means the device processes "
         "speech faster than it arrives. Bars are drawn only for arms that have "
         "been measured.",
         runs=[run["run_id"] for _, run, _, _ in MEASURED_ARMS])
else:
    print("no canonical audio runs to compare on cost")

  figure  research_results/figures/stt_engine_cost.png  (+ .pdf)  <- 5 run(s)


In [24]:
# --- end-to-end privacy, per recognizer, on one scoring definition -------------
if MEASURED_ARMS:
    rows = []
    for name, run, version, device in MEASURED_ARMS:
        stored = run["stored_privacy"] or {}
        arms_ = stored.get("control_arms") or {}
        reference = (arms_.get("reference_text") or {}).get("comparable") or {}
        rescored = run["rescored"] or {}
        rows.append({
            "arm": name.replace("\n", " - "),
            "run_id": run["run_id"],
            "engine": run["engine"],
            "pipeline_version": version,
            "system": SYSTEM_SHORT.get(version, version),
            "device": DEVICE_LABEL[device],
            "pipeline_has_spoken_form_stage": run["has_spoken_form_stage"],
            # Alignment-free, and therefore true regardless of any of the above.
            "clips_with_identifiers": run["released"]["clips_with_identifiers"],
            "clips_released_clean": run["released"]["clips_released_clean"],
            "released_clean_rate": run["released"]["clean_rate"],
            "identifiers_rebuilt_by_the_renderer": len(
                (stored.get("postprocessing_amplification") or {}).get("cases", [])
            ),
            # Reference-text control: same text into the same masker on both
            # sides, so this one was always comparable and is the check that
            # nothing downstream changed between generations.
            "control_reference_entity_recall": reference.get("entity_recall"),
            "control_reference_character_recall": reference.get("character_recall"),
            # Recomputed for every arm by privacy_eval.rescore, one definition.
            "expected_entities": rescored.get("expected_entities"),
            "entities_protected": rescored.get("entities_protected"),
            "entity_protection_rate": rescored.get("entity_protection_rate"),
            "entities_fully_masked_as_spoken": rescored.get("entities_fully_masked"),
            "entities_preserved_by_stt": rescored.get("entities_preserved_by_stt"),
            "stt_preservation_rate": rescored.get("stt_preservation_rate"),
            "character_recall_after_stt": rescored.get("character_recall_after_stt"),
            "character_leakage_after_stt": rescored.get("character_leakage_after_stt"),
            "fully_sanitized_clips": rescored.get("fully_sanitized_clips"),
            "clips_scored": rescored.get("clips_scored"),
            "hard_negative_false_positive_clips":
                rescored.get("hard_negative_false_positive_clips"),
            "hard_negative_clips": rescored.get("hard_negative_clips"),
            "privacy_model_failures": (
                (rescored.get("outcomes") or {}).get("leaked_privacy_model", 0)
                + (rescored.get("outcomes") or {}).get("partially_masked", 0)
            ),
            "stt_caused_leaks": (rescored.get("outcomes") or {}).get("stt_altered_leaked", 0),
        })

    save_table(
        rows,
        ["arm", "run_id", "engine", "pipeline_version", "system", "device",
         "pipeline_has_spoken_form_stage", "clips_with_identifiers",
         "clips_released_clean", "released_clean_rate",
         "identifiers_rebuilt_by_the_renderer", "control_reference_entity_recall",
         "control_reference_character_recall", "expected_entities",
         "entities_protected", "entity_protection_rate",
         "entities_fully_masked_as_spoken", "entities_preserved_by_stt",
         "stt_preservation_rate", "character_recall_after_stt",
         "character_leakage_after_stt", "fully_sanitized_clips", "clips_scored",
         "hard_negative_false_positive_clips", "hard_negative_clips",
         "privacy_model_failures", "stt_caused_leaks"],
        "stt_engine_privacy",
        "End-to-end privacy per recognizer arm. The entity- and character-level "
        "columns are recomputed for every arm by privacy_eval.rescore on one "
        "definition - spoken numbers folded on both sides, located regions trimmed "
        "to their alphanumeric core - so they are comparable across engines; the "
        "released-identifier columns need no alignment at all.",
        runs=[row["run_id"] for row in rows],
    )

    # The comparable control: identical text into the identical masker. If these
    # differ between generations, something downstream changed and the whole
    # comparison is void.
    controls = {
        row["arm"]: row["control_reference_entity_recall"]
        for row in rows if row["control_reference_entity_recall"] is not None
    }
    if len(set(round(value, 6) for value in controls.values())) == 1:
        print(f"control arm identical across every measured run: "
              f"reference-text entity recall {next(iter(controls.values())):.4f}")
        print("-> the masking path is the same code in every generation, as intended")
    elif controls:
        print("control arm differs across runs. Expected where one arm ran with the")
        print("spoken-address rewrite and another did not; any other difference means")
        print("something downstream changed and the comparison is not safe:")
        for arm_name, value in controls.items():
            has_stage = next(row["pipeline_has_spoken_form_stage"]
                             for row in rows if row["arm"] == arm_name)
            print(f"  {arm_name:34} {value:.4f}  "
                  f"(spoken-address rewrite: {'yes' if has_stage else 'no'})")

    figure, axes = plt.subplots(1, 2, figsize=(14.6, 4.8))
    names = [name for name, _, _, _ in MEASURED_ARMS]
    colours = [ENGINE_COLOUR[row["engine"]] for row in rows]

    left = axes[0]
    values = [row["released_clean_rate"] or 0.0 for row in rows]
    bars = left.bar(range(len(names)), values, 0.55, color=colours)
    for bar, row in zip(bars, rows):
        left.annotate(
            f"{row['clips_released_clean']}/{row['clips_with_identifiers']} clips",
            (bar.get_x() + bar.get_width() / 2, bar.get_height()), ha="center",
            va="bottom", xytext=(0, 3), textcoords="offset points",
            fontsize=9.5, color=INK)
    left.set_xticks(range(len(names)), names, fontsize=8.5)
    left.set_ylim(0, 1.18)
    left.yaxis.set_major_formatter(PercentFormatter(1.0))
    left.set_ylabel("Clips released with no legible identifier")
    left.set_title("No long number reached the screen (higher is better)")
    tidy(left)

    right = axes[1]
    values = [row["entity_protection_rate"] or 0.0 for row in rows]
    bars = right.bar(range(len(names)), values, 0.55, color=colours)
    for bar, row in zip(bars, rows):
        right.annotate(
            f"{row['entities_protected']}/{row['expected_entities']}",
            (bar.get_x() + bar.get_width() / 2, bar.get_height()), ha="center",
            va="bottom", xytext=(0, 3), textcoords="offset points",
            fontsize=9.5, color=INK)
    right.set_xticks(range(len(names)), names, fontsize=8.5)
    right.set_ylim(0, 1.18)
    right.yaxis.set_major_formatter(PercentFormatter(1.0))
    right.set_ylabel("Gold entities that reached no released text")
    right.set_title("Entities protected end to end (higher is better)")
    tidy(right)

    subtitle = ("both panels after masking and after the spoken-digit renderer; "
                "right panel rescored for every arm on one definition")
    if PENDING:
        subtitle += f"\nnot measured: {', '.join(PENDING)}"
    figure.suptitle(f"End-to-end privacy, recognizer against recognizer\n{subtitle}",
                    fontsize=12.5, y=1.05)
    figure.tight_layout()
    save(figure, "stt_engine_released_identifiers",
         "End-to-end privacy per recognizer. Left: the measure that survives a "
         "change of recognizer without any alignment - after masking and after the "
         "display renderer, is an identifier-scale number still legible in what the "
         "device released? Right: the share of annotated entities that reached no "
         "released text, recomputed for every arm by privacy_eval.rescore so that "
         "one scoring definition is applied throughout.",
         runs=[row["run_id"] for row in rows])
else:
    print("no canonical audio runs to compare on privacy")

  table   research_results/tables/stt_engine_privacy.csv  (5 rows)
control arm identical across every measured run: reference-text entity recall 0.7333
-> the masking path is the same code in every generation, as intended


  figure  research_results/figures/stt_engine_released_identifiers.png  (+ .pdf)  <- 5 run(s)


In [25]:
# --- what each arm is, and what is still missing -------------------------------
print("Recognizer comparison status")
print("=" * 78)
for name, run, version, device in ARMS:
    label = name.replace("\n", " - ")
    if run is None:
        print(f"  {label:36} NOT MEASURED ON THIS DEVICE")
        continue
    if run["engine"] != "whisper":
        # The rewrite is a Whisper-era addition. Saying a Canary run "predates"
        # it would read as a gap in that run rather than as V3 being V3.
        stage = "V3 privacy path (the spoken-address rewrite is later)"
    elif run["has_spoken_form_stage"]:
        stage = "with the spoken-address rewrite"
    else:
        stage = "before the spoken-address rewrite"
    print(f"  {label:36} {run['run_id']}")
    print(f"  {'':36} {stage}")

if PENDING:
    print()
    print("Arms never measured on that device:")
    for label in PENDING:
        print(f"  {label}")
    print()
    print("Nothing above substitutes a zero or an estimate for them: the bar is")
    print("absent and the arm is listed as not measured.")
else:
    print()
    print("Every arm has been measured on hardware.")

CANARY_PI = ENGINE_RUNS.get(("V3", "pi"))
CURRENT_PI_ARM = ENGINE_RUNS.get((CURRENT_PIPELINE, "pi"))
if CANARY_PI and CURRENT_PI_ARM:
    canary_pi = CANARY_PI["stt_real_time_factor"]["mean"]
    current_pi = CURRENT_PI_ARM["stt_real_time_factor"]["mean"]
    print()
    print("The question the Raspberry Pi runs settled:")
    print(f"  Canary needed {canary_pi:.2f}x real time on the Raspberry Pi 5, so it")
    print("  could not keep up with speech there. The current system needs "
          f"{current_pi:.2f}x on")
    print("  the same device, which is the condition for live use. That is the")
    print("  deployment result; it is not a claim about transcription or about")
    print("  privacy, and it is a mean - the per-clip spread is in section 12.")

Recognizer comparison status
  Canary generation - Workstation      20260818T062503Z_windows-amd64_audio_full_workstation
                                       V3 privacy path (the spoken-address rewrite is later)
  Canary generation - Raspberry Pi 5   20260818T081310Z_linux-aarch64_audio_full
                                       V3 privacy path (the spoken-address rewrite is later)
  Current system - Workstation         20260823T142306Z_windows-amd64_audio-v6-whisper_full_workstation
                                       with the spoken-address rewrite
  Current system - Raspberry Pi 5      20260823T204339Z_linux-aarch64_audio-v6-whisper_full
                                       with the spoken-address rewrite
  Parakeet demo - Workstation          20260823T095553Z_windows-amd64_audio-parakeet_full
                                       V3 privacy path (the spoken-address rewrite is later)
  Parakeet demo - Raspberry Pi 5       NOT MEASURED ON THIS DEVICE

Arms never measured on

## 12. The current system: workstation against Raspberry Pi 5

The headline device comparison, and the reason the repository can say anything
at all about running OnKith on a Raspberry Pi. Both sides are direct
measurements of `FullPipeline_V6` - nothing here is inferred from an earlier
pipeline.

Two kinds of number are kept apart on purpose:

* **Cost** - real-time factor, latency, memory, throughput. These move with the
  device, and the Pi is expected to be slower. The question is whether it is
  still fast enough.
* **Quality** - word error rate, privacy recall, leakage, typed F1. These
  should *not* move with the device. Where they do, the cause is INT8 kernel
  arithmetic differing between x86-64 and aarch64, and the size of the
  difference is the interesting number.

### Reading the real-time factor

`RTF < 1` means the device finishes processing a clip in less time than the
clip lasts, which is the condition for live use. The mean is reported with the
P90 beside it, because a mean below 1 and a P90 above 1 is not the same claim
as "always faster than real time".

The per-clip scatter in section 10 says where that P90 comes from, and it is
the opposite of the intuitive answer: on the Pi the clips that exceed real time
are the **shortest** ones, not the longest. Whisper decodes a fixed 30-second
window whatever the clip length, so the cost per clip has a floor; divided by
1.8 seconds of audio that floor is an RTF above 1, and divided by 15 seconds it
is not. The cell below counts it exactly.

In [26]:
# --- cost: what the Raspberry Pi pays -----------------------------------------
device_rows = []
if CURRENT_AUDIO_WS and CURRENT_AUDIO_PI:
    pair = [CURRENT_AUDIO_WS, CURRENT_AUDIO_PI]
    names = [device_name(run) for run in pair]
    colours = [DEVICE_COLOUR[run["device_class"]] for run in pair]
    positions = list(range(len(names)))
    width = 0.36

    def rtf_of(run, which, statistic="mean_ms"):
        block = ((run["latency"] or {}).get("real_time_factor") or {}).get(which) or {}
        return block.get(statistic)

    def stage_of(run, statistic):
        return (((run["latency"] or {}).get("stages") or {}).get("end_to_end") or {}).get(statistic)

    figure, axes = plt.subplots(1, 3, figsize=(15.0, 4.6))

    # --- panel 1: real-time factor --------------------------------------------
    # Colour here separates the two measures, not the two devices: the x axis
    # already carries the device, and colouring by device would leave the legend
    # describing only the first pair of bars.
    left = axes[0]
    stt_mean = [rtf_of(run, "stt") for run in pair]
    pipe_mean = [rtf_of(run, "pipeline") for run in pair]
    stt_p90 = [rtf_of(run, "stt", "p90_ms") for run in pair]
    pipe_p90 = [rtf_of(run, "pipeline", "p90_ms") for run in pair]
    left.bar([p - width / 2 for p in positions], stt_mean, width,
             color=SERIES["primary"], label="Speech recognition")
    left.bar([p + width / 2 for p in positions], pipe_mean, width,
             color=SERIES["primary"], alpha=0.45, label="Whole local pipeline")
    for offset, means, p90s in ((-width / 2, stt_mean, stt_p90),
                                (width / 2, pipe_mean, pipe_p90)):
        for position, mean, p90 in zip(positions, means, p90s):
            left.annotate(f"{mean:.2f}x", (position + offset, mean), ha="center",
                          va="top", xytext=(0, -5), textcoords="offset points",
                          fontsize=9.5, color="white")
            if p90 is None:
                continue
            left.plot([position + offset - width / 2.4, position + offset + width / 2.4],
                      [p90, p90], color=INK, linewidth=1.8, solid_capstyle="butt")
            left.annotate(f"P90 {p90:.2f}x", (position + offset, p90), ha="center",
                          va="bottom", xytext=(0, 4), textcoords="offset points",
                          fontsize=8.5, color=INK)
    left.axhline(1.0, color=SERIES["secondary"], linewidth=1.3, linestyle="--")
    # Axes-fraction x: a data-coordinate anchor left of the first bar falls
    # outside the auto x limits and the label is silently clipped.
    left.annotate("RTF = 1 (real time)", (0.02, 1.0),
                  xycoords=("axes fraction", "data"), ha="left", va="bottom",
                  xytext=(0, 4), textcoords="offset points", fontsize=9,
                  color=SERIES["secondary"])
    left.set_xticks(positions, names)
    left.set_ylim(0, max([1.0] + [v for v in stt_p90 + pipe_p90 if v] + stt_mean) * 1.45)
    left.set_ylabel("Real-time factor (lower is faster)")
    left.set_title("Cost per second of audio" + chr(10) + "bar = mean, black rule = P90",
                   loc="left", fontsize=11)
    left.legend(loc="upper center", fontsize=8.5, ncol=2)
    tidy(left)

    # --- panel 2: end-to-end latency ------------------------------------------
    middle = axes[1]
    metrics = [("mean_ms", "Mean"), ("p50_ms", "P50"), ("p95_ms", "P95")]
    group = 0.26
    for index, run in enumerate(pair):
        values = [(stage_of(run, key) or 0) / 1000.0 for key, _ in metrics]
        offset = (index - 0.5) * group
        bars = middle.bar([p + offset for p in range(len(metrics))], values, group,
                          color=DEVICE_COLOUR[run["device_class"]], label=device_name(run))
        for bar, value in zip(bars, values):
            middle.annotate(f"{value:.2f}", (bar.get_x() + bar.get_width() / 2, value),
                            ha="center", va="bottom", xytext=(0, 3),
                            textcoords="offset points", fontsize=9, color=INK)
    middle.set_xticks(range(len(metrics)), [label for _, label in metrics])
    middle.set_ylim(0, max((stage_of(run, key) or 0) / 1000.0
                           for run in pair for key, _ in metrics) * 1.3)
    middle.set_ylabel("Seconds per clip (lower is better)")
    middle.set_title("Audio in to post-mask text out\n44 committed clips", loc="left",
                     fontsize=11)
    middle.legend(loc="upper left", fontsize=9)
    tidy(middle)

    # --- panel 3: peak memory --------------------------------------------------
    right = axes[2]
    peak = [((run["memory"] or {}).get("process_peak_rss_bytes") or 0) / 1024 ** 2
            for run in pair]
    bars = right.bar(positions, peak, 0.5, color=colours)
    for bar, run, value in zip(bars, pair, peak):
        total = (run["system"].get("total_ram_bytes") or 0) / 1024 ** 3
        right.annotate(f"{value:,.0f} MiB" + (f"\nof {total:.1f} GiB" if total else ""),
                       (bar.get_x() + bar.get_width() / 2, value), ha="center",
                       va="bottom", xytext=(0, 3), textcoords="offset points",
                       fontsize=9.5, color=INK)
    right.axhline(4096, color=MUTED, linestyle="--", linewidth=1.2)
    right.annotate("4 GB Raspberry Pi budget", (0.02, 4096),
                   xycoords=("axes fraction", "data"), xytext=(0, 4),
                   textcoords="offset points", fontsize=9, color=MUTED)
    right.set_xticks(positions, names)
    right.set_ylim(0, 4096 * 1.2)
    right.set_ylabel("Peak resident set size (MiB) - lower is better")
    right.set_title("Memory the whole stack needs\naudio benchmark, all models loaded",
                    loc="left", fontsize=11)
    tidy(right)

    figure.suptitle(
        f"{SYSTEM_LABEL[CURRENT_PIPELINE]}: cost on the workstation and on a Raspberry Pi 5\n"
        f"same 44 clips, same INT8 artifact, same recognizer",
        fontsize=12.5, y=1.05)
    figure.tight_layout()
    save(figure, "current_system_device_cost",
         "Real-time factor, end-to-end latency and peak memory for the current "
         "system on the workstation and on a Raspberry Pi 5. RTF below the dashed "
         "line means the device processes speech faster than it arrives; the black "
         "rule on each bar is the P90. On the Raspberry Pi the P90 sits above the "
         "line, and the clips responsible are the shortest ones - Whisper decodes a "
         "fixed window whatever the clip length, so the per-clip cost has a floor.",
         runs=[run["run_id"] for run in pair])

    for run in pair:
        device_rows.append({
            "run_id": run["run_id"], "benchmark": "audio_pipeline",
            "device": device_name(run),
            "board": run["system"].get("board_model") or "not reported",
            "os": run["system"].get("platform"),
            "architecture": run["system"].get("architecture"),
            "logical_cpus": run["system"].get("logical_cpus"),
            "system_ram_gib": (run["system"].get("total_ram_bytes") or 0) / 1024 ** 3,
            "python": run["system"].get("python"),
            "onnxruntime": run["system"].get("onnxruntime"),
            "mean_stt_rtf": rtf_of(run, "stt"),
            "p90_stt_rtf": rtf_of(run, "stt", "p90_ms"),
            "mean_pipeline_rtf": rtf_of(run, "pipeline"),
            "p90_pipeline_rtf": rtf_of(run, "pipeline", "p90_ms"),
            "mean_end_to_end_ms": stage_of(run, "mean_ms"),
            "p50_end_to_end_ms": stage_of(run, "p50_ms"),
            "p95_end_to_end_ms": stage_of(run, "p95_ms"),
            "mean_stt_ms": (((run["latency"] or {}).get("stages") or {}).get("stt") or {}).get("mean_ms"),
            "mean_privacy_ms": (((run["latency"] or {}).get("stages") or {}).get("privacy") or {}).get("mean_ms"),
            "peak_rss_mib": ((run["memory"] or {}).get("process_peak_rss_bytes") or 0) / 1024 ** 2,
        })

    print(f"{'metric':30} {'Workstation':>14} {'Raspberry Pi 5':>16} {'Pi / workstation':>18}")
    print("-" * 82)
    for label, key in (("mean STT RTF", "mean_stt_rtf"),
                       ("P90 STT RTF", "p90_stt_rtf"),
                       ("mean full-pipeline RTF", "mean_pipeline_rtf"),
                       ("P90 full-pipeline RTF", "p90_pipeline_rtf"),
                       ("P50 end to end (ms)", "p50_end_to_end_ms"),
                       ("P95 end to end (ms)", "p95_end_to_end_ms"),
                       ("peak RSS (MiB)", "peak_rss_mib")):
        ws, pi = device_rows[0][key], device_rows[1][key]
        ratio = f"{pi / ws:.2f}x" if ws else "-"
        print(f"{label:30} {ws:>14,.3f} {pi:>16,.3f} {ratio:>18}")

    # Where the P90 above 1 comes from. Counted per clip rather than described.
    print()
    print("Clips processed slower than real time (whole local pipeline):")
    for run in pair:
        rtfs = [(clip["audio_duration_s"], clip["pipeline_real_time_factor"])
                for clip in run["clips"]
                if clip["pipeline_real_time_factor"] is not None
                and clip["audio_duration_s"] is not None]
        over = [item for item in rtfs if item[1] > 1.0]
        long_enough = [item for item in rtfs if item[0] >= 3.0]
        summary = f"  {device_name(run):16} {len(over)} of {len(rtfs)} clips"
        if over:
            summary += (f", all of them shorter than "
                        f"{max(item[0] for item in over):.1f} s")
        if long_enough:
            summary += (f"; every clip of 3 s or more stayed under real time "
                        f"(worst {max(item[1] for item in long_enough):.2f}x)")
        print(summary)
else:
    print("the device cost comparison needs the canonical current-system audio run "
          "on both devices")

  figure  research_results/figures/current_system_device_cost.png  (+ .pdf)  <- 2 run(s)
metric                            Workstation   Raspberry Pi 5   Pi / workstation
----------------------------------------------------------------------------------
mean STT RTF                            0.371            0.617              1.67x
P90 STT RTF                             0.702            1.090              1.55x
mean full-pipeline RTF                  0.394            0.634              1.61x
P90 full-pipeline RTF                   0.729            1.109              1.52x
P50 end to end (ms)                 1,537.993        2,615.189              1.70x
P95 end to end (ms)                 2,741.504        5,387.533              1.97x
peak RSS (MiB)                      1,271.578        1,571.156              1.24x

Clips processed slower than real time (whole local pipeline):
  Workstation      0 of 44 clips; every clip of 3 s or more stayed under real time (worst 0.44x)
  Raspberry 

In [27]:
# --- quality: what must not move with the device ------------------------------
quality_rows = []
if CURRENT_AUDIO_WS and CURRENT_AUDIO_PI and CURRENT_TEXT_WS and CURRENT_TEXT_PI:
    audio_pair = [CURRENT_AUDIO_WS, CURRENT_AUDIO_PI]
    text_pair = [CURRENT_TEXT_WS, CURRENT_TEXT_PI]
    names = [device_name(run) for run in audio_pair]
    group = 0.34

    figure, axes = plt.subplots(1, 3, figsize=(15.0, 4.6))

    def grouped(axis, runs, series, *, ylim_pad=1.32, percent=False):
        """One group per metric, one bar per device. Bars start at zero.

        ``percent`` keeps the bar labels in the same units as the axis: a
        percent axis with decimal labels reads as two different numbers.
        """

        labels = [label for label, _ in series]
        highest = 0.0
        for index, run in enumerate(runs):
            values = [getter(run) or 0.0 for _, getter in series]
            highest = max(highest, max(values))
            offset = (index - 0.5) * group
            bars = axis.bar([p + offset for p in range(len(labels))], values, group,
                            color=DEVICE_COLOUR[run["device_class"]], label=device_name(run))
            for bar, value in zip(bars, values):
                label = f"{value:.2%}" if percent else f"{value:.4f}"
                axis.annotate(label,
                              (bar.get_x() + bar.get_width() / 2, value), ha="center",
                              va="bottom", xytext=(0, 3), textcoords="offset points",
                              fontsize=8.5, color=INK)
        axis.set_xticks(range(len(labels)), labels, fontsize=9)
        axis.set_ylim(0, highest * ylim_pad)
        tidy(axis)

    # --- panel 1: recognition, lower is better ---------------------------------
    grouped(axes[0], audio_pair, [
        ("Aggregate WER", lambda r: ((r["stt"] or {}).get("overall") or {}).get("aggregate_wer")),
        ("WER,\nnumber-normalised",
         lambda r: ((r["stt"] or {}).get("number_normalised") or {}).get("aggregate_wer")),
    ], percent=True)
    axes[0].yaxis.set_major_formatter(PercentFormatter(1.0))
    axes[0].set_ylabel("Word error rate (lower is better)")
    axes[0].set_title("Speech recognition\n44 committed clips", loc="left", fontsize=11)
    axes[0].legend(loc="upper left", fontsize=9)

    # --- panel 2: privacy after speech recognition -----------------------------
    grouped(axes[1], audio_pair, [
        ("Private character\nrecall\n(higher is better)",
         lambda r: ((r["privacy"] or {}).get("end_to_end") or {}).get("character_recall_after_stt")),
        ("Character leakage\n(lower is better)",
         lambda r: ((r["privacy"] or {}).get("end_to_end") or {}).get("character_leakage_after_stt")),
    ], ylim_pad=1.22, percent=True)
    axes[1].yaxis.set_major_formatter(PercentFormatter(1.0))
    axes[1].set_ylabel("Share of private characters")
    axes[1].set_title("Privacy after speech recognition\n32 privacy-annotated clips",
                      loc="left", fontsize=11)

    # --- panel 3: the text benchmark, higher is better -------------------------
    grouped(axes[2], text_pair, [
        ("Typed entity F1",
         lambda r: ((r["evaluation"] or {}).get("production_path") or {}).get("typed_entity", {}).get("f1")),
        ("Private character\nrecall",
         lambda r: ((r["evaluation"] or {}).get("production_path") or {}).get("private_character_recall")),
    ], ylim_pad=1.18)
    axes[2].set_ylim(0, 1.18)
    axes[2].set_ylabel("Score (higher is better)")
    axes[2].set_title("Privacy model on text\n2,000 committed rows", loc="left", fontsize=11)
    # This panel draws a different pair of runs from the two beside it, so it
    # carries its own legend rather than borrowing the one on the left.
    axes[2].legend(loc="lower right", fontsize=9)

    figure.suptitle(
        f"{SYSTEM_LABEL[CURRENT_PIPELINE]}: quality on the workstation and on a Raspberry Pi 5\n"
        "these are the numbers that should not move with the device",
        fontsize=12.5, y=1.06)
    figure.tight_layout()
    save(figure, "current_system_device_quality",
         "Recognition accuracy, end-to-end privacy after speech recognition, and "
         "privacy-model quality on text, for the current system on both devices. "
         "Every pair is measured on the same committed fixtures through the same "
         "INT8 artifact; the residual differences are INT8 kernel arithmetic on "
         "aarch64 against x86-64.",
         runs=[run["run_id"] for run in audio_pair + text_pair])

    def delta_row(metric, benchmark, ws_run, pi_run, getter, better):
        ws, pi = getter(ws_run), getter(pi_run)
        return {
            "metric": metric, "benchmark": benchmark, "better": better,
            "workstation": ws, "raspberry_pi_5": pi,
            "difference": None if ws is None or pi is None else pi - ws,
            "workstation_run": ws_run["run_id"], "raspberry_pi_5_run": pi_run["run_id"],
        }

    quality_rows = [
        delta_row("Aggregate WER", "audio_pipeline", *audio_pair,
                  lambda r: ((r["stt"] or {}).get("overall") or {}).get("aggregate_wer"), "lower"),
        delta_row("Aggregate WER, number-normalised", "audio_pipeline", *audio_pair,
                  lambda r: ((r["stt"] or {}).get("number_normalised") or {}).get("aggregate_wer"), "lower"),
        delta_row("Private character recall after STT", "audio_pipeline", *audio_pair,
                  lambda r: ((r["privacy"] or {}).get("end_to_end") or {}).get("character_recall_after_stt"), "higher"),
        delta_row("Private character leakage after STT", "audio_pipeline", *audio_pair,
                  lambda r: ((r["privacy"] or {}).get("end_to_end") or {}).get("character_leakage_after_stt"), "lower"),
        delta_row("Fully sanitized clip rate", "audio_pipeline", *audio_pair,
                  lambda r: ((r["privacy"] or {}).get("end_to_end") or {}).get("fully_sanitized_clip_rate"), "higher"),
        delta_row("Typed entity F1", "cpu_text", *text_pair,
                  lambda r: ((r["evaluation"] or {}).get("production_path") or {}).get("typed_entity", {}).get("f1"), "higher"),
        delta_row("Private character recall", "cpu_text", *text_pair,
                  lambda r: ((r["evaluation"] or {}).get("production_path") or {}).get("private_character_recall"), "higher"),
        delta_row("Private character leakage", "cpu_text", *text_pair,
                  lambda r: ((r["evaluation"] or {}).get("production_path") or {}).get("private_character_leakage"), "lower"),
        delta_row("Non-private overmasking", "cpu_text", *text_pair,
                  lambda r: ((r["evaluation"] or {}).get("production_path") or {}).get("non_private_overmasking_rate"), "lower"),
        delta_row("ood_dev typed F1 (model-only cross-check)", "cpu_text", *text_pair,
                  lambda r: ((r["evaluation"] or {}).get("ood_dev") or {}).get("typed_entity_f1"), "higher"),
    ]

    print(f"{'metric':44} {'better':>7} {'Workstation':>12} {'Pi 5':>12} {'difference':>12}")
    print("-" * 92)
    for row in quality_rows:
        if row["workstation"] is None or row["raspberry_pi_5"] is None:
            continue
        print(f"{row['metric']:44} {row['better']:>7} {row['workstation']:>12.4f} "
              f"{row['raspberry_pi_5']:>12.4f} {row['difference']:>+12.4f}")
else:
    print("the device quality comparison needs the canonical current-system audio "
          "and text runs on both devices")

save_table(
    device_rows,
    ["run_id", "benchmark", "device", "board", "os", "architecture", "logical_cpus",
     "system_ram_gib", "python", "onnxruntime", "mean_stt_rtf", "p90_stt_rtf",
     "mean_pipeline_rtf", "p90_pipeline_rtf", "mean_end_to_end_ms",
     "p50_end_to_end_ms", "p95_end_to_end_ms", "mean_stt_ms", "mean_privacy_ms",
     "peak_rss_mib"],
    "current_system_device_cost",
    "Cost of the current system on each device, with the environment each run "
    "recorded for itself.",
    runs=[row["run_id"] for row in device_rows],
)
save_table(
    quality_rows,
    ["metric", "benchmark", "better", "workstation", "raspberry_pi_5", "difference",
     "workstation_run", "raspberry_pi_5_run"],
    "current_system_device_quality",
    "Every quality metric of the current system measured on both devices, with the "
    "difference and the run id behind each side.",
    runs=[row[key] for row in quality_rows
          for key in ("workstation_run", "raspberry_pi_5_run")],
)

  figure  research_results/figures/current_system_device_quality.png  (+ .pdf)  <- 4 run(s)
metric                                        better  Workstation         Pi 5   difference
--------------------------------------------------------------------------------------------
Aggregate WER                                  lower       0.1795       0.1841      +0.0046
Aggregate WER, number-normalised               lower       0.0854       0.0903      +0.0050
Private character recall after STT            higher       0.8314       0.8300      -0.0013
Private character leakage after STT            lower       0.1686       0.1700      +0.0013
Fully sanitized clip rate                     higher       0.6400       0.6400      +0.0000
Typed entity F1                               higher       0.9452       0.9448      -0.0004
Private character recall                      higher       0.9950       0.9949      -0.0001
Private character leakage                      lower       0.0050       0.0051 

In [28]:
# --- did adding the assistant change anything local? ---------------------------
# V6 is V4 plus the assistant lane. If the local recognition and masking core is
# really unchanged, then on the same device, the same fixtures and the same INT8
# artifact, the quality numbers must be identical - not close, identical. Cost
# may differ: it is a different process with different imports, measured on a
# different day.
inheritance_rows = []
PREVIOUS_PIPELINE = "V4"


def compare_generation(kind, getters, loader, table):
    for device in ("workstation", "pi"):
        old, new = table.get((PREVIOUS_PIPELINE, device)), table.get((CURRENT_PIPELINE, device))
        if not old or not new:
            continue
        for metric, getter, moves_with_device in getters:
            before, after = getter(old), getter(new)
            if before is None or after is None:
                continue
            inheritance_rows.append({
                "benchmark": kind,
                "device": DEVICE_LABEL[device],
                "metric": metric,
                "expected_to_move": moves_with_device,
                "previous_system": before,
                "current_system": after,
                "difference": after - before,
                "identical": abs(after - before) < 1e-12,
                "previous_run": old["run_id"],
                "current_run": new["run_id"],
            })


compare_generation(
    "cpu_text",
    [("Typed entity F1",
      lambda r: ((r["evaluation"] or {}).get("production_path") or {}).get("typed_entity", {}).get("f1"), False),
     ("Private character recall",
      lambda r: ((r["evaluation"] or {}).get("production_path") or {}).get("private_character_recall"), False),
     ("Private character leakage",
      lambda r: ((r["evaluation"] or {}).get("production_path") or {}).get("private_character_leakage"), False),
     ("Mean masking latency (ms)",
      lambda r: ((r["latency"] or {}).get("end_to_end_masking_call") or {}).get("mean_ms"), True)],
    None, TEXT_BY_KEY,
)
compare_generation(
    "audio_pipeline",
    [("Aggregate WER",
      lambda r: ((r["stt"] or {}).get("overall") or {}).get("aggregate_wer"), False),
     ("Aggregate WER, number-normalised",
      lambda r: ((r["stt"] or {}).get("number_normalised") or {}).get("aggregate_wer"), False),
     ("Private character recall after STT",
      lambda r: ((r["privacy"] or {}).get("end_to_end") or {}).get("character_recall_after_stt"), False),
     ("Fully sanitized clip rate",
      lambda r: ((r["privacy"] or {}).get("end_to_end") or {}).get("fully_sanitized_clip_rate"), False),
     ("Mean full-pipeline RTF",
      lambda r: (((r["latency"] or {}).get("real_time_factor") or {}).get("pipeline") or {}).get("mean_ms"), True)],
    None, AUDIO_BY_KEY,
)

if inheritance_rows:
    quality = [row for row in inheritance_rows if not row["expected_to_move"]]
    drift = [row for row in quality if not row["identical"]]
    print(f"{SYSTEM_SHORT[PREVIOUS_PIPELINE]} against {SYSTEM_SHORT[CURRENT_PIPELINE]}, "
          "same device, same fixtures")
    print("=" * 100)
    print(f"{'benchmark':16} {'device':16} {'metric':38} {'moves?':>7} {'difference':>12}")
    print("-" * 100)
    for row in inheritance_rows:
        print(f"{row['benchmark']:16} {row['device']:16} {row['metric']:38} "
              f"{'cost' if row['expected_to_move'] else 'quality':>7} "
              f"{row['difference']:>+12.6f}")
    print()
    if drift:
        print(f"WARNING: {len(drift)} quality metric(s) moved between generations:")
        for row in drift:
            print(f"  {row['device']} / {row['metric']}: {row['difference']:+.6f}")
    else:
        print(f"Every quality metric is identical across {len(quality)} comparisons: "
              "adding the assistant lane changed nothing in the local recognition "
              "and masking core. The cost rows differ, which is expected - a "
              "different process on a different day.")

save_table(
    inheritance_rows,
    ["benchmark", "device", "metric", "expected_to_move", "previous_system",
     "current_system", "difference", "identical", "previous_run", "current_run"],
    "current_system_inherits_previous_core",
    "The previous system against the current one on the same device and the same "
    "fixtures. Quality metrics are expected to be identical because the local core "
    "is unchanged; cost metrics are expected to differ.",
    runs=sorted({row[key] for row in inheritance_rows
                 for key in ("previous_run", "current_run")}),
)

Previous system against Current system, same device, same fixtures
benchmark        device           metric                                  moves?   difference
----------------------------------------------------------------------------------------------------
cpu_text         Workstation      Typed entity F1                        quality    +0.000000
cpu_text         Workstation      Private character recall               quality    +0.000000
cpu_text         Workstation      Private character leakage              quality    +0.000000
cpu_text         Workstation      Mean masking latency (ms)                 cost    +0.189220
cpu_text         Raspberry Pi 5   Typed entity F1                        quality    +0.000000
cpu_text         Raspberry Pi 5   Private character recall               quality    +0.000000
cpu_text         Raspberry Pi 5   Private character leakage              quality    +0.000000
cpu_text         Raspberry Pi 5   Mean masking latency (ms)                 cost

## 13. Assistant integration and the privacy boundary

What `Full_Pipeline_V6_EndToEnd_Benchmark.py` measured: six committed clips
driven through the whole current system - audio, Whisper, DeBERTa masking, a
conversation-scoped placeholder map, the outbound request, the reply, local
rehydration and local speech.

The privacy claim here is narrow and checkable. Every number in the boundary
panel is read off **the bytes the LLM client handed to `urllib`**, not off the
arguments a function was called with, and every payload is written to the run
folder verbatim so the claim can be verified rather than believed.

Three counts that are often confused, kept apart:

| Count | What it means | Where it can be fixed |
|---|---|---|
| Values the masker replaced that reappeared outbound | a boundary failure | must be zero |
| Gold entities the detector never found, that then went outbound | a recall miss | the privacy model |
| Raw transcripts in the payload | the masked text was bypassed | must be zero |

### What each device actually measured

The workstation run had a `GROQ_API_KEY` and made six live requests. The
Raspberry Pi run did not, so it is an **outbound-payload audit**: the real
client built each request and the bytes were audited, but nothing was sent.
Everything local - recognition, masking, the placeholder map, rehydration
probes, the local-command route - was measured on both. The live-only
measurements (network latency, placeholder preservation by a real model, API
success rate) exist for the workstation only, and are drawn as such.

In [29]:
# --- the assistant round trip --------------------------------------------------
def load_e2e_run(directory: Path) -> dict:
    return {
        "directory": directory,
        "run_id": directory.name,
        "pipeline_version": pipeline_version_of(directory),
        "device_class": device_class_of(directory),
        "config": load_json(directory / "benchmark_config.json", f"{directory.name} config"),
        "system": load_json(directory / "system_info.json", f"{directory.name} system"),
        "summary": load_json(directory / "e2e_summary.json", f"{directory.name} e2e summary"),
        "memory": load_json(directory / "e2e_memory_metrics.json", f"{directory.name} e2e memory"),
    }


E2E_BY_KEY = {
    (version, device): load_e2e_run(path)
    for (version, _, device), path in CANONICAL_E2E.items()
}
E2E_BY_KEY = {k: v for k, v in E2E_BY_KEY.items() if v["summary"]}
CURRENT_E2E = [E2E_BY_KEY[key]
               for key in ((CURRENT_PIPELINE, "workstation"), (CURRENT_PIPELINE, "pi"))
               if key in E2E_BY_KEY]

e2e_rows = []
if CURRENT_E2E:
    for run in CURRENT_E2E:
        s = run["summary"]
        boundary = s.get("privacy_round_trip") or {}
        local = s.get("local") or {}
        llm = s.get("llm") or {}
        placeholders = s.get("placeholder_preservation") or {}
        rehydration = s.get("rehydration") or {}
        command = s.get("local_command") or {}
        e2e_rows.append({
            "run_id": run["run_id"],
            "device": device_name(run),
            "system": system_name(run, short=True),
            "llm_mode": s.get("llm_mode"),
            "llm_measured": bool(llm.get("measured")),
            "cases": s.get("cases"),
            "audio_seconds": s.get("audio_seconds"),
            "successful_cases": (s.get("end_to_end") or {}).get("successful_cases"),
            # --- the boundary ---------------------------------------------------
            "outbound_payloads_captured": boundary.get("payloads_captured"),
            "masker_values_tested": boundary.get("masker_values_tested"),
            "masked_values_reappearing_outbound": boundary.get("masked_but_leaked"),
            "raw_transcripts_outbound": boundary.get("raw_transcripts_in_payload"),
            "gold_entities_tested": boundary.get("gold_entities_tested"),
            "detector_misses_outbound": boundary.get("missed_by_detector_and_in_payload"),
            "leakage_rate": boundary.get("leakage_rate"),
            # --- local -----------------------------------------------------------
            "mean_stt_ms": (local.get("stt_seconds") or {}).get("mean_ms"),
            "mean_masking_ms": (local.get("privacy_seconds") or {}).get("mean_ms"),
            "mean_local_pre_llm_ms": (local.get("pre_llm_seconds") or {}).get("mean_ms"),
            "local_real_time_factor_mean": local.get("real_time_factor_mean"),
            "peak_rss_mib": ((run["memory"] or {}).get("process_peak_rss_bytes") or 0) / 1024 ** 2,
            # --- rehydration, local and deterministic ----------------------------
            "deterministic_probe_cases": rehydration.get("deterministic_probe_cases"),
            "deterministic_probe_passed": rehydration.get("deterministic_probe_passed"),
            "malformed_probe_count": rehydration.get("malformed_probe_count"),
            "malformed_probe_passed": rehydration.get("malformed_probe_passed"),
            "local_command_held_locally": command.get("unaddressed_held_locally"),
            "local_command_routed_locally": command.get("addressed_routed_locally"),
            "local_command_requests_built": command.get("requests_built"),
            # --- live only --------------------------------------------------------
            "api_requests_attempted": llm.get("requests_attempted"),
            "api_requests_succeeded": llm.get("requests_succeeded"),
            "mean_network_round_trip_ms": (llm.get("round_trip_seconds") or {}).get("mean_ms"),
            "replies_received": placeholders.get("replies_received"),
            "replies_using_a_placeholder": placeholders.get("replies_referring_to_a_placeholder"),
            "replies_with_only_intact_placeholders":
                placeholders.get("replies_with_only_intact_placeholders"),
            "live_replies_fully_resolved_locally": rehydration.get("live_replies_fully_resolved"),
            "tts_backend": (s.get("tts") or {}).get("backend"),
            "tts_cases_measured": (s.get("tts") or {}).get("measured_cases"),
        })

    names = [row["device"] for row in e2e_rows]
    colours = [DEVICE_COLOUR[run["device_class"]] for run in CURRENT_E2E]
    group = 0.34

    figure, axes = plt.subplots(1, 3, figsize=(15.4, 4.8))

    # --- panel 1: the outbound boundary ----------------------------------------
    left = axes[0]
    boundary_series = [
        ("Values the masker\nreplaced, tested", "masker_values_tested", SERIES["tertiary"]),
        ("...reappearing\noutbound", "masked_values_reappearing_outbound", SERIES["secondary"]),
        ("Raw transcripts\noutbound", "raw_transcripts_outbound", SERIES["secondary"]),
        ("Detector misses\noutbound", "detector_misses_outbound", PURPLE),
    ]
    labels = [label for label, _, _ in boundary_series]
    for index, row in enumerate(e2e_rows):
        values = [row[key] or 0 for _, key, _ in boundary_series]
        offset = (index - 0.5) * group
        bars = left.bar([p + offset for p in range(len(labels))], values, group,
                        color=colours[index], label=row["device"])
        for bar, value in zip(bars, values):
            left.annotate(str(value), (bar.get_x() + bar.get_width() / 2, value),
                          ha="center", va="bottom", xytext=(0, 3),
                          textcoords="offset points", fontsize=10, color=INK)
    left.set_xticks(range(len(labels)), labels, fontsize=8.5)
    left.set_ylim(0, max(row[key] or 0 for row in e2e_rows for _, key, _ in boundary_series) * 1.35)
    left.set_ylabel("Count (the middle two must be zero)")
    left.set_title("What reached the outbound request\nread off the bytes handed to urllib",
                   loc="left", fontsize=11)
    left.legend(loc="upper right", fontsize=9)
    tidy(left)

    # --- panel 2: local cost ----------------------------------------------------
    middle = axes[1]
    local_series = [("Whisper STT", "mean_stt_ms"),
                    ("Privacy masking", "mean_masking_ms"),
                    ("Local total,\nbefore the request", "mean_local_pre_llm_ms")]
    labels = [label for label, _ in local_series]
    for index, row in enumerate(e2e_rows):
        values = [(row[key] or 0) / 1000.0 for _, key in local_series]
        offset = (index - 0.5) * group
        bars = middle.bar([p + offset for p in range(len(labels))], values, group,
                          color=colours[index], label=row["device"])
        for bar, value in zip(bars, values):
            middle.annotate(f"{value:.3f}", (bar.get_x() + bar.get_width() / 2, value),
                            ha="center", va="bottom", xytext=(0, 3),
                            textcoords="offset points", fontsize=9, color=INK)
    middle.set_xticks(range(len(labels)), labels, fontsize=9)
    middle.set_ylim(0, max((row[key] or 0) / 1000.0
                           for row in e2e_rows for _, key in local_series) * 1.35)
    middle.set_ylabel("Seconds per case (lower is better)")
    live = [row for row in e2e_rows if row["mean_network_round_trip_ms"]]
    network_note = (
        f"network round trip, {live[0]['device']} only: "
        f"{live[0]['mean_network_round_trip_ms'] / 1000.0:.2f} s mean"
        if live else "network round trip: not measured"
    )
    middle.set_title(f"Local cost, mean over 6 cases\n{network_note}", loc="left", fontsize=11)
    middle.legend(loc="upper left", fontsize=9)
    tidy(middle)

    # --- panel 3: the local guarantees ------------------------------------------
    right = axes[2]
    checks = [
        ("Deterministic\nrehydration probes", "deterministic_probe_passed", "deterministic_probe_cases"),
        ("Malformed-placeholder\nprobes", "malformed_probe_passed", "malformed_probe_count"),
        ("Cases completed\nend to end", "successful_cases", "cases"),
    ]
    labels = [label for label, _, _ in checks]
    for index, row in enumerate(e2e_rows):
        values = [row[passed] or 0 for _, passed, _ in checks]
        totals = [row[total] or 0 for _, _, total in checks]
        offset = (index - 0.5) * group
        bars = right.bar([p + offset for p in range(len(labels))], values, group,
                         color=colours[index], label=row["device"])
        for bar, value, total in zip(bars, values, totals):
            right.annotate(f"{value}/{total}", (bar.get_x() + bar.get_width() / 2, value),
                           ha="center", va="bottom", xytext=(0, 3),
                           textcoords="offset points", fontsize=9.5, color=INK)
    right.set_xticks(range(len(labels)), labels, fontsize=9)
    right.set_ylim(0, max(row[passed] or 0 for row in e2e_rows
                          for _, passed, _ in checks) * 1.4)
    right.set_ylabel("Passed (higher is better)")
    right.set_title("Local guarantees\nno network needed for any of these",
                    loc="left", fontsize=11)
    right.legend(loc="upper left", fontsize=9)
    tidy(right)

    audited_only = [row for row in e2e_rows if not row["llm_measured"]]
    if audited_only:
        figure.text(
            0.012, -0.06,
            "  ".join(f"{row['device']}: no API key, so the four transport-dependent "
                      "success conditions (request succeeded, no corrupted placeholder "
                      "in the reply, every returned placeholder resolved, non-empty "
                      "restored answer) were not counted."
                      for row in audited_only),
            va="bottom", ha="left", fontsize=8.5, color=MUTED,
        )
    modes = " | ".join(f"{row['device']}: {row['llm_mode']}" for row in e2e_rows)
    figure.suptitle(
        f"{SYSTEM_LABEL[CURRENT_PIPELINE]}: the assistant round trip and the privacy boundary\n"
        f"6 committed clips, 27.0 s of audio - {modes}",
        fontsize=12.5, y=1.06)
    figure.tight_layout()
    save(figure, "assistant_privacy_boundary",
         "What reached the outbound request, what the local pipeline cost, and the "
         "local guarantees, for the current system on both devices. The boundary "
         "counts are read off the bytes handed to urllib. The Raspberry Pi run had "
         "no API key, so its requests were built and audited but never sent; the "
         "network, placeholder-preservation and TTS measurements are workstation "
         "only and are reported as such.",
         runs=[run["run_id"] for run in CURRENT_E2E])

    print(f"{'':30} " + " ".join(f"{row['device']:>22}" for row in e2e_rows))
    print("-" * (30 + 23 * len(e2e_rows)))
    for label, key in (("LLM leg", "llm_mode"),
                       ("outbound payloads captured", "outbound_payloads_captured"),
                       ("masked values tested", "masker_values_tested"),
                       ("...reappearing outbound", "masked_values_reappearing_outbound"),
                       ("raw transcripts outbound", "raw_transcripts_outbound"),
                       ("detector misses outbound", "detector_misses_outbound"),
                       ("local RTF (mean)", "local_real_time_factor_mean"),
                       ("peak RSS (MiB)", "peak_rss_mib"),
                       ("local command built requests", "local_command_requests_built"),
                       ("cases completed", "successful_cases")):
        cells = []
        for row in e2e_rows:
            value = row[key]
            cells.append(f"{value:>22.3f}" if isinstance(value, float) else f"{str(value):>22}")
        print(f"{label:30} " + " ".join(cells))
else:
    print("no canonical current-system round-trip run is available")

save_table(
    e2e_rows,
    ["run_id", "device", "system", "llm_mode", "llm_measured", "cases", "audio_seconds",
     "successful_cases", "outbound_payloads_captured", "masker_values_tested",
     "masked_values_reappearing_outbound", "raw_transcripts_outbound",
     "gold_entities_tested", "detector_misses_outbound", "leakage_rate",
     "mean_stt_ms", "mean_masking_ms", "mean_local_pre_llm_ms",
     "local_real_time_factor_mean", "peak_rss_mib", "deterministic_probe_cases",
     "deterministic_probe_passed", "malformed_probe_count", "malformed_probe_passed",
     "local_command_held_locally", "local_command_routed_locally",
     "local_command_requests_built", "api_requests_attempted",
     "api_requests_succeeded", "mean_network_round_trip_ms", "replies_received",
     "replies_using_a_placeholder", "replies_with_only_intact_placeholders",
     "live_replies_fully_resolved_locally", "tts_backend", "tts_cases_measured"],
    "assistant_privacy_boundary",
    "The assistant round trip on each device: what reached the outbound request, "
    "what the local legs cost, and which live-only measurements exist. Blank or "
    "zero live columns on the Raspberry Pi row mean the leg was not attempted "
    "because no API key was configured, not that it failed.",
    runs=[row["run_id"] for row in e2e_rows],
)

  figure  research_results/figures/assistant_privacy_boundary.png  (+ .pdf)  <- 2 run(s)
                                          Workstation         Raspberry Pi 5
----------------------------------------------------------------------------
LLM leg                                          live     payload audit only
outbound payloads captured                          6                      6
masked values tested                                9                      9
...reappearing outbound                             0                      0
raw transcripts outbound                            0                      0
detector misses outbound                            2                      2
local RTF (mean)                                0.301                  0.587
peak RSS (MiB)                               1251.184               1528.578
local command built requests                        0                      0
cases completed                                     6           

## 14. Index and overview

In [30]:
figures = [item for item in GENERATED if item["kind"] == "figure"]
tables = [item for item in GENERATED if item["kind"] == "table"]

index = {
    "generated_from": "Research_Paper_Figures.ipynb",
    "repository_root": REPO_ROOT.name,
    "current_pipeline": CURRENT_PIPELINE,
    "current_pipeline_root": PIPELINE_ROOT.name,
    # The whole provenance chain in one place: which committed run backs which
    # figure and which table. No entry here is "the latest run".
    "canonical_runs": {
        "/".join(key): value for key, value in sorted(CANONICAL_RUNS.items())
    },
    "canonical_runs_resolved": {
        "/".join(key): path.relative_to(RESULTS).as_posix()
        for key, path in sorted(CANONICAL_PATHS.items())
    },
    "figure_provenance": {name: runs for name, runs in sorted(FIGURE_PROVENANCE.items())},
    "figures": figures,
    "tables": tables,
    "benchmark_runs_on_disk": [directory.name for directory in DISCOVERED],
    "runs_not_canonical": [directory.name for directory in NON_CANONICAL],
    "recognizer_arms": {
        name.replace("\n", " - "): (None if run is None else run["run_id"])
        for name, run, _, _ in ARMS
    },
    "arms_not_measured": PENDING,
    "unavailable_inputs": UNAVAILABLE,
}
index_path = SUMMARIES / "figure_index.json"
with index_path.open("w", encoding="utf-8", newline="\n") as stream:
    stream.write(json.dumps(index, indent=2) + "\n")

lines = [
    "# Research results overview",
    "",
    "Generated by `Research_Paper_Figures.ipynb`. Every number here comes from a "
    "saved evaluation artifact in this repository; nothing was recomputed.",
    "",
    f"The current system is **FullPipeline_{CURRENT_PIPELINE}**, measured directly on "
    "the development workstation and on a Raspberry Pi 5.",
    "",
    f"- Figures: **{len(figures)}** (PNG at 300 dpi and PDF, in `research_results/figures/`)",
    f"- Tables: **{len(tables)}** (CSV, in `research_results/tables/`)",
    f"- Benchmark run folders on disk: **{len(DISCOVERED)}**, of which "
    f"**{len(CANONICAL_PATHS)}** are canonical",
    "",
    "Every figure names the exact run ids behind it in "
    "`research_results/summaries/figure_index.json`.",
    "",
]

if split_rows:
    lines += ["## Privacy-model quality (text only, device-independent)", "",
              "| View | Rows | Typed F1 | Character recall | Leakage |",
              "|---|---:|---:|---:|---:|"]
    for row in split_rows:
        lines.append(
            f"| {row['view']} | {row['rows']:,} | {row['typed_f1']:.4f} | "
            f"{row['private_character_recall']:.4f} | {row['private_character_leakage']:.4f} |"
        )
    lines.append("")

if quantization_rows:
    worst = max(quantization_rows, key=lambda row: row["typed_f1_drop"])
    lines += [
        "## Quantization",
        "",
        f"INT8 costs at most **{worst['typed_f1_drop']:.4f}** typed entity F1 "
        f"(on `{worst['view']}`), against a release gate of {GATE_LIMIT}. "
        "Every view passed.",
        "",
    ]

if ood_rows:
    lines += ["## Out-of-distribution robustness", "",
              "| Fixture | Model | Typed F1 | Character recall | Hard-negative FP |",
              "|---|---|---:|---:|---:|"]
    for row in ood_rows:
        lines.append(
            f"| {row['fixture']} | {row['model']} | {row['typed_f1']:.4f} | "
            f"{row['character_recall']:.4f} | {row['hard_negative_fp']:.4f} |"
        )
    lines.append("")

if device_rows:
    lines += [
        f"## FullPipeline_{CURRENT_PIPELINE} deployment: workstation and Raspberry Pi 5",
        "",
        "Audio benchmark, 44 committed clips. `RTF < 1` means the device processes "
        "audio faster than it arrives.",
        "",
        "| Device | Mean STT RTF | P90 STT RTF | Mean pipeline RTF | P90 pipeline RTF | "
        "P50 latency | P95 latency | Peak RSS | Run |",
        "|---|---:|---:|---:|---:|---:|---:|---:|---|",
    ]
    for row in device_rows:
        lines.append(
            f"| {row['device']} | {row['mean_stt_rtf']:.3f} | {row['p90_stt_rtf']:.3f} | "
            f"{row['mean_pipeline_rtf']:.3f} | {row['p90_pipeline_rtf']:.3f} | "
            f"{row['p50_end_to_end_ms']:,.0f} ms | {row['p95_end_to_end_ms']:,.0f} ms | "
            f"{row['peak_rss_mib']:,.0f} MiB | `{row['run_id']}` |"
        )
    lines.append("")

if quality_rows:
    lines += [
        "## Workstation against Raspberry Pi 5: the quality metrics",
        "",
        "These should not move with the device. Where they do, the cause is INT8 "
        "kernel arithmetic differing between x86-64 and aarch64.",
        "",
        "| Metric | Benchmark | Better | Workstation | Raspberry Pi 5 | Difference |",
        "|---|---|---|---:|---:|---:|",
    ]
    for row in quality_rows:
        if row["workstation"] is None or row["raspberry_pi_5"] is None:
            continue
        lines.append(
            f"| {row['metric']} | {row['benchmark']} | {row['better']} | "
            f"{row['workstation']:.4f} | {row['raspberry_pi_5']:.4f} | "
            f"{row['difference']:+.4f} |"
        )
    lines.append("")

if parity_rows:
    lines += ["## Cross-device prediction agreement", "",
              "| Scope | Shared | Identical | Agreement |", "|---|---:|---:|---:|"]
    for row in parity_rows:
        lines.append(
            f"| {row['scope']} | {row['shared_items']:,} | {row['identical']:,} | "
            f"{row['agreement']:.4%} |"
        )
    lines += ["",
              f"Reference `{parity_rows[0]['reference_run']}`, comparison "
              f"`{parity_rows[0]['comparison_run']}`.", ""]

if e2e_rows:
    lines += [
        "## Assistant integration and the privacy boundary",
        "",
        "Six committed clips through the whole system. The boundary counts are read "
        "off the bytes handed to `urllib`, not off a function's arguments.",
        "",
        "| Device | LLM leg | Payloads | Masked values tested | ...reappearing outbound | "
        "Raw transcripts outbound | Detector misses outbound | Cases completed | Run |",
        "|---|---|---:|---:|---:|---:|---:|---:|---|",
    ]
    for row in e2e_rows:
        lines.append(
            f"| {row['device']} | {row['llm_mode']} | {row['outbound_payloads_captured']} | "
            f"{row['masker_values_tested']} | **{row['masked_values_reappearing_outbound']}** | "
            f"**{row['raw_transcripts_outbound']}** | {row['detector_misses_outbound']} | "
            f"{row['successful_cases']}/{row['cases']} | `{row['run_id']}` |"
        )
    lines += [
        "",
        "`...reappearing outbound` is a boundary failure and must be zero. "
        "`Detector misses outbound` is a recall miss by the privacy model, which is "
        "why the 44-clip audio benchmark exists; the two must not be added "
        "together. A Raspberry Pi row reading `payload audit only` means no API key "
        "was configured, so the requests were built and audited but not sent.",
        "",
    ]

if inheritance_rows:
    drift = [row for row in inheritance_rows
             if not row["expected_to_move"] and not row["identical"]]
    quality_count = sum(1 for row in inheritance_rows if not row["expected_to_move"])
    lines += [
        "## Did adding the assistant change the local core?",
        "",
        (f"No. All {quality_count} quality comparisons between "
         f"{SYSTEM_SHORT[PREVIOUS_PIPELINE]} and {SYSTEM_SHORT[CURRENT_PIPELINE]}, on "
         "the same device and the same fixtures, are identical to machine precision."
         if not drift else
         f"{len(drift)} of {quality_count} quality comparisons differ - see "
         "`current_system_inherits_previous_core.csv`."),
        "",
    ]

if deployment_rows:
    lines += [
        "## Audio pipeline benchmark, every canonical run",
        "",
        "A controlled end-to-end benchmark on a few dozen clips - a **synthetic "
        "privacy speech benchmark** plus a LibriSpeech dev-clean subset. Not a "
        "speech-recognition benchmark, and not a replacement for the held-out text "
        "evaluation above.",
        "",
        "`Mean WER` is each run's own raw figure and is **not comparable between "
        "recognizers** - Canary wrote spoken numbers as words and Whisper writes "
        "numerals. The engine-comparable, number-normalised column is beside it.",
        "",
        "| System | Device | Clips | WER | WER (norm.) | Mean STT RTF | Mean pipeline RTF | "
        "P95 latency | Peak RSS | Character recall | Leakage |",
        "|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|",
    ]
    for row in deployment_rows:
        def cell(value, spec=".4f"):
            return "not recorded" if value is None else format(value, spec)

        lines.append(
            f"| {row['system']} | {row['device']} | {row['audio_clips']} | "
            f"{cell(row['aggregate_wer'])} | {cell(row['aggregate_wer_number_normalised'])} | "
            f"{cell(row['mean_stt_rtf'], '.3f')} | {cell(row['mean_pipeline_rtf'], '.3f')} | "
            f"{cell(row['p95_end_to_end_ms'], ',.0f')} ms | "
            f"{cell(row['peak_rss_mib'], ',.0f')} MiB | "
            f"{cell(row['character_recall_after_stt'])} | "
            f"{cell(row['character_leakage_after_stt'])} |"
        )
    lines.append("")

lines += ["## Canonical runs", "",
          "Which committed run backs which claim. Nothing in this notebook selects "
          "a run by recency.", "",
          "| Pipeline | Benchmark | Device | Run |", "|---|---|---|---|"]
for key in sorted(CANONICAL_PATHS, reverse=True):
    version, kind, device = key
    lines.append(
        f"| {SYSTEM_SHORT.get(version, version)} ({version}) | {kind} | "
        f"{DEVICE_LABEL[device]} | `{CANONICAL_PATHS[key].relative_to(RESULTS).as_posix()}` |"
    )
lines.append("")

lines += ["## Figures", ""]
# A nested bullet rather than a two-space hard break: trailing whitespace is
# invisible, `git diff --check` flags it, and this renders the same.
lines += [
    f"- `{item['name']}` - {item['caption']}"
    + (f"\n  - Runs: {', '.join('`' + run + '`' for run in item['runs'])}"
       if item.get("runs") else "")
    for item in figures
] or ["- none"]
lines += ["", "## Tables", ""]
lines += [f"- `{item['name']}.csv` ({item['rows']} rows) - {item['caption']}"
          for item in tables] or ["- none"]

if UNAVAILABLE:
    lines += ["", "## Inputs not found, and runs held back", ""]
    lines += [f"- {message}" for message in UNAVAILABLE]
lines.append("")

overview_path = SUMMARIES / "results_overview.md"
overview_path.write_text("\n".join(lines), encoding="utf-8", newline="\n")

print(f"{len(figures)} figures, {len(tables)} tables")
print(f"wrote {index_path.relative_to(REPO_ROOT).as_posix()}")
print(f"wrote {overview_path.relative_to(REPO_ROOT).as_posix()}")
if UNAVAILABLE:
    print("\ninputs not found, and runs held back:")
    for message in UNAVAILABLE:
        print(" ", message)

24 figures, 27 tables
wrote research_results/summaries/figure_index.json
wrote research_results/summaries/results_overview.md

inputs not found, and runs held back:
  20260819T073431Z_windows-amd64_audio-whisper_full_workstation: discovered but not canonical for V4/audio_pipeline/workstation; kept on disk as evidence, out of the figures
  20260819T080341Z_windows-amd64_whisper_full_workstation: discovered but not canonical for V4/cpu_text/workstation; kept on disk as evidence, out of the figures
  20260820T071755Z_linux-aarch64_audio-whisper_full: discovered but not canonical for V4/audio_pipeline/pi; kept on disk as evidence, out of the figures
  20260820T072331Z_linux-aarch64_whisper_full: discovered but not canonical for V4/cpu_text/pi; kept on disk as evidence, out of the figures
  20260821T201401Z_linux-aarch64_audio-whisper_full: discovered but not canonical for V4/audio_pipeline/pi; kept on disk as evidence, out of the figures
  20260821T201941Z_linux-aarch64_whisper_full: disco

---

## Done

`research_results/figures/` holds a PNG and a PDF of every plot; drop either
straight into the report. `research_results/tables/` holds the numbers behind
them, so a figure can be redrawn or a value quoted without reopening this
notebook. `research_results/summaries/results_overview.md` is the written
digest, and `research_results/summaries/figure_index.json` records the exact
benchmark run ids behind every figure.

### Adding a new benchmark run

Run the benchmark, commit the folder it writes, then add its run id to
`CANONICAL_RUNS` in section 1 and press Run All. A new folder that is not added
to that table is reported as discovered-not-canonical and does not enter any
figure - which is the point: no run is ever picked up silently because it
happens to be the newest.

```bash
python Full_Pipeline_CPU_Benchmark.py
python Full_Pipeline_Audio_Benchmark.py
python Full_Pipeline_V6_EndToEnd_Benchmark.py
```

### What is not drawn

An arm that was never measured on a device is printed as not measured and gets
no bar, no zero and no estimate. The Parakeet demonstration (V5) was never run
on a Raspberry Pi, and the Raspberry Pi round trip ran without an API key, so
its network, placeholder-preservation and speech measurements do not exist.
Both are stated on the figures rather than filled in.